In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/roggi/rogii-wellbore-geology-prediction.zip

Archive:  /content/drive/MyDrive/roggi/rogii-wellbore-geology-prediction.zip
  inflating: AI_wellbore_geology_prediction_task_en.pptx  
  inflating: sample_submission.csv   
  inflating: test/000d7d20__horizontal_well.csv  
  inflating: test/000d7d20__typewell.csv  
  inflating: test/00bbac68__horizontal_well.csv  
  inflating: test/00bbac68__typewell.csv  
  inflating: test/00e12e8b__horizontal_well.csv  
  inflating: test/00e12e8b__typewell.csv  
  inflating: train/000d7d20.png      
  inflating: train/000d7d20__horizontal_well.csv  
  inflating: train/000d7d20__typewell.csv  
  inflating: train/00bbac68.png      
  inflating: train/00bbac68__horizontal_well.csv  
  inflating: train/00bbac68__typewell.csv  
  inflating: train/00e12e8b.png      
  inflating: train/00e12e8b__horizontal_well.csv  
  inflating: train/00e12e8b__typewell.csv  
  inflating: train/015fe0d2.png      
  inflating: train/015fe0d2__horizontal_well.csv  
  inflating: train/015fe0d2__typewell.csv  
  inflating: tr

In [ ]:
import os
import math
import glob
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# Alyaev-original-style ROGII synthetic generator（合成数据方案）
#
# Key idea:
#   generated position = synthetic geo_delta / stratigraphic surface delta
#
#   geo_ps  = last_tvt + z_ps
#   TVT_syn = geo_ps + position - Z_future
#   GR_syn  = interp(typewell_GR, TVT_syn) + synthetic colored noise
#
# This version does NOT use real future TVT or real future TVT range.
# ============================================================


TRAIN_DIR = "/content/train"
OUT_DIR   = "./synthetic_alyaev_original_train_x4"


# ============================================================
# Config
# ============================================================

class SynConfig:
    SEED = 20260627

    # Generate multiple independent synthetic variants for each real well.
    # First expansion run: 4x is safer than jumping directly to 8x/16x.
    SYN_PER_WELL = 4

    # Alyaev-like random curve process
    ANGLE_MIN_DEG = 72.0
    ANGLE_MAX_DEG = 108.0

    ANGLE_RANDOM_UNIFORM = 0.010       # original: (random - 0.5) * 0.01
    MAX_ANGLE_CHANGE = 0.035           # original: _max_angle_change = 0.035

    # Fault-like jump in geo_delta domain.
    # For first pretrain run, keep it low.
    FAULT_PROB = 0
    FAULT_MAX_FT = 10.0

    # synthetic TVT-delta trend distribution, independent of true future TVT
    # mixture controls final target TVT range.
    P_FLAT = 0.25
    P_MEDIUM = 0.55
    P_LARGE = 0.18
    P_STRESS = 0.02

    FLAT_RANGE = (8.0, 22.0)
    MEDIUM_RANGE = (22.0, 55.0)
    LARGE_RANGE = (55.0, 95.0)
    STRESS_RANGE = (95.0, 130.0)

    N_CTRL_MIN = 4
    N_CTRL_MAX = 7
    CTRL_BEND_FRAC = 0.25

    # optional local stretch/squeeze of the trend
    WARP_STRENGTH = 0.10

    # GR synthetic noise, based only on typewell statistics
    GR_NOISE_STD_FRAC_LOW = 0.03
    GR_NOISE_STD_FRAC_HIGH = 0.10
    GR_AR_PHI_LOW = 0.70
    GR_AR_PHI_HIGH = 0.95

    GR_SCALE_LOW = 0.98
    GR_SCALE_HIGH = 1.02
    GR_SHIFT_LOW = -2.0
    GR_SHIFT_HIGH = 2.0

    # synthetic missing mask
    POINT_MISSING_PROB_LOW = 0.00
    POINT_MISSING_PROB_HIGH = 0.08
    SEGMENT_MISSING_PROB = 0.25
    SEGMENT_MISSING_LEN = (30, 250)
    HIGH_MISSING_PROB = 0.08
    HIGH_MISSING_FRAC = (0.25, 0.60)

    # typewell boundary
    TVT_MARGIN_FT = 2.0

    # QC
    SYN_TVT_RANGE_MIN = 6.0
    SYN_TVT_RANGE_MAX = 140.0
    MAX_ABS_DTVT_STEP = 3.0
    MAX_ABS_DDTVT_STEP = 1.5

    MIN_FORWARD_GR_STD = 1.0
    MIN_CORR_GR_CLEAN_SYN = 0.75

    # Extra fake-short guard: very small TVT range must still be explained by
    # forward GR rather than mostly synthetic noise.
    FAKE_SHORT_RANGE_FT = 8.0
    FAKE_SHORT_MIN_CORR = 0.85
    FAKE_SHORT_MIN_CLEAN_TO_SYN_STD_RATIO = 0.60

    MAX_TRIES = 50

    # if fail all attempts, skip instead of saving bad sample
    SKIP_IF_FAILED = True


# ============================================================
# Basic utils
# ============================================================

def cot(angle):
    return math.cos(angle) / math.sin(angle)


def angle_from_cot(c):
    # cot(theta)=c -> theta = atan2(1, c)
    return math.atan2(1.0, float(c))


def smooth1d(y, win=101):
    y = np.asarray(y, dtype=float)
    win = int(win)

    if win < 3:
        return y.copy()

    if win % 2 == 0:
        win += 1

    if win >= len(y):
        win = len(y) if len(y) % 2 == 1 else len(y) - 1

    if win < 3:
        return y.copy()

    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    kernel = np.ones(win, dtype=float) / win
    return np.convolve(yp, kernel, mode="valid")


def p95_abs_diff(y, order=1):
    y = np.asarray(y, dtype=float)
    if len(y) <= order:
        return np.nan

    d = y.copy()
    for _ in range(order):
        d = np.diff(d)

    return float(np.nanpercentile(np.abs(d), 95))


def get_ps_index(hw):
    """
    PS = last known TVT_input.
    This does NOT use future TVT.
    """
    if "TVT_input" in hw.columns and hw["TVT_input"].notna().any():
        return int(np.flatnonzero(hw["TVT_input"].notna().to_numpy())[-1])

    raise ValueError("No TVT_input found; cannot anchor synthetic TVT without PS.")


def find_train_pairs(train_dir):
    train_dir = Path(train_dir)

    typewell_files = []
    typewell_files += list(train_dir.glob("*__typewell.csv"))
    typewell_files += list(train_dir.glob("*_typewell.csv"))
    typewell_files += list(train_dir.rglob("*__typewell.csv"))
    typewell_files += list(train_dir.rglob("*_typewell.csv"))

    typewell_files = sorted(set(typewell_files))

    pairs = []
    for tw_path in typewell_files:
        name = tw_path.name

        if name.endswith("__typewell.csv"):
            sid = name.replace("__typewell.csv", "")
            hw_name = f"{sid}__horizontal_well.csv"
        elif name.endswith("_typewell.csv"):
            sid = name.replace("_typewell.csv", "")
            hw_name = f"{sid}_horizontal_well.csv"
        else:
            continue

        hw_path = tw_path.parent / hw_name

        if not hw_path.exists():
            cand = list(train_dir.rglob(f"{sid}__horizontal_well.csv"))
            cand += list(train_dir.rglob(f"{sid}_horizontal_well.csv"))

            if len(cand) == 0:
                continue

            hw_path = cand[0]

        pairs.append((sid, str(tw_path), str(hw_path)))

    return pairs


# ============================================================
# Synthetic curve generation
# ============================================================

def sample_target_tvt_delta_trend(x_norm, rng, cfg=SynConfig):
    """
    Generate a random TVT_delta trend without using real future TVT.

    This is only a guide. It is converted to:
        geo_delta_trend = z_delta + tvt_delta_trend

    Then the Alyaev angle process generates the actual geo_delta curve.
    """
    u = rng.random()

    if u < cfg.P_FLAT:
        target_range = rng.uniform(*cfg.FLAT_RANGE)
    elif u < cfg.P_FLAT + cfg.P_MEDIUM:
        target_range = rng.uniform(*cfg.MEDIUM_RANGE)
    elif u < cfg.P_FLAT + cfg.P_MEDIUM + cfg.P_LARGE:
        target_range = rng.uniform(*cfg.LARGE_RANGE)
    else:
        target_range = rng.uniform(*cfg.STRESS_RANGE)

    sign = rng.choice([-1.0, 1.0])

    # end drift uses a fraction of target_range; internal bends create full range.
    end_delta = sign * rng.uniform(0.35, 1.00) * target_range

    k = int(rng.integers(cfg.N_CTRL_MIN, cfg.N_CTRL_MAX + 1))
    ctrl_x = np.linspace(0.0, 1.0, k)

    ctrl_y = np.linspace(0.0, end_delta, k)

    # low-frequency bends
    bend_amp = cfg.CTRL_BEND_FRAC * target_range
    bend = rng.normal(0.0, bend_amp, size=k)
    bend[0] = 0.0
    bend[-1] = 0.0

    ctrl_y = ctrl_y + bend

    # mild monotone-ish warp in x for stretch/squeeze
    if cfg.WARP_STRENGTH > 0:
        warp_ctrl = np.linspace(0, 1, k)
        warp_noise = rng.normal(0, cfg.WARP_STRENGTH, size=k)
        warp_noise[0] = 0.0
        warp_noise[-1] = 0.0
        warped_x = ctrl_x + warp_noise
        warped_x = np.clip(warped_x, 0, 1)
        warped_x = np.sort(warped_x)
        warped_x[0] = 0.0
        warped_x[-1] = 1.0
        ctrl_x = warped_x

    tvt_delta = np.interp(x_norm, ctrl_x, ctrl_y)

    # smooth low frequency
    L = len(x_norm)
    tvt_delta = smooth1d(
        tvt_delta,
        win=max(31, min(151, L // 25 * 2 + 1)),
    )

    # anchor start to 0
    tvt_delta = tvt_delta - tvt_delta[0]

    return tvt_delta, target_range


def generate_ar_noise(n, std, phi, rng):
    """
    Colored GR noise. Does not use true horizontal residual.
    """
    eps = rng.normal(0.0, std * math.sqrt(max(1e-6, 1 - phi ** 2)), size=n)
    y = np.zeros(n, dtype=float)

    for i in range(1, n):
        y[i] = phi * y[i - 1] + eps[i]

    y = y - np.mean(y)
    if np.std(y) > 1e-6:
        y = y / np.std(y) * std

    return y


def make_synthetic_missing_mask(n, rng, cfg=SynConfig):
    """
    Synthetic missing mask, not copied from horizontal well future GR.
    """
    if rng.random() < cfg.HIGH_MISSING_PROB:
        point_prob = rng.uniform(*cfg.HIGH_MISSING_FRAC)
    else:
        point_prob = rng.uniform(cfg.POINT_MISSING_PROB_LOW, cfg.POINT_MISSING_PROB_HIGH)

    mask = rng.random(n) < point_prob

    if rng.random() < cfg.SEGMENT_MISSING_PROB and n > 200:
        n_seg = int(rng.integers(1, 4))
        for _ in range(n_seg):
            seg_len = int(rng.integers(cfg.SEGMENT_MISSING_LEN[0], cfg.SEGMENT_MISSING_LEN[1] + 1))
            seg_len = min(seg_len, n)
            start = int(rng.integers(0, max(1, n - seg_len + 1)))
            mask[start:start + seg_len] = True

    return mask


# ============================================================
# One synthetic sample
# ============================================================

def _generate_once_original_style(typewell_csv, horizontal_csv, seed, cfg=SynConfig):
    rng = np.random.default_rng(seed)

    tw = pd.read_csv(typewell_csv).sort_values("TVT").reset_index(drop=True)
    hw = pd.read_csv(horizontal_csv).reset_index(drop=True)

    t_tvt = tw["TVT"].astype(float).to_numpy()
    t_gr = tw["GR"].astype(float).to_numpy()

    h_md = hw["MD"].astype(float).to_numpy()
    h_z = hw["Z"].astype(float).to_numpy()

    h_ps = get_ps_index(hw)

    idx_f = np.arange(h_ps + 1, len(hw))
    if len(idx_f) < 32:
        return None, None

    # Anchor only from TVT_input prefix
    tvt_input = hw["TVT_input"].astype(float).to_numpy()
    last_tvt = float(tvt_input[h_ps])

    md_ps = float(h_md[h_ps])
    z_ps = float(h_z[h_ps])
    geo_ps = last_tvt + z_ps

    md_f = h_md[idx_f]
    z_f = h_z[idx_f]

    L = len(idx_f)

    x_dist = md_f - md_ps
    x_norm = (x_dist - x_dist.min()) / (x_dist.max() - x_dist.min() + 1e-12)

    dmd = np.diff(np.r_[md_ps, md_f])
    dmd = np.maximum(dmd, 1e-6)

    z_delta = z_f - z_ps

    # --------------------------------------------------------
    # 1. Generate random TVT_delta trend independent of truth
    # --------------------------------------------------------

    tvt_delta_trend, target_range = sample_target_tvt_delta_trend(
        x_norm=x_norm,
        rng=rng,
        cfg=cfg,
    )

    # Key mapping:
    # position is geo_delta, therefore guide is z_delta + tvt_delta.
    geo_delta_trend = z_delta + tvt_delta_trend
    geo_delta_trend = smooth1d(
        geo_delta_trend,
        win=max(31, min(151, L // 25 * 2 + 1)),
    )

    expected_cot = np.gradient(geo_delta_trend, md_f)
    expected_cot = smooth1d(
        expected_cot,
        win=max(31, min(101, L // 35 * 2 + 1)),
    )

    # --------------------------------------------------------
    # 2. Alyaev-style SubsurfaceState process
    # --------------------------------------------------------

    angle_min = math.radians(cfg.ANGLE_MIN_DEG)
    angle_max = math.radians(cfg.ANGLE_MAX_DEG)

    positions = np.zeros(L, dtype=float)
    angles = np.zeros(L, dtype=float)

    angle = angle_from_cot(expected_cot[0])
    angle = float(np.clip(angle, angle_min, angle_max))

    pos = 0.0
    positions[0] = pos
    angles[0] = angle

    fault_count = 0

    for i in range(1, L):
        new_thl_distance = float(dmd[i])

        # original-like random angle perturbation
        new_angle = angle + (rng.random() - 0.5) * cfg.ANGLE_RANDOM_UNIFORM

        expected_cot_i = float(expected_cot[i])
        new_cot = cot(new_angle)

        delta_angle = 0.0

        # same logic as Alyaev:
        # if current cot is far from expected cot, push angle toward expected trend
        if (0.5 + rng.random()) * 0.03 < abs(expected_cot_i - new_cot) and rng.random() > 0.2:
            sign = -np.sign(expected_cot_i - new_cot)
            magnitude = abs(expected_cot_i - new_cot) * (1.0 + (rng.random() - 0.5) * 0.8)
            delta_angle = sign * min(cfg.MAX_ANGLE_CHANGE, magnitude)

        new_angle = new_angle + delta_angle
        new_angle = float(np.clip(new_angle, angle_min, angle_max))

        # integrate position
        new_position = pos + new_thl_distance * cot(new_angle)

        # original-like fault/position correction toward expected position
        expected_position = float(geo_delta_trend[i] - geo_delta_trend[0])

        delta = 0.0
        if rng.random() < cfg.FAULT_PROB:
            mismatch = abs(expected_position - new_position)
            if mismatch > 1e-6:
                sign = np.sign(expected_position - new_position)
                magnitude = mismatch * (1.0 + (rng.random() - 0.5))
                delta = sign * min(cfg.FAULT_MAX_FT, magnitude)
                fault_count += 1

        new_position = new_position + delta

        angle = new_angle
        pos = new_position

        positions[i] = pos
        angles[i] = angle

    geo_delta_syn = geo_delta_trend[0] + positions

    # --------------------------------------------------------
    # 3. Convert geo_delta -> TVT
    # --------------------------------------------------------

    tvt_syn_f = geo_ps + geo_delta_syn - z_f

    # exact PS continuity taper
    delta0 = tvt_syn_f[0] - last_tvt
    taper = np.exp(-np.linspace(0, 7, L))
    tvt_syn_f = tvt_syn_f - delta0 * taper

    geo_delta_syn = tvt_syn_f + z_f - geo_ps

    # boundary QC, no clipping in accepted sample
    clip_min = float(t_tvt.min() + cfg.TVT_MARGIN_FT)
    clip_max = float(t_tvt.max() - cfg.TVT_MARGIN_FT)

    out_of_bounds = int(np.sum((tvt_syn_f < clip_min) | (tvt_syn_f > clip_max)))

    # --------------------------------------------------------
    # 4. Forward GR from typewell + synthetic colored noise
    # --------------------------------------------------------

    gr_clean = np.interp(tvt_syn_f, t_tvt, t_gr)

    typewell_gr_std = float(np.nanstd(t_gr))
    noise_std = rng.uniform(cfg.GR_NOISE_STD_FRAC_LOW, cfg.GR_NOISE_STD_FRAC_HIGH) * max(typewell_gr_std, 1.0)
    phi = rng.uniform(cfg.GR_AR_PHI_LOW, cfg.GR_AR_PHI_HIGH)

    noise = generate_ar_noise(L, noise_std, phi, rng)

    gr_syn = gr_clean + noise

    gr_syn = (
        gr_syn * rng.uniform(cfg.GR_SCALE_LOW, cfg.GR_SCALE_HIGH)
        + rng.uniform(cfg.GR_SHIFT_LOW, cfg.GR_SHIFT_HIGH)
    )

    missing_mask = make_synthetic_missing_mask(L, rng, cfg)
    gr_syn_raw = gr_syn.copy()
    gr_syn_raw[missing_mask] = np.nan

    # --------------------------------------------------------
    # 5. Build synthetic horizontal well
    # --------------------------------------------------------

    syn_hw = hw.copy()

    # Future target becomes synthetic.
    # Prefix remains original known history.
    syn_hw.loc[idx_f, "TVT"] = tvt_syn_f
    syn_hw.loc[idx_f, "GR"] = gr_syn_raw

    # Keep TVT_input unchanged.
    # This keeps the anchor/prefix exactly like the real task.

    # --------------------------------------------------------
    # 6. Stats / QC
    # --------------------------------------------------------

    syn_tvt_range = float(np.nanmax(tvt_syn_f) - np.nanmin(tvt_syn_f))
    gr_clean_std = float(np.nanstd(gr_clean))
    gr_syn_std = float(np.nanstd(gr_syn))
    noise_std_actual = float(np.nanstd(noise))

    corr = np.nan
    if L > 2 and gr_clean_std > 1e-6 and gr_syn_std > 1e-6:
        corr = float(np.corrcoef(gr_clean, gr_syn)[0, 1])

    stats = {
        "ps_index": int(h_ps),
        "ps_md": float(md_ps),
        "future_rows": int(L),

        "target_range": float(target_range),
        "syn_range_tvt": syn_tvt_range,
        "syn_end_delta_tvt": float(tvt_syn_f[-1] - last_tvt),
        "syn_geo_delta_range": float(np.nanmax(geo_delta_syn) - np.nanmin(geo_delta_syn)),

        "abs_dtvt_p95": p95_abs_diff(tvt_syn_f, 1),
        "abs_ddtvt_p95": p95_abs_diff(tvt_syn_f, 2),

        "gr_clean_std": gr_clean_std,
        "gr_syn_std": gr_syn_std,
        "noise_std": noise_std_actual,
        "corr_gr_clean_syn": corr,

        "missing_frac": float(np.mean(missing_mask)),
        "out_of_bounds_count": out_of_bounds,
        "fault_count": int(fault_count),
    }

    return syn_hw, stats


def generate_one_original_style(
    typewell_csv,
    horizontal_csv,
    out_csv=None,
    seed=SynConfig.SEED,
    cfg=SynConfig,
):
    """
    Regenerate until QC passes.
    Does NOT use true future TVT.
    """
    for attempt in range(cfg.MAX_TRIES):
        syn_hw, stats = _generate_once_original_style(
            typewell_csv=typewell_csv,
            horizontal_csv=horizontal_csv,
            seed=seed + attempt * 10007,
            cfg=cfg,
        )

        if syn_hw is None:
            return None, None

        ok = True

        ok &= stats["out_of_bounds_count"] == 0
        ok &= cfg.SYN_TVT_RANGE_MIN <= stats["syn_range_tvt"] <= cfg.SYN_TVT_RANGE_MAX
        ok &= stats["abs_dtvt_p95"] <= cfg.MAX_ABS_DTVT_STEP
        ok &= stats["abs_ddtvt_p95"] <= cfg.MAX_ABS_DDTVT_STEP
        ok &= stats["gr_clean_std"] >= cfg.MIN_FORWARD_GR_STD
        ok &= stats["corr_gr_clean_syn"] >= cfg.MIN_CORR_GR_CLEAN_SYN

        # Guard against fake-short samples: if TVT barely moves, GR should not
        # be dominated by noise. This keeps real flat wells but rejects noisy
        # synthetic flat wells.
        clean_to_syn_ratio = stats["gr_clean_std"] / (stats["gr_syn_std"] + 1e-6)
        if stats["syn_range_tvt"] < cfg.FAKE_SHORT_RANGE_FT:
            ok &= stats["corr_gr_clean_syn"] >= cfg.FAKE_SHORT_MIN_CORR
            ok &= clean_to_syn_ratio >= cfg.FAKE_SHORT_MIN_CLEAN_TO_SYN_STD_RATIO

        if ok:
            stats["accepted_attempt"] = attempt
            if out_csv is not None:
                syn_hw.to_csv(out_csv, index=False)
            return syn_hw, stats

    if cfg.SKIP_IF_FAILED:
        return None, None

    stats["accepted_attempt"] = -1
    if out_csv is not None:
        syn_hw.to_csv(out_csv, index=False)

    return syn_hw, stats


# ============================================================
# Directory generation
# ============================================================

def generate_synthetic_train_dir_original_style(
    train_dir=TRAIN_DIR,
    out_dir=OUT_DIR,
    max_wells=None,
    seed=SynConfig.SEED,
):
    """
    Generate SYN_PER_WELL independent synthetic variants per real well.

    Output naming:
        source sid: 000d7d20
        variant sid: 000d7d20_syn00, 000d7d20_syn01, ...

    This naming is important because the training Dataset looks for:
        {sid}__horizontal_well.csv
        {sid}__typewell.csv
    so each synthetic variant needs its own matching typewell filename.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pairs = find_train_pairs(train_dir)
    print(f"Found train pairs: {len(pairs)}")
    print(f"Synthetic variants per well: {SynConfig.SYN_PER_WELL}")

    if max_wells is not None:
        pairs = pairs[:max_wells]
        print(f"Using first {len(pairs)} wells")

    all_stats = []
    n_ok = 0
    n_skip = 0

    for i, (sid, tw_path, hw_path) in enumerate(pairs):
        if i % 50 == 0:
            print(f"processing source well {i}/{len(pairs)} | ok={n_ok} skip={n_skip}")

        for k in range(int(SynConfig.SYN_PER_WELL)):
            syn_sid = f"{sid}_syn{k:02d}"
            variant_seed = int(seed + i * 1009 + k * 9176)

            tw_out = out_dir / f"{syn_sid}__typewell.csv"
            hw_out = out_dir / f"{syn_sid}__horizontal_well.csv"

            # Copy typewell unchanged, but with the synthetic sid so SDFDataset
            # can resolve it from the synthetic horizontal filename.
            pd.read_csv(tw_path).to_csv(tw_out, index=False)

            syn_hw, stats = generate_one_original_style(
                typewell_csv=tw_path,
                horizontal_csv=hw_path,
                out_csv=hw_out,
                seed=variant_seed,
            )

            if syn_hw is None:
                n_skip += 1
                # Remove copied typewell if no synthetic horizontal was produced.
                if tw_out.exists():
                    tw_out.unlink()
                if hw_out.exists():
                    hw_out.unlink()
                continue

            row = {
                "well": syn_sid,              # keeps compatibility with old QC code
                "syn_sid": syn_sid,
                "source_sid": sid,
                "variant": int(k),
                "seed": int(variant_seed),
            }
            row.update(stats)
            all_stats.append(row)
            n_ok += 1

    stats_df = pd.DataFrame(all_stats)
    stats_path = out_dir / "synthetic_generation_stats.csv"
    stats_df.to_csv(stats_path, index=False)

    print("\nDONE")
    print(f"saved synthetic dir: {out_dir}")
    print(f"stats: {stats_path}")
    print(f"ok synthetic wells: {n_ok} | skipped variants: {n_skip}")
    print(f"expected max: {len(pairs) * int(SynConfig.SYN_PER_WELL)}")

    if len(stats_df):
        print(stats_df.describe(include="all"))

    return stats_df


def make_recommended_keep_drop_from_stats(stats_df, out_dir=OUT_DIR):
    """
    Optional QC helper. This uses syn_sid-level filtering, not source-well-level
    filtering, so a bad variant can be dropped while the other variants from
    the same real well are kept.
    """
    out_dir = Path(out_dir)

    if len(stats_df) == 0:
        keep = stats_df.copy()
        drop = stats_df.copy()
    else:
        clean_to_syn_ratio = stats_df["gr_clean_std"] / (stats_df["gr_syn_std"] + 1e-6)
        bad = (
            (stats_df["out_of_bounds_count"] > 0)
            | (stats_df["gr_clean_std"] < SynConfig.MIN_FORWARD_GR_STD)
            | (stats_df["corr_gr_clean_syn"] < SynConfig.MIN_CORR_GR_CLEAN_SYN)
            | (stats_df["abs_dtvt_p95"] > SynConfig.MAX_ABS_DTVT_STEP)
            | (stats_df["abs_ddtvt_p95"] > SynConfig.MAX_ABS_DDTVT_STEP)
            | (
                (stats_df["syn_range_tvt"] < SynConfig.FAKE_SHORT_RANGE_FT)
                & (
                    (stats_df["corr_gr_clean_syn"] < SynConfig.FAKE_SHORT_MIN_CORR)
                    | (clean_to_syn_ratio < SynConfig.FAKE_SHORT_MIN_CLEAN_TO_SYN_STD_RATIO)
                )
            )
        )

        keep = stats_df.loc[~bad].copy()
        drop = stats_df.loc[bad].copy()

    keep_path = out_dir / "synthetic_generation_keep.csv"
    drop_path = out_dir / "synthetic_generation_drop.csv"
    keep.to_csv(keep_path, index=False)
    drop.to_csv(drop_path, index=False)

    print(f"QC keep: {len(keep)} -> {keep_path}")
    print(f"QC drop: {len(drop)} -> {drop_path}")

    return keep, drop

# ============================================================
# Run
# ============================================================

if __name__ == "__main__":
    # Debug first:
    # stats_df = generate_synthetic_train_dir_original_style(
    #     train_dir=TRAIN_DIR,
    #     out_dir=OUT_DIR,
    #     max_wells=20,
    #     seed=SynConfig.SEED,
    # )

    #After checking stats, run full:
    stats_df = generate_synthetic_train_dir_original_style(
        train_dir=TRAIN_DIR,
        out_dir=OUT_DIR,
        max_wells=None,
        seed=SynConfig.SEED,
    )

Found train pairs: 773
Synthetic variants per well: 4
processing source well 0/773 | ok=0 skip=0
processing source well 50/773 | ok=200 skip=0
processing source well 100/773 | ok=400 skip=0
processing source well 150/773 | ok=600 skip=0
processing source well 200/773 | ok=800 skip=0
processing source well 250/773 | ok=1000 skip=0
processing source well 300/773 | ok=1200 skip=0
processing source well 350/773 | ok=1400 skip=0
processing source well 400/773 | ok=1600 skip=0
processing source well 450/773 | ok=1800 skip=0
processing source well 500/773 | ok=2000 skip=0
processing source well 550/773 | ok=2200 skip=0
processing source well 600/773 | ok=2400 skip=0
processing source well 650/773 | ok=2600 skip=0
processing source well 700/773 | ok=2800 skip=0
processing source well 750/773 | ok=3000 skip=0

DONE
saved synthetic dir: synthetic_alyaev_original_train_x4
stats: synthetic_alyaev_original_train_x4/synthetic_generation_stats.csv
ok synthetic wells: 3092 | skipped variants: 0
expect

In [ ]:
# ============================================================
# ROGII - Simple Pure Gaussian Heatmap ResUNet-GN + Horizontal Dilated ASPP
#         + Deep Supervision + Soft Heatmap CE
# cv = 9, lb = 8.6
# ============================================================
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Config
# ============================================================
class Config:
    TRAIN_DIR = "/kaggle/input/datasets/zhuyifanss/roggi-data/train"
    TEST_DIR = "/kaggle/input/datasets/zhuyifanss/roggi-data/test"
    TYPEWELL_SUFFIX = "__typewell.csv"
    HORIZONTAL_SUFFIX = "__horizontal_well.csv"

    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    SEED = 42
    NUM_WORKERS = 12
    BATCH_SIZE = 4
    EPOCHS = 20
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    N_FOLDS = 4
    FOLDS_TO_RUN = [0]

    # Grid
    H_S = 16
    H_H = 64
    H_F = 704
    T_H = 128
    T_F = 128
    H_GR_FILTER = 51

    # Pure Gaussian heatmap settings.
    # Target is constructed directly from TVT-distance in feet:
    #   target[t,h] = exp(-dist_ft[t,h]^2 / (2*sigma_ft^2))
    HEATMAP_TARGET_SIGMA_FT = 16.0
    HEATMAP_DECODE_TEMP = 0.35
    HISTORY_SIGMA = 2.0

    ORIG_PAD_LEN = 16384

    # Core augmentation
    START_JITTER = 512
    RANDOM_CUTOFF_PROB = 0.5
    RANDOM_CUTOFF_MIN_FRAC = 0.2
    RANDOM_CUTOFF_MAX_FRAC = 0.8
    RANDOM_CUTOFF_MIN_HISTORY = 512
    RANDOM_CUTOFF_MIN_FUTURE = 512
    # HFlip is intentionally removed for this simple heatmap baseline.
    AUG_REPEATS = 4

    # Trajectory Z
    DZ_SCALE = 2.0

    # Deep Supervision weights
    DS_WEIGHTS = [0.4, 0.2, 0.1]

    # GroupNorm groups
    GN_GROUPS = 8

    # ASPP dilation rates (along H-axis)
    ASPP_DILATIONS = (1, 4, 8, 16)



np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
print("DEVICE:", Config.DEVICE)


# ============================================================
# Preprocess utils
# ============================================================
def resample_typewell(t, target_step=0.5):
    t_tvt = t["TVT"].values.astype(np.float64)
    t_gr = t["GR"].values.astype(np.float64)
    diffs = np.abs(np.diff(t_tvt))
    diffs = diffs[diffs > 0]
    ratio = (np.median(diffs) if len(diffs) else target_step) / target_step
    if np.isclose(ratio, 1.0):
        return t_tvt, t_gr
    if ratio < 1.0:
        group = max(int(round(1 / ratio)), 1)
        pad = (-len(t_tvt)) % group
        if pad:
            t_tvt = np.pad(t_tvt, (0, pad), mode="edge")
            t_gr = np.pad(t_gr, (0, pad), mode="edge")
        return t_tvt.reshape(-1, group).mean(1), t_gr.reshape(-1, group).mean(1)
    up = max(int(round(ratio)), 1)
    old = np.arange(len(t_tvt))
    new = np.linspace(0, len(t_tvt) - 1, (len(t_tvt) - 1) * up + 1)
    return np.interp(new, old, t_tvt), np.interp(new, old, t_gr)


def bin_mean(arr, step, back):
    arr = np.asarray(arr)
    if len(arr) == 0:
        return arr
    pad = (-len(arr)) % step
    if pad < step // 2:
        if pad:
            arr = np.pad(arr, (0, pad) if back else (pad, 0), mode="edge")
    elif pad:
        arr = arr[:-(step - pad)] if back else arr[(step - pad):]
    if len(arr) == 0:
        return arr
    return arr.reshape(-1, step).mean(1)


def safe_savgol(x, win=51, poly=2):
    x = np.asarray(x, dtype=float)
    if len(x) <= win:
        return x
    win = min(win, len(x))
    if win % 2 == 0:
        win -= 1
    if win < 7:
        return x
    return savgol_filter(x, win, poly)


def _get_numeric_col(h, name, default_value=0.0):
    if name in h.columns:
        return (
            h[name]
            .astype(float)
            .interpolate()
            .bfill()
            .ffill()
            .fillna(default_value)
            .values
        )
    return np.full(len(h), float(default_value), dtype=np.float64)


def resample_horizontal(h, step, offset=0, forced_h_ps=None):
    gr_raw = (
        h["GR"]
        .astype(float)
        .interpolate()
        .bfill()
        .ffill()
        .fillna(85.0)
        .values
    )
    gr = safe_savgol(gr_raw, Config.H_GR_FILTER, 2)

    if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
        tvt = h["TVT"].values.astype(float)
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        tvt = (
            h["TVT_input"]
            .astype(float)
            .ffill()
            .bfill()
            .fillna(0.0)
            .values
        )
    else:
        tvt = h["Z"].values.astype(float)

    z = _get_numeric_col(h, "Z", 0.0)
    md = _get_numeric_col(h, "MD", 0.0)
    if "MD" not in h.columns:
        md = np.arange(len(h), dtype=np.float64)
    x = _get_numeric_col(h, "X", 0.0)
    y = _get_numeric_col(h, "Y", 0.0)

    if forced_h_ps is not None:
        h_ps = int(forced_h_ps)
        h_ps = max(0, min(h_ps, len(h) - 1))
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        h_ps = int(np.flatnonzero(h["TVT_input"].notna().values)[-1]) + offset
        h_ps = max(0, min(h_ps, len(h) - 1))
    else:
        h_ps = len(h) // 2

    tvt0 = bin_mean(tvt[:h_ps + 1], step, back=False)
    gr0 = bin_mean(gr[:h_ps + 1], step, back=False)
    z0 = bin_mean(z[:h_ps + 1], step, back=False)
    md0 = bin_mean(md[:h_ps + 1], step, back=False)
    x0 = bin_mean(x[:h_ps + 1], step, back=False)
    y0 = bin_mean(y[:h_ps + 1], step, back=False)

    tvt1 = bin_mean(tvt[h_ps + 1:], step, back=True)
    gr1 = bin_mean(gr[h_ps + 1:], step, back=True)
    z1 = bin_mean(z[h_ps + 1:], step, back=True)
    md1 = bin_mean(md[h_ps + 1:], step, back=True)
    x1 = bin_mean(x[h_ps + 1:], step, back=True)
    y1 = bin_mean(y[h_ps + 1:], step, back=True)

    return tvt0, gr0, tvt1, gr1, h_ps, z0, z1, md0, md1, x0, x1, y0, y1


def crop_pad_1d(n, center, history, future):
    raw_i0 = center - history
    raw_i1 = center + future
    i0 = max(raw_i0, 0)
    i1 = min(raw_i1, n)
    pad_left = max(0, -raw_i0)
    pad_right = max(0, raw_i1 - n)
    return i0, i1, pad_left, pad_right


def get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt):
    if "TVT_input" in h.columns:
        val = h["TVT_input"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if "TVT" in h.columns:
        val = h["TVT"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if len(h_tvt0):
        return float(h_tvt0[-1])
    return float(t_tvt[len(t_tvt) // 2])


def get_official_h_ps(h):
    if "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        return int(np.flatnonzero(h["TVT_input"].notna().values)[-1])
    return len(h) // 2


def choose_train_cutoff(h, cfg, start_jitter=None):
    n = len(h)
    official_ps = get_official_h_ps(h)
    sj = cfg.START_JITTER if start_jitter is None else int(start_jitter)

    can_random = (
        "TVT" in h.columns
        and h["TVT"].notna().sum() > max(32, cfg.RANDOM_CUTOFF_MIN_HISTORY // 4)
        and n > cfg.RANDOM_CUTOFF_MIN_HISTORY + cfg.RANDOM_CUTOFF_MIN_FUTURE + 8
    )

    if can_random and np.random.rand() < cfg.RANDOM_CUTOFF_PROB:
        lo = max(
            int(round(n * cfg.RANDOM_CUTOFF_MIN_FRAC)),
            int(cfg.RANDOM_CUTOFF_MIN_HISTORY),
            1,
        )
        hi = min(
            int(round(n * cfg.RANDOM_CUTOFF_MAX_FRAC)),
            n - 1 - int(cfg.RANDOM_CUTOFF_MIN_FUTURE),
        )
        if hi > lo:
            return int(np.random.randint(lo, hi + 1)), "random_cutoff"

    offset = 0
    if sj > 0:
        offset = -np.random.randint(0, sj + 1)
    h_ps = int(np.clip(official_ps + offset, 0, n - 1))
    return h_ps, "official_jitter"


def make_history_line(t_seg_tvt, h_seg_tvt, hist_mask):
    diff = np.abs(t_seg_tvt[:, None] - h_seg_tvt[None, :])
    matched = diff.argmin(axis=0)
    rows = np.arange(len(t_seg_tvt))[:, None]
    line = np.exp(
        -0.5 * ((rows - matched[None, :]) / Config.HISTORY_SIGMA) ** 2
    ).astype(np.float32)
    line *= hist_mask[None, :]
    return line



# ============================================================
# Dataset
# ============================================================
class HeatmapDataset(Dataset):
    def __init__(self, files, is_train=True, aug_repeats=None, start_jitter=None):
        self.files = list(files)
        self.is_train = is_train
        self.aug_repeats = int(aug_repeats if aug_repeats is not None else (Config.AUG_REPEATS if is_train else 1))
        self.start_jitter = int(start_jitter if start_jitter is not None else Config.START_JITTER)

    def __len__(self):
        if self.is_train:
            return len(self.files) * self.aug_repeats
        return len(self.files)

    def __getitem__(self, idx):
        cfg = Config
        if self.is_train:
            idx = idx % len(self.files)
        hp = Path(self.files[idx])
        sid = hp.name.split("__")[0]
        h = pd.read_csv(hp)
        t = pd.read_csv(hp.parent / f"{sid}{cfg.TYPEWELL_SUFFIX}")
        t_tvt, t_gr = resample_typewell(t, target_step=0.5)

        forced_h_ps = None
        offset = 0
        if self.is_train:
            forced_h_ps, cutoff_mode = choose_train_cutoff(
                h, cfg, start_jitter=self.start_jitter,
            )
        else:
            cutoff_mode = "official"

        h_tvt0, h_gr0, h_tvt1, h_gr1, h_ps, h_z0, h_z1, h_md0, h_md1, h_x0, h_x1, h_y0, h_y1 = resample_horizontal(
            h, cfg.H_S, offset=offset, forced_h_ps=forced_h_ps,
        )

        last_tvt_exact = get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt)
        last_tvt = float(h_tvt0[-1]) if len(h_tvt0) else last_tvt_exact
        last_idx = int(np.abs(t_tvt - last_tvt).argmin()) if len(t_tvt) else 0
        last_z = float(h["Z"].iloc[h_ps])
        last_md = float(h["MD"].iloc[h_ps]) if "MD" in h.columns else float(h_ps)
        last_x = float(h["X"].iloc[h_ps]) if "X" in h.columns else 0.0
        last_y = float(h["Y"].iloc[h_ps]) if "Y" in h.columns else 0.0

        # typewell crop
        i0, i1, pl, pr = crop_pad_1d(len(t_tvt), last_idx + 1, cfg.T_H, cfg.T_F)
        t_mask = np.pad(np.ones(i1 - i0, dtype=np.float32), (pl, pr), constant_values=0.0)
        t_seg_tvt = np.pad(t_tvt[i0:i1], (pl, pr), mode="edge")
        t_seg_gr = np.pad(t_gr[i0:i1], (pl, pr), mode="edge")

        # horizontal history crop
        i0_h, i1_h, p0l, p0r = crop_pad_1d(len(h_tvt0), len(h_tvt0), cfg.H_H, 0)
        hm0 = np.pad(np.ones(i1_h - i0_h, dtype=np.float32), (p0l, p0r), constant_values=0.0)
        if len(h_tvt0):
            ht0 = np.pad(h_tvt0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hg0 = np.pad(h_gr0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hz0 = np.pad(h_z0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hmd0 = np.pad(h_md0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hx0 = np.pad(h_x0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hy0 = np.pad(h_y0[i0_h:i1_h], (p0l, p0r), mode="edge")
        else:
            ht0 = np.zeros(cfg.H_H, dtype=np.float64)
            hg0 = np.zeros(cfg.H_H, dtype=np.float64)
            hz0 = np.full(cfg.H_H, last_z, dtype=np.float64)
            hmd0 = np.full(cfg.H_H, last_md, dtype=np.float64)
            hx0 = np.full(cfg.H_H, last_x, dtype=np.float64)
            hy0 = np.full(cfg.H_H, last_y, dtype=np.float64)

        # horizontal future crop
        i0_f, i1_f, p1l, p1r = crop_pad_1d(len(h_tvt1), 0, 0, cfg.H_F)
        hm1 = np.pad(np.ones(i1_f - i0_f, dtype=np.float32), (p1l, p1r), constant_values=0.0)
        if len(h_tvt1):
            ht1 = np.pad(h_tvt1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hg1 = np.pad(h_gr1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hz1 = np.pad(h_z1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hmd1 = np.pad(h_md1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hx1 = np.pad(h_x1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hy1 = np.pad(h_y1[i0_f:i1_f], (p1l, p1r), mode="edge")
        else:
            ht1 = np.zeros(cfg.H_F, dtype=np.float64)
            hg1 = np.zeros(cfg.H_F, dtype=np.float64)
            hz1 = np.full(cfg.H_F, last_z, dtype=np.float64)
            hmd1 = np.full(cfg.H_F, last_md, dtype=np.float64)
            hx1 = np.full(cfg.H_F, last_x, dtype=np.float64)
            hy1 = np.full(cfg.H_F, last_y, dtype=np.float64)

        h_mask = np.concatenate([hm0, hm1]).astype(np.float32)
        h_seg_tvt = np.concatenate([ht0, ht1]).astype(np.float64)
        h_seg_gr = np.concatenate([hg0, hg1]).astype(np.float64)
        h_seg_z = np.concatenate([hz0, hz1]).astype(np.float64)
        h_seg_md = np.concatenate([hmd0, hmd1]).astype(np.float64)
        h_seg_x = np.concatenate([hx0, hx1]).astype(np.float64)
        h_seg_y = np.concatenate([hy0, hy1]).astype(np.float64)

        hist_mask = np.zeros(cfg.H_H + cfg.H_F, dtype=np.float32)
        hist_mask[:cfg.H_H] = hm0

        h_dz = np.gradient(h_seg_z)
        h_dz = np.clip(h_dz / cfg.DZ_SCALE, -3.0, 3.0).astype(np.float32)

        H = cfg.H_H + cfg.H_F
        col_center = np.zeros(H, dtype=np.float64)
        for j in range(cfg.H_H):
            k = cfg.H_H - 1 - j
            col_center[j] = h_ps - k * cfg.H_S - (cfg.H_S - 1) / 2.0
        for k in range(cfg.H_F):
            col_center[cfg.H_H + k] = h_ps + 1 + k * cfg.H_S + (cfg.H_S - 1) / 2.0
        h_pos_rel = np.clip((col_center - float(h_ps)) / 4096.0, -4.0, 4.0).astype(np.float32)

        dmd = np.gradient(h_seg_md)
        dmd = np.where(np.abs(dmd) < 1e-6, 1.0, dmd)
        dx_dmd = np.gradient(h_seg_x) / dmd
        dy_dmd = np.gradient(h_seg_y) / dmd
        dxy = np.sqrt(dx_dmd ** 2 + dy_dmd ** 2) + 1e-6
        az_cos = np.clip(dx_dmd / dxy, -1.0, 1.0).astype(np.float32)
        az_sin = np.clip(dy_dmd / dxy, -1.0, 1.0).astype(np.float32)

        history_line = make_history_line(t_seg_tvt, h_seg_tvt, hist_mask)

        if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
            orig_tvt = h["TVT"].values.astype(np.float32)
        else:
            orig_tvt = np.zeros(len(h), dtype=np.float32)

        orig_len = len(orig_tvt)
        padded_tvt = np.zeros(cfg.ORIG_PAD_LEN, dtype=np.float32)
        ncopy = min(orig_len, cfg.ORIG_PAD_LEN)
        padded_tvt[:ncopy] = orig_tvt[:ncopy]

        return {
            "t_gr": torch.tensor(t_seg_gr, dtype=torch.float32),
            "h_gr": torch.tensor(h_seg_gr, dtype=torch.float32),
            "hist_mask": torch.tensor(hist_mask, dtype=torch.float32),
            "history_line": torch.tensor(history_line, dtype=torch.float32),
            "h_dz": torch.tensor(h_dz, dtype=torch.float32),
            "h_pos_rel": torch.tensor(h_pos_rel, dtype=torch.float32),
            "az_sin": torch.tensor(az_sin, dtype=torch.float32),
            "az_cos": torch.tensor(az_cos, dtype=torch.float32),
            "t_seg_tvt": torch.tensor(t_seg_tvt, dtype=torch.float32),
            "h_seg_tvt": torch.tensor(h_seg_tvt, dtype=torch.float32),
            "t_mask": torch.tensor(t_mask, dtype=torch.float32),
            "h_mask": torch.tensor(h_mask, dtype=torch.float32),
            "orig_tvt": torch.tensor(padded_tvt, dtype=torch.float32),
            "orig_len": torch.tensor(orig_len, dtype=torch.int64),
            "h_ps": torch.tensor(h_ps, dtype=torch.int64),
            "last_tvt_exact": torch.tensor(last_tvt_exact, dtype=torch.float32),
            "cutoff_mode": torch.tensor(1 if cutoff_mode == "random_cutoff" else 0, dtype=torch.int64),
        }


# ============================================================
# Model
# ============================================================

class ResidualConvBlock(nn.Module):
    def __init__(self, ci, co, groups=None):
        super().__init__()
        g = min(groups or Config.GN_GROUPS, co)
        while co % g != 0 and g > 1:
            g -= 1
        self.conv1 = nn.Conv2d(ci, co, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(g, co)
        self.conv2 = nn.Conv2d(co, co, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(g, co)
        self.act = nn.GELU()
        self.skip = nn.Conv2d(ci, co, 1, bias=False) if ci != co else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + residual)


class HorizontalDilatedASPP(nn.Module):
    def __init__(self, ci, co, dilations=None, groups=None):
        super().__init__()
        dilations = dilations or Config.ASPP_DILATIONS
        g = min(groups or Config.GN_GROUPS, co)
        while co % g != 0 and g > 1:
            g -= 1
        n_branches = len(dilations)
        branch_ch = co // n_branches
        self.branches = nn.ModuleList()
        for d in dilations:
            bg = min(g, branch_ch)
            while branch_ch % bg != 0 and bg > 1:
                bg -= 1
            self.branches.append(nn.Sequential(
                nn.Conv2d(ci, branch_ch, 3, padding=(1, d), dilation=(1, d), bias=False),
                nn.GroupNorm(bg, branch_ch),
                nn.GELU(),
            ))
        bg_pool = min(g, branch_ch)
        while branch_ch % bg_pool != 0 and bg_pool > 1:
            bg_pool -= 1
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ci, branch_ch, 1, bias=False),
            nn.GroupNorm(bg_pool, branch_ch),
            nn.GELU(),
        )
        total_ch = branch_ch * (n_branches + 1)
        fg = min(g, co)
        while co % fg != 0 and fg > 1:
            fg -= 1
        self.fuse = nn.Sequential(
            nn.Conv2d(total_ch, co, 1, bias=False),
            nn.GroupNorm(fg, co),
            nn.GELU(),
        )

    def forward(self, x):
        outs = [branch(x) for branch in self.branches]
        gp = self.global_pool(x)
        gp = gp.expand(-1, -1, x.shape[2], x.shape[3])
        outs.append(gp)
        return self.fuse(torch.cat(outs, dim=1))


class HeatmapResUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.gr_norm = nn.InstanceNorm2d(2)

        # Encoder
        self.e0 = ResidualConvBlock(9, base)
        self.e1 = ResidualConvBlock(base, base * 2)
        self.e2 = ResidualConvBlock(base * 2, base * 4)
        self.e3 = ResidualConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        # ASPP bottleneck on pooled encoder feature. No spatial attention in H-Base0.
        self.bott = HorizontalDilatedASPP(base * 8, base * 16)

        # Decoder
        self.u3 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.d3 = ResidualConvBlock(base * 16, base * 8)
        self.u2 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.d2 = ResidualConvBlock(base * 8, base * 4)
        self.u1 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.d1 = ResidualConvBlock(base * 4, base * 2)
        self.u0 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.d0 = ResidualConvBlock(base * 2, base)

        # Head
        self.head = nn.Conv2d(base, 1, 1)
        self.aux3 = nn.Conv2d(base * 8, 1, 1)
        self.aux2 = nn.Conv2d(base * 4, 1, 1)
        self.aux1 = nn.Conv2d(base * 2, 1, 1)

    def build_image(self, batch):
        dev = next(self.parameters()).device
        t_gr = batch["t_gr"].to(dev)
        h_gr = batch["h_gr"].to(dev)
        hist_mask = batch["hist_mask"].to(dev)
        history = batch["history_line"].to(dev)
        h_dz = batch["h_dz"].to(dev)
        B, T = t_gr.shape
        _, H = h_gr.shape
        t_img = t_gr.view(B, 1, T, 1).expand(B, 1, T, H)
        h_img = h_gr.view(B, 1, 1, H).expand(B, 1, T, H)
        gr_pair = self.gr_norm(torch.cat([t_img, h_img], dim=1))
        gr_diff = torch.clamp((t_img - h_img) / 40.0, -4.0, 4.0)
        mask_img = hist_mask.view(B, 1, 1, H).expand(B, 1, T, H)
        history_img = history[:, None, :, :]
        dz_img = h_dz.view(B, 1, 1, H).expand(B, 1, T, H)
        h_pos_rel = batch["h_pos_rel"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_sin = batch["az_sin"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_cos = batch["az_cos"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        return torch.cat([gr_pair, gr_diff, mask_img, history_img, dz_img, h_pos_rel, az_sin, az_cos], dim=1)

    def forward(self, batch):
        x = self.build_image(batch)

        # Encoder
        e0 = self.e0(x)
        e1 = self.e1(self.pool(e0))
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))

        # Bottleneck: pool -> ASPP
        pooled_e3 = self.pool(e3)
        b = self.bott(pooled_e3)

        # Decoder
        d3 = self.d3(torch.cat([self.u3(b), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        d0 = self.d0(torch.cat([self.u0(d1), e0], dim=1))

        logits = self.head(d0)

        if self.training:
            full_h, full_w = x.shape[2], x.shape[3]
            a3 = self.aux3(d3)
            a2 = self.aux2(d2)
            a1 = self.aux1(d1)
            a3 = F.interpolate(a3, size=(full_h, full_w), mode="bilinear", align_corners=False)
            a2 = F.interpolate(a2, size=(full_h, full_w), mode="bilinear", align_corners=False)
            a1 = F.interpolate(a1, size=(full_h, full_w), mode="bilinear", align_corners=False)
            return logits, a1, a2, a3
        return logits


# ============================================================
# Loss
# ============================================================
def make_heatmap_target(logits, batch):
    """Pure Gaussian TVT-distance heatmap target."""
    dev = logits.device
    t_seg_tvt = batch["t_seg_tvt"].to(dev).float()  # [B, T]
    h_seg_tvt = batch["h_seg_tvt"].to(dev).float()  # [B, H]
    t_mask = batch["t_mask"].to(dev).float()        # [B, T]
    h_mask = batch["h_mask"].to(dev).float()        # [B, H]

    dist_ft = torch.abs(t_seg_tvt[:, None, :, None] - h_seg_tvt[:, None, None, :])
    sigma = float(Config.HEATMAP_TARGET_SIGMA_FT)
    target = torch.exp(-(dist_ft ** 2) / (2.0 * sigma * sigma))

    mask = t_mask[:, None, :, None] * h_mask[:, None, None, :]
    target = target * mask
    target = target / target.sum(dim=2, keepdim=True).clamp_min(1e-8)
    valid_col = (h_mask[:, None, :] > 0).float()  # [B, 1, H]
    return target, valid_col, t_mask


def heatmap_ce_loss_single(logits, batch):
    """Column-wise soft cross entropy over the T dimension."""
    target, valid_col, t_mask = make_heatmap_target(logits, batch)
    logits = logits.float()
    logits = logits.masked_fill(t_mask[:, None, :, None] <= 0, -1e4)
    log_prob = F.log_softmax(logits, dim=2)
    ce = -(target * log_prob).sum(dim=2)  # [B, 1, H]
    return (ce * valid_col).sum() / (valid_col.sum() + 1e-8)


def heatmap_ce_loss(model_out, batch):
    """Main heatmap CE plus fixed deep supervision CE."""
    if isinstance(model_out, tuple):
        logits_main, a1, a2, a3 = model_out
        w1, w2, w3 = Config.DS_WEIGHTS
        return (
            heatmap_ce_loss_single(logits_main, batch)
            + w1 * heatmap_ce_loss_single(a1, batch)
            + w2 * heatmap_ce_loss_single(a2, batch)
            + w3 * heatmap_ce_loss_single(a3, batch)
        )
    return heatmap_ce_loss_single(model_out, batch)


# ============================================================
# Decode / Eval
# ============================================================
def map_to_original(pred_H, orig_len, h_ps, last_tvt_exact):
    centers = []
    values = []
    for k in range(Config.H_H):
        centers.append(h_ps - k * Config.H_S - (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H - 1 - k])
    centers.append(float(h_ps))
    values.append(float(last_tvt_exact))
    for k in range(Config.H_F):
        centers.append(h_ps + 1 + k * Config.H_S + (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H + k])
    centers = np.asarray(centers, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    order = np.argsort(centers)
    centers = centers[order]
    values = values[order]
    keep = np.r_[True, np.diff(centers) > 1e-6]
    return np.interp(np.arange(orig_len), centers[keep], values[keep])


def decode_heatmap_soft(logits, t_seg_tvt, h_seg_tvt, t_mask, orig_len, h_ps, last_tvt_exact):
    score = logits.astype(np.float64) / float(Config.HEATMAP_DECODE_TEMP)
    if t_mask is not None:
        score = np.where(t_mask[:, None] > 0, score, -1e9)
    score = score - score.max(axis=0, keepdims=True)
    prob = np.exp(score)
    prob = prob / (prob.sum(axis=0, keepdims=True) + 1e-12)
    pred_H = (prob * t_seg_tvt[:, None]).sum(axis=0).astype(np.float64)
    pred_H[:Config.H_H] = h_seg_tvt[:Config.H_H]
    return map_to_original(pred_H, orig_len, h_ps, last_tvt_exact)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    se = 0.0
    n = 0
    for batch in loader:
        logits = model(batch)
        logits = logits.squeeze(1).cpu().numpy()
        t_seg = batch["t_seg_tvt"].numpy()
        h_seg = batch["h_seg_tvt"].numpy()
        t_mask = batch["t_mask"].numpy()
        y = batch["orig_tvt"].numpy()
        orig_len = batch["orig_len"].numpy()
        h_ps = batch["h_ps"].numpy()
        last = batch["last_tvt_exact"].numpy()
        for b in range(logits.shape[0]):
            olen = int(orig_len[b])
            ps = int(h_ps[b])
            if ps >= olen - 1:
                continue
            pred = decode_heatmap_soft(
                logits[b], t_seg[b], h_seg[b], t_mask[b],
                olen, ps, float(last[b]),
            )
            err = pred[ps + 1:] - y[b, :olen][ps + 1:]
            se += np.sum(err ** 2)
            n += len(err)
    return np.sqrt(se / max(n, 1))


@torch.no_grad()
def evaluate_last_tvt(loader):
    se = 0.0
    n = 0
    for batch in loader:
        y = batch["orig_tvt"].numpy()
        orig_len = batch["orig_len"].numpy()
        h_ps = batch["h_ps"].numpy()
        for b in range(len(orig_len)):
            olen = int(orig_len[b])
            ps = int(h_ps[b])
            if ps >= olen - 1:
                continue
            true = y[b, :olen]
            err = true[ps] - true[ps + 1:]
            se += np.sum(err ** 2)
            n += len(err)
    return np.sqrt(se / max(n, 1))


# ============================================================
# Train / Inference
# ============================================================
def typewell_hash(horizontal_file):
    sid = horizontal_file.name.split("__")[0]
    tw = horizontal_file.parent / f"{sid}{Config.TYPEWELL_SUFFIX}"
    try:
        gr = np.round(pd.read_csv(tw)["GR"].values.astype(float), 2)
        return hashlib.md5(gr.tobytes()).hexdigest()
    except Exception:
        return horizontal_file.name


def make_loader(files, is_train=True):
    return DataLoader(
        HeatmapDataset(files, is_train=is_train),
        batch_size=Config.BATCH_SIZE, shuffle=is_train,
        num_workers=Config.NUM_WORKERS, pin_memory=True,
    )


def train_one_fold(fold, train_idx, val_idx, files):
    cfg = Config
    train_files = [files[i] for i in train_idx]
    val_files = [files[i] for i in val_idx]
    val_loader = make_loader(val_files, is_train=False)
    print("Last TVT RMSE:", evaluate_last_tvt(val_loader))
    print("train files:", len(train_files), "val files:", len(val_files))

    model = HeatmapResUNet(base=16).to(cfg.DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"model params: {n_params:,}")

    train_loader = make_loader(train_files, is_train=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
    use_amp = cfg.DEVICE.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    print(f"\naug_repeats: {cfg.AUG_REPEATS}  epochs: {cfg.EPOCHS}  lr: {cfg.LR}")
    print(f"START_JITTER={cfg.START_JITTER}  RANDOM_CUTOFF_PROB={cfg.RANDOM_CUTOFF_PROB}"
          f"  RANDOM_CUTOFF_FRAC=[{cfg.RANDOM_CUTOFF_MIN_FRAC}, {cfg.RANDOM_CUTOFF_MAX_FRAC}]")
    print(f"deep supervision: ON  weights: {cfg.DS_WEIGHTS}")
    print(f"pure Gaussian heatmap target: sigma_ft={cfg.HEATMAP_TARGET_SIGMA_FT}")
    print(f"heatmap decode_temp={cfg.HEATMAP_DECODE_TEMP}")
    print("loss: heatmap CE only")
    print("architecture: Basic ResUNet-GN + Horizontal Dilated ASPP ")

    ckpt_path = f"fold_{fold}_best.pth"
    best = 1e9

    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        total = 0.0
        n_batches = 0
        rc_count = 0
        sample_count = 0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                model_out = model(batch)
                loss = heatmap_ce_loss(model_out, batch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total += float(loss.item())
            n_batches += 1
            if "cutoff_mode" in batch:
                rc_count += int(batch["cutoff_mode"].sum().item())
                sample_count += int(batch["cutoff_mode"].numel())
        scheduler.step()

        val_rmse = evaluate(model, val_loader)
        tag = ""
        if val_rmse < best:
            best = val_rmse
            tag = "  saved"
            torch.save(model.state_dict(), ckpt_path)
        rc_frac = rc_count / max(sample_count, 1)
        print(
            f"fold {fold} | ep {epoch:02d}/{cfg.EPOCHS:02d} | "
            f"train {total / max(n_batches, 1):.5f} | "
            f"rc {rc_frac:.2f} | "
            f"val {val_rmse:.4f}{tag}"
        )
    return best


def run_training():
    files = sorted(Path(Config.TRAIN_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    groups = [typewell_hash(f) for f in files]
    gkf = GroupKFold(n_splits=Config.N_FOLDS)
    scores = []
    for fold, (train_idx, val_idx) in enumerate(gkf.split(files, groups=groups)):
        if fold not in Config.FOLDS_TO_RUN:
            continue
        print(f"\n===== Fold {fold} =====")
        print("train:", len(train_idx), "val:", len(val_idx))
        score = train_one_fold(fold, train_idx, val_idx, files)
        scores.append(score)
        print(f"Fold {fold} best: {score:.4f}")
    if scores:
        print("\nCV RMSE:", float(np.mean(scores)))
        print("fold scores:", scores)


@torch.no_grad()
def run_inference():
    test_files = sorted(Path(Config.TEST_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    ckpts = sorted(Path(".").glob("fold_*_best.pth"))
    if not test_files or not ckpts:
        print("inference skipped")
        print("test_files:", len(test_files), "ckpts:", len(ckpts))
        return
    models = []
    for ckpt in ckpts:
        model = HeatmapResUNet(base=16).to(Config.DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=Config.DEVICE))
        model.eval()
        models.append(model)
        print("loaded", ckpt.name)

    ds = HeatmapDataset(test_files, is_train=False)
    preds = {}
    for i, f in enumerate(test_files):
        sid = f.name.split("__")[0]
        sample = ds[i]
        batch = {k: v.unsqueeze(0) for k, v in sample.items()}
        logits_list = []
        for model in models:
            logits = model(batch).squeeze(1).float().cpu().numpy()[0]
            logits_list.append(logits)
        avg_logits = np.mean(logits_list, axis=0)
        pred = decode_heatmap_soft(
            avg_logits,
            sample["t_seg_tvt"].numpy(), sample["h_seg_tvt"].numpy(), sample["t_mask"].numpy(),
            int(sample["orig_len"]), int(sample["h_ps"]), float(sample["last_tvt_exact"]),
        )
        df = pd.read_csv(f)
        if "TVT_input" in df.columns:
            rows = np.flatnonzero(df["TVT_input"].isna().values)
        else:
            rows = np.arange(len(df))
        for r in rows:
            preds[f"{sid}_{int(r)}"] = float(pred[int(r)])
        print("done", sid, "submit rows:", len(rows))

    sub_path = Path(Config.TRAIN_DIR).parent / "sample_submission.csv"
    if sub_path.exists():
        sub = pd.read_csv(sub_path)[["id"]].copy()
        sub["tvt"] = sub["id"].map(preds).fillna(0.0).astype(np.float32)
    else:
        sub = pd.DataFrame({"id": list(preds.keys()), "tvt": list(preds.values())})
    sub.to_csv("submission.csv", index=False)
    print("saved submission.csv", sub.shape)
    print(sub.head())


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    run_training()
    run_inference()

In [ ]:
# ============================================================
# ROGII - SDF ResUNet-noGN + Horizontal Dilated ASPP
#         + ConvNeXt Bottleneck + ECA
#         + Deep Supervision + Soft Zero-Level Decode
#         + Zero-Level CE / Ranking Loss
# ============================================================
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Config
# ============================================================
class Config:
    TRAIN_DIR = "/kaggle/input/datasets/zhuyifanss/roggi-data/train"
    TEST_DIR = "/kaggle/input/datasets/zhuyifanss/roggi-data/test"
    TYPEWELL_SUFFIX = "__typewell.csv"
    HORIZONTAL_SUFFIX = "__horizontal_well.csv"

    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    SEED = 42
    NUM_WORKERS = 12
    BATCH_SIZE = 4
    EPOCHS = 10
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    N_FOLDS = 4
    FOLDS_TO_RUN = [0]

    # Grid
    H_S = 16
    H_H = 64
    H_F = 704
    T_H = 128
    T_F = 128
    H_GR_FILTER = 51

    # SDF settings.
    # Target is a signed TVT-distance field:
    #   sdf[t,h] = clip((typewell_tvt[t] - horizontal_tvt[h]) / scale_ft)
    # The correct matching path is the zero-level set.
    SDF_TARGET_SCALE_FT = 32.0
    SDF_TARGET_CLIP = 4.0
    SDF_NEAR_SIGMA_FT = 16.0
    SDF_NEAR_WEIGHT = 2.0
    SDF_DECODE_TEMP = 0.25
    HISTORY_SIGMA = 2.0

    ORIG_PAD_LEN = 16384

    # ---- Zero-level CE / ranking loss ----
    # For each future horizontal column, treat -|sdf|/temp as row-logits and
    # supervise the true crossing row (argmin |t_tvt - h_tvt|) with CE.
    # This directly attacks "false zeros": it forces the true row to be the
    # smallest |sdf|, i.e. lifts gt_topk_hit.
    ZERO_CE_WEIGHT = 0.3
    ZERO_CE_TEMP = 0.25
    ZERO_CE_MAX_DIST_FT = 8.0     # only supervise columns whose true crossing is inside the typewell window
    ZERO_CE_LABEL_SIGMA_FT = 0  # 0 -> hard CE (Yifan's spec); >0 -> Gaussian soft label over rows
    ZERO_CE_TOPK = 9              # gt_top-k hit monitoring in evaluate()

    # Core augmentation
    START_JITTER = 512
    RANDOM_CUTOFF_PROB = 0.5
    RANDOM_CUTOFF_MIN_FRAC = 0.2
    RANDOM_CUTOFF_MAX_FRAC = 0.8
    RANDOM_CUTOFF_MIN_HISTORY = 512
    RANDOM_CUTOFF_MIN_FUTURE = 512
    # HFlip is intentionally removed for this simple SDF baseline.
    AUG_REPEATS = 4

    # Trajectory Z
    DZ_SCALE = 2.0

    # Deep Supervision weights
    DS_WEIGHTS = [0.4, 0.2, 0.1]

    # ASPP dilation rates (along H-axis)
    ASPP_DILATIONS = (1, 4, 8, 16)



np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
print("DEVICE:", Config.DEVICE)


# ============================================================
# Preprocess utils
# ============================================================
def resample_typewell(t, target_step=0.5):
    t_tvt = t["TVT"].values.astype(np.float64)
    t_gr = t["GR"].values.astype(np.float64)
    diffs = np.abs(np.diff(t_tvt))
    diffs = diffs[diffs > 0]
    ratio = (np.median(diffs) if len(diffs) else target_step) / target_step
    if np.isclose(ratio, 1.0):
        return t_tvt, t_gr
    if ratio < 1.0:
        group = max(int(round(1 / ratio)), 1)
        pad = (-len(t_tvt)) % group
        if pad:
            t_tvt = np.pad(t_tvt, (0, pad), mode="edge")
            t_gr = np.pad(t_gr, (0, pad), mode="edge")
        return t_tvt.reshape(-1, group).mean(1), t_gr.reshape(-1, group).mean(1)
    up = max(int(round(ratio)), 1)
    old = np.arange(len(t_tvt))
    new = np.linspace(0, len(t_tvt) - 1, (len(t_tvt) - 1) * up + 1)
    return np.interp(new, old, t_tvt), np.interp(new, old, t_gr)


def bin_mean(arr, step, back):
    arr = np.asarray(arr)
    if len(arr) == 0:
        return arr
    pad = (-len(arr)) % step
    if pad < step // 2:
        if pad:
            arr = np.pad(arr, (0, pad) if back else (pad, 0), mode="edge")
    elif pad:
        arr = arr[:-(step - pad)] if back else arr[(step - pad):]
    if len(arr) == 0:
        return arr
    return arr.reshape(-1, step).mean(1)


def safe_savgol(x, win=51, poly=2):
    x = np.asarray(x, dtype=float)
    if len(x) <= win:
        return x
    win = min(win, len(x))
    if win % 2 == 0:
        win -= 1
    if win < 7:
        return x
    return savgol_filter(x, win, poly)


def _get_numeric_col(h, name, default_value=0.0):
    if name in h.columns:
        return (
            h[name]
            .astype(float)
            .interpolate()
            .bfill()
            .ffill()
            .fillna(default_value)
            .values
        )
    return np.full(len(h), float(default_value), dtype=np.float64)


def resample_horizontal(h, step, offset=0, forced_h_ps=None):
    gr_raw = (
        h["GR"]
        .astype(float)
        .interpolate()
        .bfill()
        .ffill()
        .fillna(85.0)
        .values
    )
    gr = safe_savgol(gr_raw, Config.H_GR_FILTER, 2)

    if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
        tvt = h["TVT"].values.astype(float)
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        tvt = (
            h["TVT_input"]
            .astype(float)
            .ffill()
            .bfill()
            .fillna(0.0)
            .values
        )
    else:
        tvt = h["Z"].values.astype(float)

    z = _get_numeric_col(h, "Z", 0.0)
    md = _get_numeric_col(h, "MD", 0.0)
    if "MD" not in h.columns:
        md = np.arange(len(h), dtype=np.float64)
    x = _get_numeric_col(h, "X", 0.0)
    y = _get_numeric_col(h, "Y", 0.0)

    if forced_h_ps is not None:
        h_ps = int(forced_h_ps)
        h_ps = max(0, min(h_ps, len(h) - 1))
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        h_ps = int(np.flatnonzero(h["TVT_input"].notna().values)[-1]) + offset
        h_ps = max(0, min(h_ps, len(h) - 1))
    else:
        h_ps = len(h) // 2

    tvt0 = bin_mean(tvt[:h_ps + 1], step, back=False)
    gr0 = bin_mean(gr[:h_ps + 1], step, back=False)
    z0 = bin_mean(z[:h_ps + 1], step, back=False)
    md0 = bin_mean(md[:h_ps + 1], step, back=False)
    x0 = bin_mean(x[:h_ps + 1], step, back=False)
    y0 = bin_mean(y[:h_ps + 1], step, back=False)

    tvt1 = bin_mean(tvt[h_ps + 1:], step, back=True)
    gr1 = bin_mean(gr[h_ps + 1:], step, back=True)
    z1 = bin_mean(z[h_ps + 1:], step, back=True)
    md1 = bin_mean(md[h_ps + 1:], step, back=True)
    x1 = bin_mean(x[h_ps + 1:], step, back=True)
    y1 = bin_mean(y[h_ps + 1:], step, back=True)

    return tvt0, gr0, tvt1, gr1, h_ps, z0, z1, md0, md1, x0, x1, y0, y1


def crop_pad_1d(n, center, history, future):
    raw_i0 = center - history
    raw_i1 = center + future
    i0 = max(raw_i0, 0)
    i1 = min(raw_i1, n)
    pad_left = max(0, -raw_i0)
    pad_right = max(0, raw_i1 - n)
    return i0, i1, pad_left, pad_right


def get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt):
    if "TVT_input" in h.columns:
        val = h["TVT_input"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if "TVT" in h.columns:
        val = h["TVT"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if len(h_tvt0):
        return float(h_tvt0[-1])
    return float(t_tvt[len(t_tvt) // 2])


def get_official_h_ps(h):
    if "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        return int(np.flatnonzero(h["TVT_input"].notna().values)[-1])
    return len(h) // 2


def choose_train_cutoff(h, cfg, start_jitter=None):
    n = len(h)
    official_ps = get_official_h_ps(h)
    sj = cfg.START_JITTER if start_jitter is None else int(start_jitter)

    can_random = (
        "TVT" in h.columns
        and h["TVT"].notna().sum() > max(32, cfg.RANDOM_CUTOFF_MIN_HISTORY // 4)
        and n > cfg.RANDOM_CUTOFF_MIN_HISTORY + cfg.RANDOM_CUTOFF_MIN_FUTURE + 8
    )

    if can_random and np.random.rand() < cfg.RANDOM_CUTOFF_PROB:
        lo = max(
            int(round(n * cfg.RANDOM_CUTOFF_MIN_FRAC)),
            int(cfg.RANDOM_CUTOFF_MIN_HISTORY),
            1,
        )
        hi = min(
            int(round(n * cfg.RANDOM_CUTOFF_MAX_FRAC)),
            n - 1 - int(cfg.RANDOM_CUTOFF_MIN_FUTURE),
        )
        if hi > lo:
            return int(np.random.randint(lo, hi + 1)), "random_cutoff"

    offset = 0
    if sj > 0:
        offset = -np.random.randint(0, sj + 1)
    h_ps = int(np.clip(official_ps + offset, 0, n - 1))
    return h_ps, "official_jitter"


def make_history_line(t_seg_tvt, h_seg_tvt, hist_mask):
    diff = np.abs(t_seg_tvt[:, None] - h_seg_tvt[None, :])
    matched = diff.argmin(axis=0)
    rows = np.arange(len(t_seg_tvt))[:, None]
    line = np.exp(
        -0.5 * ((rows - matched[None, :]) / Config.HISTORY_SIGMA) ** 2
    ).astype(np.float32)
    line *= hist_mask[None, :]
    return line



# ============================================================
# Dataset
# ============================================================
class HeatmapDataset(Dataset):
    def __init__(self, files, is_train=True, aug_repeats=None, start_jitter=None):
        self.files = list(files)
        self.is_train = is_train
        self.aug_repeats = int(aug_repeats if aug_repeats is not None else (Config.AUG_REPEATS if is_train else 1))
        self.start_jitter = int(start_jitter if start_jitter is not None else Config.START_JITTER)

    def __len__(self):
        if self.is_train:
            return len(self.files) * self.aug_repeats
        return len(self.files)

    def __getitem__(self, idx):
        cfg = Config
        if self.is_train:
            idx = idx % len(self.files)
        hp = Path(self.files[idx])
        sid = hp.name.split("__")[0]
        h = pd.read_csv(hp)
        t = pd.read_csv(hp.parent / f"{sid}{cfg.TYPEWELL_SUFFIX}")
        t_tvt, t_gr = resample_typewell(t, target_step=0.5)

        forced_h_ps = None
        offset = 0
        if self.is_train:
            forced_h_ps, cutoff_mode = choose_train_cutoff(
                h, cfg, start_jitter=self.start_jitter,
            )
        else:
            cutoff_mode = "official"

        h_tvt0, h_gr0, h_tvt1, h_gr1, h_ps, h_z0, h_z1, h_md0, h_md1, h_x0, h_x1, h_y0, h_y1 = resample_horizontal(
            h, cfg.H_S, offset=offset, forced_h_ps=forced_h_ps,
        )

        last_tvt_exact = get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt)
        last_tvt = float(h_tvt0[-1]) if len(h_tvt0) else last_tvt_exact
        last_idx = int(np.abs(t_tvt - last_tvt).argmin()) if len(t_tvt) else 0
        last_z = float(h["Z"].iloc[h_ps])
        last_md = float(h["MD"].iloc[h_ps]) if "MD" in h.columns else float(h_ps)
        last_x = float(h["X"].iloc[h_ps]) if "X" in h.columns else 0.0
        last_y = float(h["Y"].iloc[h_ps]) if "Y" in h.columns else 0.0

        # typewell crop
        i0, i1, pl, pr = crop_pad_1d(len(t_tvt), last_idx + 1, cfg.T_H, cfg.T_F)
        t_mask = np.pad(np.ones(i1 - i0, dtype=np.float32), (pl, pr), constant_values=0.0)
        t_seg_tvt = np.pad(t_tvt[i0:i1], (pl, pr), mode="edge")
        t_seg_gr = np.pad(t_gr[i0:i1], (pl, pr), mode="edge")

        # horizontal history crop
        i0_h, i1_h, p0l, p0r = crop_pad_1d(len(h_tvt0), len(h_tvt0), cfg.H_H, 0)
        hm0 = np.pad(np.ones(i1_h - i0_h, dtype=np.float32), (p0l, p0r), constant_values=0.0)
        if len(h_tvt0):
            ht0 = np.pad(h_tvt0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hg0 = np.pad(h_gr0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hz0 = np.pad(h_z0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hmd0 = np.pad(h_md0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hx0 = np.pad(h_x0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hy0 = np.pad(h_y0[i0_h:i1_h], (p0l, p0r), mode="edge")
        else:
            ht0 = np.zeros(cfg.H_H, dtype=np.float64)
            hg0 = np.zeros(cfg.H_H, dtype=np.float64)
            hz0 = np.full(cfg.H_H, last_z, dtype=np.float64)
            hmd0 = np.full(cfg.H_H, last_md, dtype=np.float64)
            hx0 = np.full(cfg.H_H, last_x, dtype=np.float64)
            hy0 = np.full(cfg.H_H, last_y, dtype=np.float64)

        # horizontal future crop
        i0_f, i1_f, p1l, p1r = crop_pad_1d(len(h_tvt1), 0, 0, cfg.H_F)
        hm1 = np.pad(np.ones(i1_f - i0_f, dtype=np.float32), (p1l, p1r), constant_values=0.0)
        if len(h_tvt1):
            ht1 = np.pad(h_tvt1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hg1 = np.pad(h_gr1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hz1 = np.pad(h_z1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hmd1 = np.pad(h_md1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hx1 = np.pad(h_x1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hy1 = np.pad(h_y1[i0_f:i1_f], (p1l, p1r), mode="edge")
        else:
            ht1 = np.zeros(cfg.H_F, dtype=np.float64)
            hg1 = np.zeros(cfg.H_F, dtype=np.float64)
            hz1 = np.full(cfg.H_F, last_z, dtype=np.float64)
            hmd1 = np.full(cfg.H_F, last_md, dtype=np.float64)
            hx1 = np.full(cfg.H_F, last_x, dtype=np.float64)
            hy1 = np.full(cfg.H_F, last_y, dtype=np.float64)

        h_mask = np.concatenate([hm0, hm1]).astype(np.float32)
        h_seg_tvt = np.concatenate([ht0, ht1]).astype(np.float64)
        h_seg_gr = np.concatenate([hg0, hg1]).astype(np.float64)
        h_seg_z = np.concatenate([hz0, hz1]).astype(np.float64)
        h_seg_md = np.concatenate([hmd0, hmd1]).astype(np.float64)
        h_seg_x = np.concatenate([hx0, hx1]).astype(np.float64)
        h_seg_y = np.concatenate([hy0, hy1]).astype(np.float64)

        hist_mask = np.zeros(cfg.H_H + cfg.H_F, dtype=np.float32)
        hist_mask[:cfg.H_H] = hm0

        h_dz = np.gradient(h_seg_z)
        h_dz = np.clip(h_dz / cfg.DZ_SCALE, -3.0, 3.0).astype(np.float32)

        H = cfg.H_H + cfg.H_F
        col_center = np.zeros(H, dtype=np.float64)
        for j in range(cfg.H_H):
            k = cfg.H_H - 1 - j
            col_center[j] = h_ps - k * cfg.H_S - (cfg.H_S - 1) / 2.0
        for k in range(cfg.H_F):
            col_center[cfg.H_H + k] = h_ps + 1 + k * cfg.H_S + (cfg.H_S - 1) / 2.0
        h_pos_rel = np.clip((col_center - float(h_ps)) / 4096.0, -4.0, 4.0).astype(np.float32)

        dmd = np.gradient(h_seg_md)
        dmd = np.where(np.abs(dmd) < 1e-6, 1.0, dmd)
        dx_dmd = np.gradient(h_seg_x) / dmd
        dy_dmd = np.gradient(h_seg_y) / dmd
        dxy = np.sqrt(dx_dmd ** 2 + dy_dmd ** 2) + 1e-6
        az_cos = np.clip(dx_dmd / dxy, -1.0, 1.0).astype(np.float32)
        az_sin = np.clip(dy_dmd / dxy, -1.0, 1.0).astype(np.float32)

        history_line = make_history_line(t_seg_tvt, h_seg_tvt, hist_mask)

        if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
            orig_tvt = h["TVT"].values.astype(np.float32)
        else:
            orig_tvt = np.zeros(len(h), dtype=np.float32)

        orig_len = len(orig_tvt)
        padded_tvt = np.zeros(cfg.ORIG_PAD_LEN, dtype=np.float32)
        ncopy = min(orig_len, cfg.ORIG_PAD_LEN)
        padded_tvt[:ncopy] = orig_tvt[:ncopy]

        return {
            "t_gr": torch.tensor(t_seg_gr, dtype=torch.float32),
            "h_gr": torch.tensor(h_seg_gr, dtype=torch.float32),
            "hist_mask": torch.tensor(hist_mask, dtype=torch.float32),
            "history_line": torch.tensor(history_line, dtype=torch.float32),
            "h_dz": torch.tensor(h_dz, dtype=torch.float32),
            "h_pos_rel": torch.tensor(h_pos_rel, dtype=torch.float32),
            "az_sin": torch.tensor(az_sin, dtype=torch.float32),
            "az_cos": torch.tensor(az_cos, dtype=torch.float32),
            "t_seg_tvt": torch.tensor(t_seg_tvt, dtype=torch.float32),
            "h_seg_tvt": torch.tensor(h_seg_tvt, dtype=torch.float32),
            "t_mask": torch.tensor(t_mask, dtype=torch.float32),
            "h_mask": torch.tensor(h_mask, dtype=torch.float32),
            "orig_tvt": torch.tensor(padded_tvt, dtype=torch.float32),
            "orig_len": torch.tensor(orig_len, dtype=torch.int64),
            "h_ps": torch.tensor(h_ps, dtype=torch.int64),
            "last_tvt_exact": torch.tensor(last_tvt_exact, dtype=torch.float32),
            "cutoff_mode": torch.tensor(1 if cutoff_mode == "random_cutoff" else 0, dtype=torch.int64),
        }


# ============================================================
# Model
# ============================================================

class ResidualConvBlock(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.conv1 = nn.Conv2d(ci, co, 3, padding=1, bias=False)
        self.norm1 = nn.Identity()
        self.conv2 = nn.Conv2d(co, co, 3, padding=1, bias=False)
        self.norm2 = nn.Identity()
        self.act = nn.GELU()
        self.skip = nn.Conv2d(ci, co, 1, bias=False) if ci != co else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + residual)



class LayerNorm2d(nn.Module):
    """Channel-wise LayerNorm for [B, C, T, H]."""
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(c))
        self.bias = nn.Parameter(torch.zeros(c))
        self.eps = eps

    def forward(self, x):
        u = x.mean(dim=1, keepdim=True)
        s = (x - u).pow(2).mean(dim=1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return x * self.weight[:, None, None] + self.bias[:, None, None]


class ConvNeXt2DBlock(nn.Module):
    """
    Small ConvNeXt-style residual bottleneck block.
    This is applied only after ASPP, so it refines low-resolution global features
    without changing the SDF output format or decode method.
    """
    def __init__(self, c, expansion=4, kernel=7):
        super().__init__()
        self.dw = nn.Conv2d(c, c, kernel, padding=kernel // 2, groups=c, bias=True)
        self.norm = LayerNorm2d(c)
        self.pw1 = nn.Conv2d(c, c * expansion, 1, bias=True)
        self.act = nn.GELU()
        self.pw2 = nn.Conv2d(c * expansion, c, 1, bias=True)
        # Zero-init residual scale keeps the initial network close to the original ASPP baseline.
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, x):
        y = self.dw(x)
        y = self.norm(y)
        y = self.pw2(self.act(self.pw1(y)))
        return x + self.gamma * y


class ECABlock2D(nn.Module):
    """
    Efficient Channel Attention.
    Adds almost no parameters; useful as channel reweighting after ConvNeXt bottleneck.
    """
    def __init__(self, c, k_size=5):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        # Zero-init residual scale keeps the initial network close to ConvNeXt-only.
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        y = self.avg(x).squeeze(-1).transpose(1, 2)  # [B, 1, C]
        y = self.conv(y).transpose(1, 2).unsqueeze(-1).sigmoid()
        return x + self.gamma * (x * y)


class HorizontalDilatedASPP(nn.Module):
    def __init__(self, ci, co, dilations=None):
        super().__init__()
        dilations = dilations or Config.ASPP_DILATIONS
        n_branches = len(dilations)
        branch_ch = co // n_branches
        self.branches = nn.ModuleList()
        for d in dilations:
            self.branches.append(nn.Sequential(
                nn.Conv2d(ci, branch_ch, 3, padding=(1, d), dilation=(1, d), bias=False),
                nn.Identity(),
                nn.GELU(),
            ))
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ci, branch_ch, 1, bias=False),
            nn.Identity(),
            nn.GELU(),
        )
        total_ch = branch_ch * (n_branches + 1)
        self.fuse = nn.Sequential(
            nn.Conv2d(total_ch, co, 1, bias=False),
            nn.Identity(),
            nn.GELU(),
        )

    def forward(self, x):
        outs = [branch(x) for branch in self.branches]
        gp = self.global_pool(x)
        gp = gp.expand(-1, -1, x.shape[2], x.shape[3])
        outs.append(gp)
        return self.fuse(torch.cat(outs, dim=1))


class HeatmapResUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.gr_norm = nn.InstanceNorm2d(2)

        # Encoder
        self.e0 = ResidualConvBlock(9, base)
        self.e1 = ResidualConvBlock(base, base * 2)
        self.e2 = ResidualConvBlock(base * 2, base * 4)
        self.e3 = ResidualConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        # ASPP bottleneck on pooled encoder feature.
        self.bott = HorizontalDilatedASPP(base * 8, base * 16)

        # Fixed convnext_eca bottleneck refinement.
        # Search result: ConvNeXt bottleneck + ECA is a simple, strong architecture change.
        self.cn_bott = ConvNeXt2DBlock(base * 16, kernel=7)
        self.eca_bott = ECABlock2D(base * 16, k_size=5)

        # Decoder
        self.u3 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.d3 = ResidualConvBlock(base * 16, base * 8)
        self.u2 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.d2 = ResidualConvBlock(base * 8, base * 4)
        self.u1 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.d1 = ResidualConvBlock(base * 4, base * 2)
        self.u0 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.d0 = ResidualConvBlock(base * 2, base)

        # Head
        self.head = nn.Conv2d(base, 1, 1)
        self.aux3 = nn.Conv2d(base * 8, 1, 1)
        self.aux2 = nn.Conv2d(base * 4, 1, 1)
        self.aux1 = nn.Conv2d(base * 2, 1, 1)

    def build_image(self, batch):
        dev = next(self.parameters()).device
        t_gr = batch["t_gr"].to(dev)
        h_gr = batch["h_gr"].to(dev)
        hist_mask = batch["hist_mask"].to(dev)
        history = batch["history_line"].to(dev)
        h_dz = batch["h_dz"].to(dev)
        B, T = t_gr.shape
        _, H = h_gr.shape
        t_img = t_gr.view(B, 1, T, 1).expand(B, 1, T, H)
        h_img = h_gr.view(B, 1, 1, H).expand(B, 1, T, H)
        gr_pair = self.gr_norm(torch.cat([t_img, h_img], dim=1))
        gr_diff = torch.clamp((t_img - h_img) / 40.0, -4.0, 4.0)
        mask_img = hist_mask.view(B, 1, 1, H).expand(B, 1, T, H)
        history_img = history[:, None, :, :]
        dz_img = h_dz.view(B, 1, 1, H).expand(B, 1, T, H)
        h_pos_rel = batch["h_pos_rel"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_sin = batch["az_sin"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_cos = batch["az_cos"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        return torch.cat([gr_pair, gr_diff, mask_img, history_img, dz_img, h_pos_rel, az_sin, az_cos], dim=1)

    def forward(self, batch):
        x = self.build_image(batch)

        # Encoder
        e0 = self.e0(x)
        e1 = self.e1(self.pool(e0))
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))

        # Bottleneck: pool -> ASPP -> ConvNeXt -> ECA
        pooled_e3 = self.pool(e3)
        b = self.bott(pooled_e3)
        b = self.cn_bott(b)
        b = self.eca_bott(b)

        # Decoder
        d3 = self.d3(torch.cat([self.u3(b), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        d0 = self.d0(torch.cat([self.u0(d1), e0], dim=1))

        logits = self.head(d0)

        if self.training:
            full_h, full_w = x.shape[2], x.shape[3]
            a3 = self.aux3(d3)
            a2 = self.aux2(d2)
            a1 = self.aux1(d1)
            a3 = F.interpolate(a3, size=(full_h, full_w), mode="bilinear", align_corners=False)
            a2 = F.interpolate(a2, size=(full_h, full_w), mode="bilinear", align_corners=False)
            a1 = F.interpolate(a1, size=(full_h, full_w), mode="bilinear", align_corners=False)
            return logits, a1, a2, a3
        return logits


# ============================================================
# Loss
# ============================================================
def make_sdf_target(logits, batch):
    """Signed TVT-distance field target."""
    dev = logits.device
    t_seg_tvt = batch["t_seg_tvt"].to(dev).float()  # [B, T]
    h_seg_tvt = batch["h_seg_tvt"].to(dev).float()  # [B, H]
    t_mask = batch["t_mask"].to(dev).float()        # [B, T]
    h_mask = batch["h_mask"].to(dev).float()        # [B, H]

    signed_ft = t_seg_tvt[:, None, :, None] - h_seg_tvt[:, None, None, :]
    target = signed_ft / float(Config.SDF_TARGET_SCALE_FT)
    target = target.clamp(-float(Config.SDF_TARGET_CLIP), float(Config.SDF_TARGET_CLIP))
    mask = t_mask[:, None, :, None] * h_mask[:, None, None, :]
    abs_dist = signed_ft.abs()
    near_weight = 1.0 + float(Config.SDF_NEAR_WEIGHT) * torch.exp(
        -(abs_dist ** 2) / (2.0 * float(Config.SDF_NEAR_SIGMA_FT) ** 2)
    )
    return target, mask, near_weight


def sdf_loss_single(pred_sdf, batch):
    """Weighted SmoothL1 regression of the signed distance field."""
    target, mask, near_weight = make_sdf_target(pred_sdf, batch)
    pred_sdf = pred_sdf.float().clamp(
        -float(Config.SDF_TARGET_CLIP) * 1.5,
        float(Config.SDF_TARGET_CLIP) * 1.5,
    )
    loss = F.smooth_l1_loss(pred_sdf, target, reduction="none", beta=0.25)
    weight = mask * near_weight
    return (loss * weight).sum() / (weight.sum() + 1e-8)


def zero_ce_loss_single(pred_sdf, batch):
    """
    Zero-level CE / ranking loss.

    For every future horizontal column h we build row-logits over the typewell
    axis: score[t] = -|sdf[t, h]| / temp. The label is the true crossing row
    target_row[h] = argmin_t |t_tvt[t] - h_tvt[h]|. Cross-entropy on these
    logits forces the true row to have the smallest |sdf| (i.e. be ranked #1),
    directly suppressing false zero-levels and lifting gt_topk_hit.

    Only future columns whose crossing lies inside the typewell window
    (min_dist < ZERO_CE_MAX_DIST_FT) are supervised; out-of-window columns are
    left to the (saturated) regression term.
    """
    dev = pred_sdf.device
    t_seg_tvt = batch["t_seg_tvt"].to(dev).float()   # [B, T]
    h_seg_tvt = batch["h_seg_tvt"].to(dev).float()   # [B, H]
    t_mask = batch["t_mask"].to(dev).float()         # [B, T]
    h_mask = batch["h_mask"].to(dev).float()         # [B, H]

    pred = pred_sdf.float().squeeze(1)               # [B, T, H]
    B, T, H = pred.shape

    row_valid = t_mask > 0                           # [B, T]

    # row-logits per column
    score = -pred.abs() / float(Config.ZERO_CE_TEMP)          # [B, T, H]
    score = score.masked_fill(~row_valid[:, :, None], -1e4)
    logp = F.log_softmax(score, dim=1)                        # [B, T, H]

    # ground-truth crossing row per column
    dist = (t_seg_tvt[:, :, None] - h_seg_tvt[:, None, :]).abs()   # [B, T, H]
    dist = dist.masked_fill(~row_valid[:, :, None], 1e9)
    target_row = dist.argmin(dim=1)                          # [B, H]
    min_dist = dist.gather(1, target_row[:, None, :]).squeeze(1)   # [B, H]

    # supervise only future columns with an in-window crossing
    future = torch.zeros(H, device=dev, dtype=torch.bool)
    future[Config.H_H:] = True
    col_valid = (h_mask > 0) & future[None, :] & (min_dist < float(Config.ZERO_CE_MAX_DIST_FT))

    if float(Config.ZERO_CE_LABEL_SIGMA_FT) > 0.0:
        # Gaussian soft label over rows (softer, less conflict with SmoothL1)
        sigma = float(Config.ZERO_CE_LABEL_SIGMA_FT)
        soft = torch.exp(-(dist ** 2) / (2.0 * sigma * sigma))
        soft = soft.masked_fill(~row_valid[:, :, None], 0.0)
        soft = soft / (soft.sum(dim=1, keepdim=True) + 1e-8)
        nll = -(soft * logp).sum(dim=1)                      # [B, H]
    else:
        # hard CE (spec)
        nll = -logp.gather(1, target_row[:, None, :]).squeeze(1)   # [B, H]

    nll = nll * col_valid.float()
    return nll.sum() / (col_valid.float().sum() + 1e-8)


def sdf_loss(model_out, batch):
    """
    Total loss = weighted SDF SmoothL1 (+ deep supervision) + ZERO_CE_WEIGHT * zero-level CE.
    Returns (total, reg, ce) so the training loop can log the two terms separately.
    """
    if isinstance(model_out, tuple):
        pred_main, a1, a2, a3 = model_out
        w1, w2, w3 = Config.DS_WEIGHTS
        reg = (
            sdf_loss_single(pred_main, batch)
            + w1 * sdf_loss_single(a1, batch)
            + w2 * sdf_loss_single(a2, batch)
            + w3 * sdf_loss_single(a3, batch)
        )
    else:
        pred_main = model_out
        reg = sdf_loss_single(pred_main, batch)

    # CE ranking is applied on the main (full-resolution) head only.
    ce = zero_ce_loss_single(pred_main, batch)
    total = reg + float(Config.ZERO_CE_WEIGHT) * ce
    return total, reg, ce


# ============================================================
# Decode / Eval
# ============================================================
def map_to_original(pred_H, orig_len, h_ps, last_tvt_exact):
    centers = []
    values = []
    for k in range(Config.H_H):
        centers.append(h_ps - k * Config.H_S - (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H - 1 - k])
    centers.append(float(h_ps))
    values.append(float(last_tvt_exact))
    for k in range(Config.H_F):
        centers.append(h_ps + 1 + k * Config.H_S + (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H + k])
    centers = np.asarray(centers, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    order = np.argsort(centers)
    centers = centers[order]
    values = values[order]
    keep = np.r_[True, np.diff(centers) > 1e-6]
    return np.interp(np.arange(orig_len), centers[keep], values[keep])


def decode_sdf(pred_sdf, t_seg_tvt, h_seg_tvt, t_mask, orig_len, h_ps, last_tvt_exact):
    """Pure soft-argmax decode: per future column, TVT = softmax(-|sdf|/temp) . t_tvt."""
    score = -np.abs(pred_sdf.astype(np.float64)) / float(Config.SDF_DECODE_TEMP)
    if t_mask is not None:
        score = np.where(t_mask[:, None] > 0, score, -1e9)
    score = score - score.max(axis=0, keepdims=True)
    prob = np.exp(score)
    prob = prob / (prob.sum(axis=0, keepdims=True) + 1e-12)
    pred_H = (prob * t_seg_tvt[:, None]).sum(axis=0).astype(np.float64)
    pred_H[:Config.H_H] = h_seg_tvt[:Config.H_H]
    return map_to_original(pred_H, orig_len, h_ps, last_tvt_exact)


def compute_topk_hit(pred_sdf, t_seg_tvt, h_seg_tvt, t_mask, h_mask, K):
    """
    Fraction bookkeeping for gt_top-k hit: for each in-window future column,
    is the true crossing row among the K rows with smallest |sdf|?
    Returns (hit_count, valid_column_count).
    """
    T, H = pred_sdf.shape
    valid_t = np.asarray(t_mask) > 0
    abs_sdf = np.abs(pred_sdf.astype(np.float64))
    abs_sdf = np.where(valid_t[:, None], abs_sdf, np.inf)

    dist = np.abs(t_seg_tvt[:, None] - h_seg_tvt[None, :])
    dist = np.where(valid_t[:, None], dist, np.inf)
    target_row = dist.argmin(axis=0)                 # [H]
    min_dist = dist[target_row, np.arange(H)]

    future = np.zeros(H, dtype=bool)
    future[Config.H_H:] = True
    col_valid = (np.asarray(h_mask) > 0) & future & (min_dist < float(Config.ZERO_CE_MAX_DIST_FT))
    if not col_valid.any():
        return 0, 0

    k = min(K, T - 1)
    topk_idx = np.argpartition(abs_sdf, k, axis=0)[:k, :]    # [k, H]
    hit_mask = (topk_idx == target_row[None, :]).any(axis=0) # [H]
    return int((hit_mask & col_valid).sum()), int(col_valid.sum())


@torch.no_grad()
def evaluate(model, loader):
    """Returns (rmse, gt_topk_hit)."""
    model.eval()
    se = 0.0
    n = 0
    hit = 0
    hit_n = 0
    K = Config.ZERO_CE_TOPK
    for batch in loader:
        logits = model(batch)
        logits = logits.squeeze(1).cpu().numpy()
        t_seg = batch["t_seg_tvt"].numpy()
        h_seg = batch["h_seg_tvt"].numpy()
        t_mask = batch["t_mask"].numpy()
        h_mask = batch["h_mask"].numpy()
        y = batch["orig_tvt"].numpy()
        orig_len = batch["orig_len"].numpy()
        h_ps = batch["h_ps"].numpy()
        last = batch["last_tvt_exact"].numpy()
        for b in range(logits.shape[0]):
            olen = int(orig_len[b])
            ps = int(h_ps[b])
            if ps >= olen - 1:
                continue
            pred = decode_sdf(
                logits[b], t_seg[b], h_seg[b], t_mask[b],
                olen, ps, float(last[b]),
            )
            err = pred[ps + 1:] - y[b, :olen][ps + 1:]
            se += np.sum(err ** 2)
            n += len(err)

            h_hit, h_cnt = compute_topk_hit(
                logits[b], t_seg[b], h_seg[b], t_mask[b], h_mask[b], K,
            )
            hit += h_hit
            hit_n += h_cnt
    rmse = np.sqrt(se / max(n, 1))
    topk = hit / max(hit_n, 1)
    return rmse, topk


@torch.no_grad()
def evaluate_last_tvt(loader):
    se = 0.0
    n = 0
    for batch in loader:
        y = batch["orig_tvt"].numpy()
        orig_len = batch["orig_len"].numpy()
        h_ps = batch["h_ps"].numpy()
        for b in range(len(orig_len)):
            olen = int(orig_len[b])
            ps = int(h_ps[b])
            if ps >= olen - 1:
                continue
            true = y[b, :olen]
            err = true[ps] - true[ps + 1:]
            se += np.sum(err ** 2)
            n += len(err)
    return np.sqrt(se / max(n, 1))


# ============================================================
# Train / Inference
# ============================================================
def typewell_hash(horizontal_file):
    sid = horizontal_file.name.split("__")[0]
    tw = horizontal_file.parent / f"{sid}{Config.TYPEWELL_SUFFIX}"
    try:
        gr = np.round(pd.read_csv(tw)["GR"].values.astype(float), 2)
        return hashlib.md5(gr.tobytes()).hexdigest()
    except Exception:
        return horizontal_file.name


def make_loader(files, is_train=True):
    return DataLoader(
        HeatmapDataset(files, is_train=is_train),
        batch_size=Config.BATCH_SIZE, shuffle=is_train,
        num_workers=Config.NUM_WORKERS, pin_memory=True,
    )


def train_one_fold(fold, train_idx, val_idx, files):
    cfg = Config
    train_files = [files[i] for i in train_idx]
    val_files = [files[i] for i in val_idx]
    val_loader = make_loader(val_files, is_train=False)
    print("Last TVT RMSE:", evaluate_last_tvt(val_loader))
    print("train files:", len(train_files), "val files:", len(val_files))

    model = HeatmapResUNet(base=16).to(cfg.DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"model params: {n_params:,}")

    train_loader = make_loader(train_files, is_train=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
    use_amp = cfg.DEVICE.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    print(f"\naug_repeats: {cfg.AUG_REPEATS}  epochs: {cfg.EPOCHS}  lr: {cfg.LR}")
    print(f"START_JITTER={cfg.START_JITTER}  RANDOM_CUTOFF_PROB={cfg.RANDOM_CUTOFF_PROB}"
          f"  RANDOM_CUTOFF_FRAC=[{cfg.RANDOM_CUTOFF_MIN_FRAC}, {cfg.RANDOM_CUTOFF_MAX_FRAC}]")
    print(f"deep supervision: ON  weights: {cfg.DS_WEIGHTS}")
    print(f"SDF target: scale_ft={cfg.SDF_TARGET_SCALE_FT}"
          f"  clip={cfg.SDF_TARGET_CLIP}"
          f"  near_sigma_ft={cfg.SDF_NEAR_SIGMA_FT}"
          f"  near_weight={cfg.SDF_NEAR_WEIGHT}")
    print(f"SDF decode: pure soft-argmax  decode_temp={cfg.SDF_DECODE_TEMP}")
    print(f"zero-CE: weight={cfg.ZERO_CE_WEIGHT}  temp={cfg.ZERO_CE_TEMP}"
          f"  max_dist_ft={cfg.ZERO_CE_MAX_DIST_FT}"
          f"  label_sigma_ft={cfg.ZERO_CE_LABEL_SIGMA_FT}"
          f"  topk={cfg.ZERO_CE_TOPK}")
    print("loss: weighted SDF SmoothL1 + zero-level CE ranking")
    print("architecture: SDF ResUNet-noGN + Horizontal Dilated ASPP + ConvNeXt bottleneck + ECA")

    ckpt_path = f"fold_{fold}_convnext_eca_best.pth"
    best = 1e9

    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        total = 0.0
        total_reg = 0.0
        total_ce = 0.0
        n_batches = 0
        rc_count = 0
        sample_count = 0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                model_out = model(batch)
                loss, reg_l, ce_l = sdf_loss(model_out, batch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total += float(loss.item())
            total_reg += float(reg_l.item())
            total_ce += float(ce_l.item())
            n_batches += 1
            if "cutoff_mode" in batch:
                rc_count += int(batch["cutoff_mode"].sum().item())
                sample_count += int(batch["cutoff_mode"].numel())
        scheduler.step()

        val_rmse, val_top9 = evaluate(model, val_loader)
        tag = ""
        if val_rmse < best:
            best = val_rmse
            tag = "  saved"
            torch.save(model.state_dict(), ckpt_path)
        rc_frac = rc_count / max(sample_count, 1)
        nb = max(n_batches, 1)
        print(
            f"fold {fold} | ep {epoch:02d}/{cfg.EPOCHS:02d} | "
            f"train {total / nb:.5f} (reg {total_reg / nb:.5f} ce {total_ce / nb:.5f}) | "
            f"rc {rc_frac:.2f} | "
            f"val {val_rmse:.4f} | top{cfg.ZERO_CE_TOPK} {val_top9:.3f}{tag}"
        )
    return best


def run_training():
    files = sorted(Path(Config.TRAIN_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    groups = [typewell_hash(f) for f in files]
    gkf = GroupKFold(n_splits=Config.N_FOLDS)
    scores = []
    for fold, (train_idx, val_idx) in enumerate(gkf.split(files, groups=groups)):
        if fold not in Config.FOLDS_TO_RUN:
            continue
        print(f"\n===== Fold {fold} =====")
        print("train:", len(train_idx), "val:", len(val_idx))
        score = train_one_fold(fold, train_idx, val_idx, files)
        scores.append(score)
        print(f"Fold {fold} best: {score:.4f}")
    if scores:
        print("\nCV RMSE:", float(np.mean(scores)))
        print("fold scores:", scores)


@torch.no_grad()
def run_inference():
    test_files = sorted(Path(Config.TEST_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    ckpts = sorted(Path(".").glob("fold_*_convnext_eca_best.pth"))
    if not test_files or not ckpts:
        print("inference skipped")
        print("test_files:", len(test_files), "ckpts:", len(ckpts))
        return
    models = []
    for ckpt in ckpts:
        model = HeatmapResUNet(base=16).to(Config.DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=Config.DEVICE))
        model.eval()
        models.append(model)
        print("loaded", ckpt.name)

    ds = HeatmapDataset(test_files, is_train=False)
    preds = {}
    for i, f in enumerate(test_files):
        sid = f.name.split("__")[0]
        sample = ds[i]
        batch = {k: v.unsqueeze(0) for k, v in sample.items()}
        logits_list = []
        for model in models:
            logits = model(batch).squeeze(1).float().cpu().numpy()[0]
            logits_list.append(logits)
        avg_logits = np.mean(logits_list, axis=0)
        pred = decode_sdf(
            avg_logits,
            sample["t_seg_tvt"].numpy(), sample["h_seg_tvt"].numpy(), sample["t_mask"].numpy(),
            int(sample["orig_len"]), int(sample["h_ps"]), float(sample["last_tvt_exact"]),
        )
        df = pd.read_csv(f)
        if "TVT_input" in df.columns:
            rows = np.flatnonzero(df["TVT_input"].isna().values)
        else:
            rows = np.arange(len(df))
        for r in rows:
            preds[f"{sid}_{int(r)}"] = float(pred[int(r)])
        print("done", sid, "submit rows:", len(rows))

    sub_path = Path(Config.TRAIN_DIR).parent / "sample_submission.csv"
    if sub_path.exists():
        sub = pd.read_csv(sub_path)[["id"]].copy()
        sub["tvt"] = sub["id"].map(preds).fillna(0.0).astype(np.float32)
    else:
        sub = pd.DataFrame({"id": list(preds.keys()), "tvt": list(preds.values())})
    sub.to_csv("submission.csv", index=False)
    print("saved submission.csv", sub.shape)
    print(sub.head())


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    run_training()
    run_inference()

In [ ]:
# ============================================================
# ROGII - Pure 1D GR Three-Resolution + Hidden160 + Local2 single model
# ============================================================
# Fixed architecture selected from the capacity search:
#   hidden=160
#   dilations=(1,2,4,8,16,32,64,1,2)
#   expected params=535,853
# cv = 9.6,lb = 10
# ============================================================

from __future__ import annotations

import hashlib
import math
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# ============================================================
# Config
# ============================================================
class Config:
    DATA_ROOT: Optional[str] = None
    HORIZONTAL_SUFFIX = "__horizontal_well.csv"
    TYPEWELL_SUFFIX = "__typewell.csv"

    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    SEED = 42

    N_FOLDS = 4
    FOLDS_TO_RUN = [0]

    # 1D grid
    H_S = 16
    H_H = 64
    H_F = 704
    H_TOTAL = H_H + H_F
    ORIG_PAD_LEN = 16384

    # augmentation
    AUG_REPEATS = 4
    START_JITTER = 512
    RANDOM_CUTOFF_PROB = 0.50
    RANDOM_CUTOFF_MIN_FRAC = 0.20
    RANDOM_CUTOFF_MAX_FRAC = 0.80
    RANDOM_CUTOFF_MIN_HISTORY = 512
    RANDOM_CUTOFF_MIN_FUTURE = 512

    # scales
    TARGET_SCALE_FT = 32.0
    DZ_SCALE = 2.0
    FEATURE_CLIP = 6.0

    # model
    INPUT_CHANNELS = 7
    HIDDEN_CHANNELS = 160
    DILATIONS = (1, 2, 4, 8, 16, 32, 64, 1, 2)
    KERNEL_SIZE = 5
    DROPOUT = 0.10

    # optimization (FP32)
    EPOCHS = 12
    BATCH_SIZE = 4
    NUM_WORKERS = 12
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    WARMUP_FRAC = 0.10
    GRAD_CLIP = 1

    # loss (normalized offsets)
    TVT_BETA_NORM = 0.125
    SLOPE_BETA_NORM = 0.05
    SLOPE_LOSS_WEIGHT = 0.10
    HISTORY_LOSS_WEIGHT = 0.05

    CHECKPOINT_PATTERN = "fold_{fold}_gr3_hidden160_local2_best.pth"
    CHECKPOINT_GLOB = "fold_*_gr3_hidden160_local2_best.pth"
    SUBMISSION_NAME = "submission_gr3_hidden160_local2.csv"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(Config.SEED)
print("DEVICE:", Config.DEVICE)


# ============================================================
# Data root
# ============================================================
def find_data_root() -> Path:
    candidates = [
        Path(Config.DATA_ROOT) if Config.DATA_ROOT else None,
        Path("/kaggle/input/datasets/zhuyifanss/roggi-data"),
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    for root in candidates:
        if root is not None and (root / "train").is_dir():
            return root.resolve()
    raise FileNotFoundError("No data root with train/ found. Set Config.DATA_ROOT.")


DATA_ROOT = find_data_root()
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
SAMPLE_SUB_PATH = DATA_ROOT / "sample_submission.csv"
print("DATA_ROOT:", DATA_ROOT)


# ============================================================
# Utilities
# ============================================================
def well_id_from_path(path: Path) -> str:
    return path.name.split("__", 1)[0]


def fill_numeric_series(series: pd.Series, default: float = 0.0) -> np.ndarray:
    s = pd.to_numeric(series, errors="coerce").astype(float)
    if s.notna().sum() == 0:
        return np.full(len(s), float(default), dtype=np.float64)
    return (
        s.interpolate(limit_direction="both")
        .bfill()
        .ffill()
        .fillna(default)
        .to_numpy(dtype=np.float64)
    )


def get_numeric_column(df: pd.DataFrame, name: str, default: float = 0.0) -> np.ndarray:
    if name not in df.columns:
        return np.full(len(df), float(default), dtype=np.float64)
    return fill_numeric_series(df[name], default=default)


def bin_mean(arr: np.ndarray, step: int, back: bool) -> np.ndarray:
    """History 对齐末端, future 对齐起点的分箱均值。"""
    arr = np.asarray(arr)
    if len(arr) == 0:
        return arr.astype(np.float64)

    pad = (-len(arr)) % step
    if pad < step // 2:
        if pad:
            arr = np.pad(arr, (0, pad) if back else (pad, 0), mode="edge")
    elif pad:
        trim = step - pad
        arr = arr[:-trim] if back else arr[trim:]

    if len(arr) == 0:
        return arr.astype(np.float64)
    return arr.reshape(-1, step).mean(axis=1)


def crop_pad_1d(n: int, center: int, history: int, future: int) -> Tuple[int, int, int, int]:
    raw_i0, raw_i1 = center - history, center + future
    return max(raw_i0, 0), min(raw_i1, n), max(0, -raw_i0), max(0, raw_i1 - n)


def pad_1d(arr, i0, i1, pad_left, pad_right, default: float) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float64)
    piece = arr[i0:i1]
    if len(piece) == 0:
        return np.full(pad_left + pad_right, float(default), dtype=np.float64)
    return np.pad(piece, (pad_left, pad_right), mode="edge")


def get_official_h_ps(h: pd.DataFrame) -> int:
    if "TVT_input" in h.columns:
        idx = np.flatnonzero(h["TVT_input"].notna().to_numpy())
        if len(idx):
            return int(idx[-1])
    return max(0, len(h) // 2)


def choose_train_cutoff(h: pd.DataFrame) -> int:
    n = len(h)
    official_ps = get_official_h_ps(h)

    can_random = (
        "TVT" in h.columns
        and h["TVT"].notna().sum() > max(32, Config.RANDOM_CUTOFF_MIN_HISTORY // 4)
        and n > Config.RANDOM_CUTOFF_MIN_HISTORY + Config.RANDOM_CUTOFF_MIN_FUTURE + 8
    )
    if can_random and np.random.rand() < Config.RANDOM_CUTOFF_PROB:
        lo = max(int(round(n * Config.RANDOM_CUTOFF_MIN_FRAC)), Config.RANDOM_CUTOFF_MIN_HISTORY, 1)
        hi = min(int(round(n * Config.RANDOM_CUTOFF_MAX_FRAC)), n - 1 - Config.RANDOM_CUTOFF_MIN_FUTURE)
        if hi > lo:
            return int(np.random.randint(lo, hi + 1))

    jitter = -int(np.random.randint(0, Config.START_JITTER + 1)) if Config.START_JITTER > 0 else 0
    return int(np.clip(official_ps + jitter, 0, n - 1))


def robust_standardize(x: np.ndarray, mask: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    mask = np.asarray(mask, dtype=bool) & np.isfinite(x)
    if mask.sum() < 2:
        return np.zeros_like(x, dtype=np.float32)

    values = x[mask]
    center = float(np.median(values))
    scale = 1.4826 * float(np.median(np.abs(values - center)))
    if not np.isfinite(scale) or scale < 1e-3:
        scale = float(np.std(values))
    if not np.isfinite(scale) or scale < 1e-3:
        scale = 1.0

    z = np.nan_to_num((x - center) / scale, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(z, -Config.FEATURE_CLIP, Config.FEATURE_CLIP).astype(np.float32)


def resample_horizontal(h: pd.DataFrame, forced_h_ps: Optional[int] = None):
    n = len(h)
    if n == 0:
        raise ValueError("Empty horizontal-well file")

    gr = fill_numeric_series(h["GR"], default=85.0)
    z = get_numeric_column(h, "Z", 0.0)
    x = get_numeric_column(h, "X", 0.0)
    y = get_numeric_column(h, "Y", 0.0)

    if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
        tvt = fill_numeric_series(h["TVT"], default=0.0)
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        tvt = fill_numeric_series(h["TVT_input"], default=0.0)
    else:
        tvt = z.copy()

    h_ps = int(np.clip(forced_h_ps, 0, n - 1)) if forced_h_ps is not None else get_official_h_ps(h)

    def split_and_bin(arr):
        a0 = bin_mean(arr[: h_ps + 1], Config.H_S, back=False)
        a1 = bin_mean(arr[h_ps + 1:], Config.H_S, back=True)
        return a0, a1

    tvt0, tvt1 = split_and_bin(tvt)
    gr0, gr1 = split_and_bin(gr)
    z0, z1 = split_and_bin(z)
    x0, x1 = split_and_bin(x)
    y0, y1 = split_and_bin(y)
    return tvt0, gr0, tvt1, gr1, h_ps, z0, z1, x0, x1, y0, y1


def get_last_tvt_exact(h: pd.DataFrame, h_ps: int, history_tvt: np.ndarray) -> float:
    for col in ("TVT_input", "TVT"):
        if col in h.columns:
            value = pd.to_numeric(h[col], errors="coerce").iloc[h_ps]
            if pd.notna(value):
                return float(value)
    return float(history_tvt[-1]) if len(history_tvt) else 0.0


def map_to_original(pred_h: np.ndarray, orig_len: int, h_ps: int, last_tvt_exact: float) -> np.ndarray:
    """把 768 个 bin 级 TVT 预测插值回原始行分辨率。"""
    pred_h = np.asarray(pred_h, dtype=np.float64)
    centers, values = [], []

    for k in range(Config.H_H):
        centers.append(h_ps - k * Config.H_S - (Config.H_S - 1) / 2.0)
        values.append(float(pred_h[Config.H_H - 1 - k]))

    centers.append(float(h_ps))
    values.append(float(last_tvt_exact))

    for k in range(Config.H_F):
        centers.append(h_ps + 1 + k * Config.H_S + (Config.H_S - 1) / 2.0)
        values.append(float(pred_h[Config.H_H + k]))

    centers = np.asarray(centers)
    values = np.asarray(values)
    order = np.argsort(centers)
    centers, values = centers[order], values[order]
    keep = np.r_[True, np.diff(centers) > 1e-6]
    return np.interp(np.arange(orig_len, dtype=np.float64), centers[keep], values[keep])


def typewell_hash(horizontal_file: Path) -> str:
    sid = well_id_from_path(horizontal_file)
    tw_path = horizontal_file.parent / f"{sid}{Config.TYPEWELL_SUFFIX}"
    try:
        tw = pd.read_csv(tw_path)
        gr = pd.to_numeric(tw["GR"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
        return hashlib.md5(np.round(gr, 2).tobytes()).hexdigest()
    except Exception:
        return horizontal_file.name


# ============================================================
# Dataset (7 channels)
# ============================================================
class Pure1DDataset(Dataset):
    FEATURE_NAMES = (
        "h_gr_norm", "h_dz", "az_sin", "az_cos",
        "hist_mask", "future_mask", "history_tvt_rel",
    )

    def __init__(self, files: Sequence[Path], is_train: bool) -> None:
        self.files = list(files)
        self.is_train = bool(is_train)
        self.aug_repeats = Config.AUG_REPEATS if self.is_train else 1

    def __len__(self) -> int:
        return len(self.files) * self.aug_repeats if self.is_train else len(self.files)

    def __getitem__(self, idx: int) -> Dict[str, object]:
        if self.is_train:
            idx %= len(self.files)

        path = Path(self.files[idx])
        well_id = well_id_from_path(path)
        h = pd.read_csv(path)

        forced_h_ps = choose_train_cutoff(h) if self.is_train else None
        tvt0, gr0, tvt1, gr1, h_ps, z0, z1, x0, x1, y0, y1 = resample_horizontal(h, forced_h_ps)
        last_tvt_exact = get_last_tvt_exact(h, h_ps, tvt0)

        # history: 取到 prediction start 为止的最后 H_H 个 bin
        i0, i1, p0l, p0r = crop_pad_1d(len(tvt0), len(tvt0), Config.H_H, 0)
        hist_valid = np.pad(np.ones(i1 - i0, dtype=np.float32), (p0l, p0r), constant_values=0.0)

        tvt_hist = pad_1d(tvt0, i0, i1, p0l, p0r, last_tvt_exact)
        gr_hist = pad_1d(gr0, i0, i1, p0l, p0r, 85.0)
        z_hist = pad_1d(z0, i0, i1, p0l, p0r, float(h["Z"].iloc[h_ps]) if "Z" in h else 0.0)
        x_hist = pad_1d(x0, i0, i1, p0l, p0r, float(h["X"].iloc[h_ps]) if "X" in h else 0.0)
        y_hist = pad_1d(y0, i0, i1, p0l, p0r, float(h["Y"].iloc[h_ps]) if "Y" in h else 0.0)

        # future: start 之后的前 H_F 个 bin
        j0, j1, p1l, p1r = crop_pad_1d(len(tvt1), 0, 0, Config.H_F)
        future_valid = np.pad(np.ones(j1 - j0, dtype=np.float32), (p1l, p1r), constant_values=0.0)

        tvt_future = pad_1d(tvt1, j0, j1, p1l, p1r, last_tvt_exact)
        gr_future = pad_1d(gr1, j0, j1, p1l, p1r, gr_hist[-1])
        z_future = pad_1d(z1, j0, j1, p1l, p1r, z_hist[-1])
        x_future = pad_1d(x1, j0, j1, p1l, p1r, x_hist[-1])
        y_future = pad_1d(y1, j0, j1, p1l, p1r, y_hist[-1])

        tvt_seq = np.concatenate([tvt_hist, tvt_future])
        gr_seq = np.concatenate([gr_hist, gr_future])
        z_seq = np.concatenate([z_hist, z_future])
        x_seq = np.concatenate([x_hist, x_future])
        y_seq = np.concatenate([y_hist, y_future])

        hist_mask = np.zeros(Config.H_TOTAL, dtype=np.float32)
        hist_mask[: Config.H_H] = hist_valid
        fut_mask = np.zeros(Config.H_TOTAL, dtype=np.float32)
        fut_mask[Config.H_H:] = future_valid
        valid_mask = (hist_mask + fut_mask) > 0.5

        # features
        h_gr_norm = robust_standardize(gr_seq, valid_mask)

        h_dz = np.gradient(z_seq) / Config.DZ_SCALE
        h_dz = np.clip(np.nan_to_num(h_dz), -Config.FEATURE_CLIP, Config.FEATURE_CLIP).astype(np.float32)

        az = np.arctan2(np.gradient(y_seq), np.gradient(x_seq))
        az_sin = np.nan_to_num(np.sin(az), nan=0.0).astype(np.float32)
        az_cos = np.nan_to_num(np.cos(az), nan=1.0).astype(np.float32)

        history_tvt_rel = np.zeros(Config.H_TOTAL, dtype=np.float32)
        history_tvt_rel[: Config.H_H] = (
            (tvt_hist - last_tvt_exact) / Config.TARGET_SCALE_FT
        ).astype(np.float32) * hist_valid
        history_tvt_rel = np.clip(history_tvt_rel, -Config.FEATURE_CLIP, Config.FEATURE_CLIP)

        x_features = np.stack(
            [h_gr_norm, h_dz, az_sin, az_cos, hist_mask, fut_mask, history_tvt_rel],
            axis=0,
        ).astype(np.float32)
        x_features *= valid_mask.astype(np.float32)[None, :]
        x_features = np.nan_to_num(x_features)

        target_offset_norm = np.nan_to_num(
            ((tvt_seq - last_tvt_exact) / Config.TARGET_SCALE_FT).astype(np.float32)
        )

        # 原始行分辨率的 target, 用于 faithful validation
        if "TVT" in h.columns:
            orig_tvt_raw = pd.to_numeric(h["TVT"], errors="coerce").to_numpy(dtype=np.float64)
            orig_valid_raw = np.isfinite(orig_tvt_raw)
            orig_tvt_filled = fill_numeric_series(pd.Series(orig_tvt_raw), default=last_tvt_exact)
        else:
            orig_valid_raw = np.zeros(len(h), dtype=bool)
            orig_tvt_filled = np.full(len(h), last_tvt_exact, dtype=np.float64)

        orig_len = len(h)
        if orig_len > Config.ORIG_PAD_LEN:
            raise ValueError(f"Well {well_id} exceeds ORIG_PAD_LEN")

        padded_tvt = np.zeros(Config.ORIG_PAD_LEN, dtype=np.float32)
        padded_valid = np.zeros(Config.ORIG_PAD_LEN, dtype=np.bool_)
        padded_tvt[:orig_len] = orig_tvt_filled.astype(np.float32)
        padded_valid[:orig_len] = orig_valid_raw

        return {
            "x": torch.from_numpy(x_features),
            "target_offset_norm": torch.from_numpy(target_offset_norm),
            "hist_mask": torch.from_numpy(hist_mask),
            "future_mask": torch.from_numpy(fut_mask),
            "valid_mask": torch.from_numpy(valid_mask.astype(np.float32)),
            "last_tvt_exact": torch.tensor(last_tvt_exact, dtype=torch.float32),
            "orig_tvt": torch.from_numpy(padded_tvt),
            "orig_valid": torch.from_numpy(padded_valid),
            "orig_len": torch.tensor(orig_len, dtype=torch.int64),
            "h_ps": torch.tensor(h_ps, dtype=torch.int64),
            "well_id": well_id,
        }


# ============================================================
# Model
# ============================================================
class ChannelLayerNorm1d(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm(x.transpose(1, 2)).transpose(1, 2)


class ConvNormAct1d(nn.Module):
    """Conv1d -> channel-wise LayerNorm -> SiLU."""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        dilation: int = 1,
    ) -> None:
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
        )
        self.norm = ChannelLayerNorm1d(out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.silu(self.norm(self.conv(x)))


class DilatedResidualBlock(nn.Module):
    def __init__(self, channels: int, dilation: int, kernel_size: int, dropout: float) -> None:
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.norm1 = ChannelLayerNorm1d(channels)
        self.dw1 = nn.Conv1d(channels, channels, kernel_size, padding=padding,
                             dilation=dilation, groups=channels)
        self.pw1 = nn.Conv1d(channels, channels, 1)
        self.norm2 = ChannelLayerNorm1d(channels)
        self.dw2 = nn.Conv1d(channels, channels, kernel_size, padding=padding,
                             dilation=dilation, groups=channels)
        self.pw2 = nn.Conv1d(channels, channels, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.pw1(self.dw1(self.norm1(x)))
        y = self.dropout(F.silu(y))
        y = self.pw2(self.dw2(self.norm2(y)))
        y = self.dropout(y)
        return x + y


class Pure1DDirectModel(nn.Module):
    """Three-resolution GR encoder + hidden160 global TCN + two local refinement blocks."""
    def __init__(self) -> None:
        super().__init__()
        hidden = Config.HIDDEN_CHANNELS
        gr_out = hidden // 2
        other_out = hidden - gr_out

        # Preserve the true-exact module construction order from feature search.
        # With a fixed seed, changing construction order changes initialization.
        # Order: other stem -> full/half/quarter GR -> GR fuse -> input fuse.
        branch_c = max(8, gr_out // 3)

        # The other six original channels are encoded separately:
        # h_dz, az_sin, az_cos, hist_mask, future_mask, history_tvt_rel.
        self.other_stem = ConvNormAct1d(6, other_out, kernel_size=3)

        # Three learned GR resolutions. hidden=160 -> gr_out=80, branch_c=26.
        self.gr_full = ConvNormAct1d(1, branch_c, kernel_size=7, stride=1)
        self.gr_half = ConvNormAct1d(1, branch_c, kernel_size=15, stride=2)
        self.gr_quarter = ConvNormAct1d(1, branch_c, kernel_size=31, stride=4)
        self.gr_fuse = ConvNormAct1d(branch_c * 3, gr_out, kernel_size=1)
        self.input_fuse = ConvNormAct1d(hidden, hidden, kernel_size=1)

        # Seven global blocks followed by two local refinement blocks.
        self.blocks = nn.ModuleList(
            [
                DilatedResidualBlock(
                    hidden,
                    dilation=d,
                    kernel_size=Config.KERNEL_SIZE,
                    dropout=Config.DROPOUT,
                )
                for d in Config.DILATIONS
            ]
        )

        self.head = nn.Sequential(
            ChannelLayerNorm1d(hidden),
            nn.SiLU(),
            nn.Conv1d(hidden, hidden // 2, 1),
            nn.SiLU(),
            nn.Conv1d(hidden // 2, 1, 1),
        )

        # Initialize at the flat last-TVT baseline.
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def encode_inputs(self, x: torch.Tensor) -> torch.Tensor:
        # Dataset channel order:
        # 0 h_gr_norm
        # 1 h_dz
        # 2 az_sin
        # 3 az_cos
        # 4 hist_mask
        # 5 future_mask
        # 6 history_tvt_rel
        gr = x[:, 0:1, :]
        other = x[:, 1:7, :]
        length = gr.shape[-1]

        gr_full = self.gr_full(gr)
        gr_half = F.interpolate(
            self.gr_half(gr),
            size=length,
            mode="linear",
            align_corners=False,
        )
        gr_quarter = F.interpolate(
            self.gr_quarter(gr),
            size=length,
            mode="linear",
            align_corners=False,
        )
        gr_feat = self.gr_fuse(torch.cat([gr_full, gr_half, gr_quarter], dim=1))
        other_feat = self.other_stem(other)
        return self.input_fuse(torch.cat([gr_feat, other_feat], dim=1))

    def forward(
        self,
        x: torch.Tensor,
        valid_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        y = self.encode_inputs(x)
        if valid_mask is not None:
            y = y * valid_mask[:, None, :]

        # Exact behavior of the feature-search model that produced the
        # reported Fold-0 gr_three_resolution result: mask before and after
        # the full TCN, not after every residual block.
        for block in self.blocks:
            y = block(y)

        if valid_mask is not None:
            y = y * valid_mask[:, None, :]

        pred = self.head(y).squeeze(1)
        if valid_mask is not None:
            pred = pred * valid_mask
        return pred

# ============================================================
# Loss
# ============================================================
def masked_smooth_l1(pred, target, mask, beta: float) -> torch.Tensor:
    loss = F.smooth_l1_loss(pred, target, reduction="none", beta=beta)
    return (loss * mask).sum() / mask.sum().clamp_min(1.0)


def trajectory_loss(pred, target, hist_mask, future_mask, valid_mask):
    future_loss = masked_smooth_l1(pred, target, future_mask, Config.TVT_BETA_NORM)
    history_loss = masked_smooth_l1(pred, target, hist_mask, Config.TVT_BETA_NORM)

    pred_slope = pred[:, 1:] - pred[:, :-1]
    true_slope = target[:, 1:] - target[:, :-1]
    slope_mask = future_mask[:, 1:] * valid_mask[:, :-1]
    slope_loss = masked_smooth_l1(pred_slope, true_slope, slope_mask, Config.SLOPE_BETA_NORM)

    total = (
        future_loss
        + Config.SLOPE_LOSS_WEIGHT * slope_loss
        + Config.HISTORY_LOSS_WEIGHT * history_loss
    )
    info = {k: float(v.detach().cpu()) for k, v in
            [("future", future_loss), ("slope", slope_loss), ("history", history_loss)]}
    return total, info


# ============================================================
# Validation (original-row resolution)
# ============================================================
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    se = ae = sum_err = n = 0.0
    base_se = 0.0

    for batch in loader:
        x = batch["x"].to(Config.DEVICE, non_blocking=True)
        valid_mask = batch["valid_mask"].to(Config.DEVICE, non_blocking=True)
        pred_offset = model(x, valid_mask=valid_mask).float().cpu().numpy() * Config.TARGET_SCALE_FT

        last = batch["last_tvt_exact"].numpy().astype(np.float64)
        orig_tvt = batch["orig_tvt"].numpy().astype(np.float64)
        orig_valid = batch["orig_valid"].numpy().astype(bool)
        orig_len = batch["orig_len"].numpy()
        h_ps = batch["h_ps"].numpy()

        for b in range(len(orig_len)):
            olen, ps = int(orig_len[b]), int(h_ps[b])
            if ps >= olen - 1:
                continue

            pred_orig = map_to_original(last[b] + pred_offset[b], olen, ps, last[b])
            idx = np.arange(ps + 1, olen)
            idx = idx[orig_valid[b, idx]]
            if len(idx) == 0:
                continue

            err = pred_orig[idx] - orig_tvt[b, idx]
            se += float(np.sum(err ** 2))
            ae += float(np.sum(np.abs(err)))
            sum_err += float(np.sum(err))
            n += len(err)
            base_se += float(np.sum((last[b] - orig_tvt[b, idx]) ** 2))

    n = max(n, 1.0)
    return {
        "rmse": float(np.sqrt(se / n)),
        "mae": float(ae / n),
        "bias": float(sum_err / n),
        "baseline_rmse": float(np.sqrt(base_se / n)),
    }


# ============================================================
# Loader / scheduler
# ============================================================
def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(files: Sequence[Path], is_train: bool) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(Config.SEED + (1 if is_train else 1000))
    return DataLoader(
        Pure1DDataset(files, is_train=is_train),
        batch_size=Config.BATCH_SIZE,
        shuffle=is_train,
        num_workers=Config.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=Config.NUM_WORKERS > 0,
        worker_init_fn=seed_worker,
        generator=generator,
    )


def make_scheduler(optimizer, total_steps: int):
    warmup_steps = max(1, int(total_steps * Config.WARMUP_FRAC))

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return max(1e-4, (step + 1) / warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ============================================================
# Training
# ============================================================
def train_one_fold(fold: int, train_idx, val_idx, files: Sequence[Path]) -> float:
    seed_everything(Config.SEED + fold)
    train_files = [Path(files[int(i)]) for i in train_idx]
    val_files = [Path(files[int(i)]) for i in val_idx]
    train_loader = make_loader(train_files, is_train=True)
    val_loader = make_loader(val_files, is_train=False)

    model = Pure1DDirectModel().to(Config.DEVICE)
    print("train:", len(train_files), "val:", len(val_files),
          "params:", f"{sum(p.numel() for p in model.parameters()):,}")
    print("architecture: GR three-resolution encoder + hidden160 TCN + local2 refinement")
    print("dilations:", Config.DILATIONS)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY, eps=1e-6
    )
    scheduler = make_scheduler(optimizer, max(1, len(train_loader) * Config.EPOCHS))

    ckpt_path = Path(Config.CHECKPOINT_PATTERN.format(fold=fold))
    best = float("inf")

    for epoch in range(1, Config.EPOCHS + 1):
        model.train()
        sums = {"loss": 0.0, "future": 0.0, "slope": 0.0, "history": 0.0}
        n_batches = 0

        for batch in train_loader:
            x = batch["x"].to(Config.DEVICE, non_blocking=True)
            target = batch["target_offset_norm"].to(Config.DEVICE, non_blocking=True)
            hist_mask = batch["hist_mask"].to(Config.DEVICE, non_blocking=True)
            future_mask = batch["future_mask"].to(Config.DEVICE, non_blocking=True)
            valid_mask = batch["valid_mask"].to(Config.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            pred = model(x, valid_mask=valid_mask)
            loss, info = trajectory_loss(pred, target, hist_mask, future_mask, valid_mask)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss at fold={fold}, epoch={epoch}")

            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
            if not torch.isfinite(grad_norm):
                raise FloatingPointError(f"Non-finite gradient at fold={fold}, epoch={epoch}")
            optimizer.step()
            scheduler.step()

            sums["loss"] += float(loss.detach().cpu())
            for k in ("future", "slope", "history"):
                sums[k] += info[k]
            n_batches += 1

        metrics = evaluate(model, val_loader)
        saved = ""
        if metrics["rmse"] < best:
            best = metrics["rmse"]
            torch.save({"model_state": model.state_dict(), "fold": fold, "best_metric": best}, ckpt_path)
            saved = " saved"

        nb = max(n_batches, 1)
        print(
            f"fold {fold} | ep {epoch:02d}/{Config.EPOCHS:02d} | "
            f"train {sums['loss'] / nb:.5f} "
            f"(tvt {sums['future'] / nb:.5f} slope {sums['slope'] / nb:.5f} "
            f"hist {sums['history'] / nb:.5f}) | "
            f"val {metrics['rmse']:.4f} | base {metrics['baseline_rmse']:.4f} | "
            f"mae {metrics['mae']:.3f} | bias {metrics['bias']:.3f}{saved}"
        )

    print(f"Fold {fold} best rmse: {best:.5f}")
    return best


def run_training() -> None:
    files = sorted(TRAIN_DIR.glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"No horizontal-well files in {TRAIN_DIR}")

    groups = np.asarray([typewell_hash(p) for p in files])
    if len(np.unique(groups)) < Config.N_FOLDS:
        raise ValueError("Not enough typewell groups for GroupKFold")

    scores: List[float] = []
    for fold, (train_idx, val_idx) in enumerate(GroupKFold(Config.N_FOLDS).split(files, groups=groups)):
        if fold not in Config.FOLDS_TO_RUN:
            continue
        print(f"\n===== Fold {fold} =====")
        scores.append(train_one_fold(fold, train_idx, val_idx, files))

    if scores:
        print("\nMean CV:", float(np.mean(scores)), "| folds:", scores)


# ============================================================
# Inference
# ============================================================
def load_checkpoint_model(path: Path) -> Pure1DDirectModel:
    payload = torch.load(path, map_location=Config.DEVICE)
    model = Pure1DDirectModel().to(Config.DEVICE)
    model.load_state_dict(payload["model_state"] if "model_state" in payload else payload)
    model.eval()
    return model


@torch.no_grad()
def predict_test_well(model: Pure1DDirectModel, path: Path):
    item = Pure1DDataset([path], is_train=False)[0]
    x = item["x"].unsqueeze(0).to(Config.DEVICE)
    valid_mask = item["valid_mask"].unsqueeze(0).to(Config.DEVICE)
    pred_offset = model(x, valid_mask=valid_mask)[0].float().cpu().numpy() * Config.TARGET_SCALE_FT

    last_tvt = float(item["last_tvt_exact"].item())
    orig_len = int(item["orig_len"].item())
    h_ps = int(item["h_ps"].item())
    pred_orig = map_to_original(last_tvt + pred_offset, orig_len, h_ps, last_tvt)
    return str(item["well_id"]), pred_orig, h_ps


def run_inference() -> None:
    test_files = sorted(TEST_DIR.glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    ckpts = sorted(Path(".").glob(Config.CHECKPOINT_GLOB))
    if not test_files or not ckpts:
        print("inference skipped | test files:", len(test_files), "| checkpoints:", len(ckpts))
        return

    models = [load_checkpoint_model(p) for p in ckpts]
    print("loaded checkpoints:", [p.name for p in ckpts])

    pred_map: Dict[str, float] = {}
    for test_path in test_files:
        well_id = well_id_from_path(test_path)
        fold_preds, h_ps_ref = [], 0
        for model in models:
            _, pred_orig, h_ps_ref = predict_test_well(model, test_path)
            fold_preds.append(pred_orig)
        ensemble = np.mean(np.stack(fold_preds, axis=0), axis=0)

        for row_idx in range(h_ps_ref + 1, len(ensemble)):
            pred_map[f"{well_id}_{row_idx}"] = float(ensemble[row_idx])
        print("done", well_id, "| future rows:", len(ensemble) - h_ps_ref - 1)

    if SAMPLE_SUB_PATH.exists():
        submission = pd.read_csv(SAMPLE_SUB_PATH)
        submission["tvt"] = submission["id"].map(pred_map)
        if submission["tvt"].isna().any():
            raise KeyError(f"Missing predictions for {submission['tvt'].isna().sum()} ids")
        submission = submission[["id", "tvt"]]
    else:
        submission = pd.DataFrame({"id": list(pred_map.keys()), "tvt": list(pred_map.values())})

    submission.to_csv(Config.SUBMISSION_NAME, index=False)
    print("saved", Config.SUBMISSION_NAME, submission.shape)


if __name__ == "__main__":
    run_training()
    run_inference()

实际提交代码

In [ ]:
#private lb = 7.5
# ============================================================
# ROGII complete pipeline + GR Shape/Derivative channels (C2)
#   1) Alyaev-style synthetic generation (only if cache is missing)
#   2) fold-safe synthetic pretraining for exactly 2 epochs
#   3) optimizer-reset real-data fine-tuning
#   4) 4-fold ensemble inference
#
# C2 change (only two edits vs the baseline pipeline):
#   * e0: ResidualConvBlock(9 -> 12, base)
#   * build_image: append 3 GR first-derivative channels (dt_img, dh_img, dgr_diff)
#   The derivative is invariant to per-well GR baseline drift (32-141 API), so it
#   gives a drift-invariant matching prior aligned with the SDF zero-level set.
#   Trunk / SDF head / loss / decode / synthetic generator / all hyperparameters
#   are unchanged. Model params: 2,034,570 -> 2,035,050.
# ============================================================
import hashlib
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Config
# ============================================================
class Config:
    # Real competition data.
    TRAIN_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"
    TEST_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/test"

    # Synthetic data produced by the Alyaev-style generator embedded below.
    # This keeps the original directory name from the supplied generator.
    SYNTHETIC_DIR = "/kaggle/input/datasets/zhuyifanss/sys-6-dataset/content/synthetic_alyaev_original_train_x4"
    GENERATE_SYNTHETIC_IF_MISSING = True

    TYPEWELL_SUFFIX = "__typewell.csv"
    HORIZONTAL_SUFFIX = "__horizontal_well.csv"

    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    SEED = 42
    NUM_WORKERS = 4
    BATCH_SIZE = 4
    WEIGHT_DECAY = 1e-4
    N_FOLDS = 4
    FOLDS_TO_RUN = [0, 1, 2, 3]

    # Stage 1: fold-safe synthetic pretraining.
    PRETRAIN_EPOCHS = 2
    PRETRAIN_LR = 1e-3
    PRETRAIN_AUG_REPEATS = 4

    # Stage 2: real-data fine-tuning. This preserves the original 10-epoch run,
    # but resets AdamW and the cosine scheduler at the start of fine-tuning.
    FINETUNE_EPOCHS = 10
    FINETUNE_LR = 5e-4
    FINETUNE_AUG_REPEATS = 4

    # Grid
    H_S = 16
    H_H = 64
    H_F = 704
    T_H = 128
    T_F = 128
    H_GR_FILTER = 51

    # SDF settings.
    # Target is a signed TVT-distance field:
    #   sdf[t,h] = clip((typewell_tvt[t] - horizontal_tvt[h]) / scale_ft)
    # The correct matching path is the zero-level set.
    SDF_TARGET_SCALE_FT = 32.0
    SDF_TARGET_CLIP = 4.0
    SDF_NEAR_SIGMA_FT = 16.0
    SDF_NEAR_WEIGHT = 2.0
    SDF_DECODE_TEMP = 0.25
    HISTORY_SIGMA = 2.0

    ORIG_PAD_LEN = 16384

    # Zero-level CE / ranking loss.
    ZERO_CE_WEIGHT = 0.3
    ZERO_CE_TEMP = 0.25
    ZERO_CE_MAX_DIST_FT = 8.0
    ZERO_CE_LABEL_SIGMA_FT = 0
    ZERO_CE_TOPK = 9

    # Core augmentation.
    START_JITTER = 512
    RANDOM_CUTOFF_PROB = 0.5
    RANDOM_CUTOFF_MIN_FRAC = 0.2
    RANDOM_CUTOFF_MAX_FRAC = 0.8
    RANDOM_CUTOFF_MIN_HISTORY = 512
    RANDOM_CUTOFF_MIN_FUTURE = 512
    DZ_SCALE = 2.0

    # Deep supervision and ASPP.
    DS_WEIGHTS = [0.4, 0.2, 0.1]
    ASPP_DILATIONS = (1, 4, 8, 16)



np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
print("DEVICE:", Config.DEVICE)


# ============================================================
# Preprocess utils
# ============================================================
def resample_typewell(t, target_step=0.5):
    t_tvt = t["TVT"].values.astype(np.float64)
    t_gr = t["GR"].values.astype(np.float64)
    diffs = np.abs(np.diff(t_tvt))
    diffs = diffs[diffs > 0]
    ratio = (np.median(diffs) if len(diffs) else target_step) / target_step
    if np.isclose(ratio, 1.0):
        return t_tvt, t_gr
    if ratio < 1.0:
        group = max(int(round(1 / ratio)), 1)
        pad = (-len(t_tvt)) % group
        if pad:
            t_tvt = np.pad(t_tvt, (0, pad), mode="edge")
            t_gr = np.pad(t_gr, (0, pad), mode="edge")
        return t_tvt.reshape(-1, group).mean(1), t_gr.reshape(-1, group).mean(1)
    up = max(int(round(ratio)), 1)
    old = np.arange(len(t_tvt))
    new = np.linspace(0, len(t_tvt) - 1, (len(t_tvt) - 1) * up + 1)
    return np.interp(new, old, t_tvt), np.interp(new, old, t_gr)


def bin_mean(arr, step, back):
    arr = np.asarray(arr)
    if len(arr) == 0:
        return arr
    pad = (-len(arr)) % step
    if pad < step // 2:
        if pad:
            arr = np.pad(arr, (0, pad) if back else (pad, 0), mode="edge")
    elif pad:
        arr = arr[:-(step - pad)] if back else arr[(step - pad):]
    if len(arr) == 0:
        return arr
    return arr.reshape(-1, step).mean(1)


def safe_savgol(x, win=51, poly=2):
    x = np.asarray(x, dtype=float)
    if len(x) <= win:
        return x
    win = min(win, len(x))
    if win % 2 == 0:
        win -= 1
    if win < 7:
        return x
    return savgol_filter(x, win, poly)


def _get_numeric_col(h, name, default_value=0.0):
    if name in h.columns:
        return (
            h[name]
            .astype(float)
            .interpolate()
            .bfill()
            .ffill()
            .fillna(default_value)
            .values
        )
    return np.full(len(h), float(default_value), dtype=np.float64)


def resample_horizontal(h, step, offset=0, forced_h_ps=None):
    gr_raw = (
        h["GR"]
        .astype(float)
        .interpolate()
        .bfill()
        .ffill()
        .fillna(85.0)
        .values
    )
    gr = safe_savgol(gr_raw, Config.H_GR_FILTER, 2)

    if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
        tvt = h["TVT"].values.astype(float)
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        tvt = (
            h["TVT_input"]
            .astype(float)
            .ffill()
            .bfill()
            .fillna(0.0)
            .values
        )
    else:
        tvt = h["Z"].values.astype(float)

    z = _get_numeric_col(h, "Z", 0.0)
    md = _get_numeric_col(h, "MD", 0.0)
    if "MD" not in h.columns:
        md = np.arange(len(h), dtype=np.float64)
    x = _get_numeric_col(h, "X", 0.0)
    y = _get_numeric_col(h, "Y", 0.0)

    if forced_h_ps is not None:
        h_ps = int(forced_h_ps)
        h_ps = max(0, min(h_ps, len(h) - 1))
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        h_ps = int(np.flatnonzero(h["TVT_input"].notna().values)[-1]) + offset
        h_ps = max(0, min(h_ps, len(h) - 1))
    else:
        h_ps = len(h) // 2

    tvt0 = bin_mean(tvt[:h_ps + 1], step, back=False)
    gr0 = bin_mean(gr[:h_ps + 1], step, back=False)
    z0 = bin_mean(z[:h_ps + 1], step, back=False)
    md0 = bin_mean(md[:h_ps + 1], step, back=False)
    x0 = bin_mean(x[:h_ps + 1], step, back=False)
    y0 = bin_mean(y[:h_ps + 1], step, back=False)

    tvt1 = bin_mean(tvt[h_ps + 1:], step, back=True)
    gr1 = bin_mean(gr[h_ps + 1:], step, back=True)
    z1 = bin_mean(z[h_ps + 1:], step, back=True)
    md1 = bin_mean(md[h_ps + 1:], step, back=True)
    x1 = bin_mean(x[h_ps + 1:], step, back=True)
    y1 = bin_mean(y[h_ps + 1:], step, back=True)

    return tvt0, gr0, tvt1, gr1, h_ps, z0, z1, md0, md1, x0, x1, y0, y1


def crop_pad_1d(n, center, history, future):
    raw_i0 = center - history
    raw_i1 = center + future
    i0 = max(raw_i0, 0)
    i1 = min(raw_i1, n)
    pad_left = max(0, -raw_i0)
    pad_right = max(0, raw_i1 - n)
    return i0, i1, pad_left, pad_right


def get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt):
    if "TVT_input" in h.columns:
        val = h["TVT_input"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if "TVT" in h.columns:
        val = h["TVT"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if len(h_tvt0):
        return float(h_tvt0[-1])
    return float(t_tvt[len(t_tvt) // 2])


def get_official_h_ps(h):
    if "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        return int(np.flatnonzero(h["TVT_input"].notna().values)[-1])
    return len(h) // 2


def choose_train_cutoff(h, cfg, start_jitter=None):
    n = len(h)
    official_ps = get_official_h_ps(h)
    sj = cfg.START_JITTER if start_jitter is None else int(start_jitter)

    can_random = (
        "TVT" in h.columns
        and h["TVT"].notna().sum() > max(32, cfg.RANDOM_CUTOFF_MIN_HISTORY // 4)
        and n > cfg.RANDOM_CUTOFF_MIN_HISTORY + cfg.RANDOM_CUTOFF_MIN_FUTURE + 8
    )

    if can_random and np.random.rand() < cfg.RANDOM_CUTOFF_PROB:
        lo = max(
            int(round(n * cfg.RANDOM_CUTOFF_MIN_FRAC)),
            int(cfg.RANDOM_CUTOFF_MIN_HISTORY),
            1,
        )
        hi = min(
            int(round(n * cfg.RANDOM_CUTOFF_MAX_FRAC)),
            n - 1 - int(cfg.RANDOM_CUTOFF_MIN_FUTURE),
        )
        if hi > lo:
            return int(np.random.randint(lo, hi + 1)), "random_cutoff"

    offset = 0
    if sj > 0:
        offset = -np.random.randint(0, sj + 1)
    h_ps = int(np.clip(official_ps + offset, 0, n - 1))
    return h_ps, "official_jitter"


def make_history_line(t_seg_tvt, h_seg_tvt, hist_mask):
    diff = np.abs(t_seg_tvt[:, None] - h_seg_tvt[None, :])
    matched = diff.argmin(axis=0)
    rows = np.arange(len(t_seg_tvt))[:, None]
    line = np.exp(
        -0.5 * ((rows - matched[None, :]) / Config.HISTORY_SIGMA) ** 2
    ).astype(np.float32)
    line *= hist_mask[None, :]
    return line



# ============================================================
# Dataset
# ============================================================
class HeatmapDataset(Dataset):
    def __init__(self, files, is_train=True, aug_repeats=None, start_jitter=None):
        self.files = list(files)
        self.is_train = is_train
        default_repeats = Config.FINETUNE_AUG_REPEATS if is_train else 1
        self.aug_repeats = int(aug_repeats if aug_repeats is not None else default_repeats)
        self.start_jitter = int(start_jitter if start_jitter is not None else Config.START_JITTER)

    def __len__(self):
        if self.is_train:
            return len(self.files) * self.aug_repeats
        return len(self.files)

    def __getitem__(self, idx):
        cfg = Config
        if self.is_train:
            idx = idx % len(self.files)
        hp = Path(self.files[idx])
        sid = hp.name.split("__")[0]
        h = pd.read_csv(hp)
        t = pd.read_csv(hp.parent / f"{sid}{cfg.TYPEWELL_SUFFIX}")
        t_tvt, t_gr = resample_typewell(t, target_step=0.5)

        forced_h_ps = None
        offset = 0
        if self.is_train:
            forced_h_ps, cutoff_mode = choose_train_cutoff(
                h, cfg, start_jitter=self.start_jitter,
            )
        else:
            cutoff_mode = "official"

        h_tvt0, h_gr0, h_tvt1, h_gr1, h_ps, h_z0, h_z1, h_md0, h_md1, h_x0, h_x1, h_y0, h_y1 = resample_horizontal(
            h, cfg.H_S, offset=offset, forced_h_ps=forced_h_ps,
        )

        last_tvt_exact = get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt)
        last_tvt = float(h_tvt0[-1]) if len(h_tvt0) else last_tvt_exact
        last_idx = int(np.abs(t_tvt - last_tvt).argmin()) if len(t_tvt) else 0
        last_z = float(h["Z"].iloc[h_ps])
        last_md = float(h["MD"].iloc[h_ps]) if "MD" in h.columns else float(h_ps)
        last_x = float(h["X"].iloc[h_ps]) if "X" in h.columns else 0.0
        last_y = float(h["Y"].iloc[h_ps]) if "Y" in h.columns else 0.0

        # typewell crop
        i0, i1, pl, pr = crop_pad_1d(len(t_tvt), last_idx + 1, cfg.T_H, cfg.T_F)
        t_mask = np.pad(np.ones(i1 - i0, dtype=np.float32), (pl, pr), constant_values=0.0)
        t_seg_tvt = np.pad(t_tvt[i0:i1], (pl, pr), mode="edge")
        t_seg_gr = np.pad(t_gr[i0:i1], (pl, pr), mode="edge")

        # horizontal history crop
        i0_h, i1_h, p0l, p0r = crop_pad_1d(len(h_tvt0), len(h_tvt0), cfg.H_H, 0)
        hm0 = np.pad(np.ones(i1_h - i0_h, dtype=np.float32), (p0l, p0r), constant_values=0.0)
        if len(h_tvt0):
            ht0 = np.pad(h_tvt0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hg0 = np.pad(h_gr0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hz0 = np.pad(h_z0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hmd0 = np.pad(h_md0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hx0 = np.pad(h_x0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hy0 = np.pad(h_y0[i0_h:i1_h], (p0l, p0r), mode="edge")
        else:
            ht0 = np.zeros(cfg.H_H, dtype=np.float64)
            hg0 = np.zeros(cfg.H_H, dtype=np.float64)
            hz0 = np.full(cfg.H_H, last_z, dtype=np.float64)
            hmd0 = np.full(cfg.H_H, last_md, dtype=np.float64)
            hx0 = np.full(cfg.H_H, last_x, dtype=np.float64)
            hy0 = np.full(cfg.H_H, last_y, dtype=np.float64)

        # horizontal future crop
        i0_f, i1_f, p1l, p1r = crop_pad_1d(len(h_tvt1), 0, 0, cfg.H_F)
        hm1 = np.pad(np.ones(i1_f - i0_f, dtype=np.float32), (p1l, p1r), constant_values=0.0)
        if len(h_tvt1):
            ht1 = np.pad(h_tvt1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hg1 = np.pad(h_gr1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hz1 = np.pad(h_z1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hmd1 = np.pad(h_md1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hx1 = np.pad(h_x1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hy1 = np.pad(h_y1[i0_f:i1_f], (p1l, p1r), mode="edge")
        else:
            ht1 = np.zeros(cfg.H_F, dtype=np.float64)
            hg1 = np.zeros(cfg.H_F, dtype=np.float64)
            hz1 = np.full(cfg.H_F, last_z, dtype=np.float64)
            hmd1 = np.full(cfg.H_F, last_md, dtype=np.float64)
            hx1 = np.full(cfg.H_F, last_x, dtype=np.float64)
            hy1 = np.full(cfg.H_F, last_y, dtype=np.float64)

        h_mask = np.concatenate([hm0, hm1]).astype(np.float32)
        h_seg_tvt = np.concatenate([ht0, ht1]).astype(np.float64)
        h_seg_gr = np.concatenate([hg0, hg1]).astype(np.float64)
        h_seg_z = np.concatenate([hz0, hz1]).astype(np.float64)
        h_seg_md = np.concatenate([hmd0, hmd1]).astype(np.float64)
        h_seg_x = np.concatenate([hx0, hx1]).astype(np.float64)
        h_seg_y = np.concatenate([hy0, hy1]).astype(np.float64)

        hist_mask = np.zeros(cfg.H_H + cfg.H_F, dtype=np.float32)
        hist_mask[:cfg.H_H] = hm0

        h_dz = np.gradient(h_seg_z)
        h_dz = np.clip(h_dz / cfg.DZ_SCALE, -3.0, 3.0).astype(np.float32)

        H = cfg.H_H + cfg.H_F
        col_center = np.zeros(H, dtype=np.float64)
        for j in range(cfg.H_H):
            k = cfg.H_H - 1 - j
            col_center[j] = h_ps - k * cfg.H_S - (cfg.H_S - 1) / 2.0
        for k in range(cfg.H_F):
            col_center[cfg.H_H + k] = h_ps + 1 + k * cfg.H_S + (cfg.H_S - 1) / 2.0
        h_pos_rel = np.clip((col_center - float(h_ps)) / 4096.0, -4.0, 4.0).astype(np.float32)

        dmd = np.gradient(h_seg_md)
        dmd = np.where(np.abs(dmd) < 1e-6, 1.0, dmd)
        dx_dmd = np.gradient(h_seg_x) / dmd
        dy_dmd = np.gradient(h_seg_y) / dmd
        dxy = np.sqrt(dx_dmd ** 2 + dy_dmd ** 2) + 1e-6
        az_cos = np.clip(dx_dmd / dxy, -1.0, 1.0).astype(np.float32)
        az_sin = np.clip(dy_dmd / dxy, -1.0, 1.0).astype(np.float32)

        history_line = make_history_line(t_seg_tvt, h_seg_tvt, hist_mask)

        if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
            orig_tvt = h["TVT"].values.astype(np.float32)
        else:
            orig_tvt = np.zeros(len(h), dtype=np.float32)

        orig_len = len(orig_tvt)
        padded_tvt = np.zeros(cfg.ORIG_PAD_LEN, dtype=np.float32)
        ncopy = min(orig_len, cfg.ORIG_PAD_LEN)
        padded_tvt[:ncopy] = orig_tvt[:ncopy]

        return {
            "t_gr": torch.tensor(t_seg_gr, dtype=torch.float32),
            "h_gr": torch.tensor(h_seg_gr, dtype=torch.float32),
            "hist_mask": torch.tensor(hist_mask, dtype=torch.float32),
            "history_line": torch.tensor(history_line, dtype=torch.float32),
            "h_dz": torch.tensor(h_dz, dtype=torch.float32),
            "h_pos_rel": torch.tensor(h_pos_rel, dtype=torch.float32),
            "az_sin": torch.tensor(az_sin, dtype=torch.float32),
            "az_cos": torch.tensor(az_cos, dtype=torch.float32),
            "t_seg_tvt": torch.tensor(t_seg_tvt, dtype=torch.float32),
            "h_seg_tvt": torch.tensor(h_seg_tvt, dtype=torch.float32),
            "t_mask": torch.tensor(t_mask, dtype=torch.float32),
            "h_mask": torch.tensor(h_mask, dtype=torch.float32),
            "orig_tvt": torch.tensor(padded_tvt, dtype=torch.float32),
            "orig_len": torch.tensor(orig_len, dtype=torch.int64),
            "h_ps": torch.tensor(h_ps, dtype=torch.int64),
            "last_tvt_exact": torch.tensor(last_tvt_exact, dtype=torch.float32),
            "cutoff_mode": torch.tensor(1 if cutoff_mode == "random_cutoff" else 0, dtype=torch.int64),
        }


# ============================================================
# Model
# ============================================================

class ResidualConvBlock(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.conv1 = nn.Conv2d(ci, co, 3, padding=1, bias=False)
        self.norm1 = nn.Identity()
        self.conv2 = nn.Conv2d(co, co, 3, padding=1, bias=False)
        self.norm2 = nn.Identity()
        self.act = nn.GELU()
        self.skip = nn.Conv2d(ci, co, 1, bias=False) if ci != co else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + residual)



class LayerNorm2d(nn.Module):
    """Channel-wise LayerNorm for [B, C, T, H]."""
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(c))
        self.bias = nn.Parameter(torch.zeros(c))
        self.eps = eps

    def forward(self, x):
        u = x.mean(dim=1, keepdim=True)
        s = (x - u).pow(2).mean(dim=1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return x * self.weight[:, None, None] + self.bias[:, None, None]


class ConvNeXt2DBlock(nn.Module):
    """
    Small ConvNeXt-style residual bottleneck block.
    This is applied only after ASPP, so it refines low-resolution global features
    without changing the SDF output format or decode method.
    """
    def __init__(self, c, expansion=4, kernel=7):
        super().__init__()
        self.dw = nn.Conv2d(c, c, kernel, padding=kernel // 2, groups=c, bias=True)
        self.norm = LayerNorm2d(c)
        self.pw1 = nn.Conv2d(c, c * expansion, 1, bias=True)
        self.act = nn.GELU()
        self.pw2 = nn.Conv2d(c * expansion, c, 1, bias=True)
        # Zero-init residual scale keeps the initial network close to the original ASPP baseline.
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, x):
        y = self.dw(x)
        y = self.norm(y)
        y = self.pw2(self.act(self.pw1(y)))
        return x + self.gamma * y


class ECABlock2D(nn.Module):
    """
    Efficient Channel Attention.
    Adds almost no parameters; useful as channel reweighting after ConvNeXt bottleneck.
    """
    def __init__(self, c, k_size=5):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        # Zero-init residual scale keeps the initial network close to ConvNeXt-only.
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        y = self.avg(x).squeeze(-1).transpose(1, 2)  # [B, 1, C]
        y = self.conv(y).transpose(1, 2).unsqueeze(-1).sigmoid()
        return x + self.gamma * (x * y)


class HorizontalDilatedASPP(nn.Module):
    def __init__(self, ci, co, dilations=None):
        super().__init__()
        dilations = dilations or Config.ASPP_DILATIONS
        n_branches = len(dilations)
        branch_ch = co // n_branches
        self.branches = nn.ModuleList()
        for d in dilations:
            self.branches.append(nn.Sequential(
                nn.Conv2d(ci, branch_ch, 3, padding=(1, d), dilation=(1, d), bias=False),
                nn.Identity(),
                nn.GELU(),
            ))
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ci, branch_ch, 1, bias=False),
            nn.Identity(),
            nn.GELU(),
        )
        total_ch = branch_ch * (n_branches + 1)
        self.fuse = nn.Sequential(
            nn.Conv2d(total_ch, co, 1, bias=False),
            nn.Identity(),
            nn.GELU(),
        )

    def forward(self, x):
        outs = [branch(x) for branch in self.branches]
        gp = self.global_pool(x)
        gp = gp.expand(-1, -1, x.shape[2], x.shape[3])
        outs.append(gp)
        return self.fuse(torch.cat(outs, dim=1))


class HeatmapResUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.gr_norm = nn.InstanceNorm2d(2)

        # Encoder
        self.e0 = ResidualConvBlock(12, base)   # 9 baseline + 3 GR-derivative (C2)
        self.e1 = ResidualConvBlock(base, base * 2)
        self.e2 = ResidualConvBlock(base * 2, base * 4)
        self.e3 = ResidualConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        # ASPP bottleneck on pooled encoder feature.
        self.bott = HorizontalDilatedASPP(base * 8, base * 16)

        # Fixed convnext_eca bottleneck refinement.
        # Search result: ConvNeXt bottleneck + ECA is a simple, strong architecture change.
        self.cn_bott = ConvNeXt2DBlock(base * 16, kernel=7)
        self.eca_bott = ECABlock2D(base * 16, k_size=5)

        # Decoder
        self.u3 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.d3 = ResidualConvBlock(base * 16, base * 8)
        self.u2 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.d2 = ResidualConvBlock(base * 8, base * 4)
        self.u1 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.d1 = ResidualConvBlock(base * 4, base * 2)
        self.u0 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.d0 = ResidualConvBlock(base * 2, base)

        # Head
        self.head = nn.Conv2d(base, 1, 1)
        self.aux3 = nn.Conv2d(base * 8, 1, 1)
        self.aux2 = nn.Conv2d(base * 4, 1, 1)
        self.aux1 = nn.Conv2d(base * 2, 1, 1)

    def build_image(self, batch):
        dev = next(self.parameters()).device
        t_gr = batch["t_gr"].to(dev)
        h_gr = batch["h_gr"].to(dev)
        hist_mask = batch["hist_mask"].to(dev)
        history = batch["history_line"].to(dev)
        h_dz = batch["h_dz"].to(dev)
        B, T = t_gr.shape
        _, H = h_gr.shape
        t_img = t_gr.view(B, 1, T, 1).expand(B, 1, T, H)
        h_img = h_gr.view(B, 1, 1, H).expand(B, 1, T, H)
        gr_pair = self.gr_norm(torch.cat([t_img, h_img], dim=1))
        gr_diff = torch.clamp((t_img - h_img) / 40.0, -4.0, 4.0)
        mask_img = hist_mask.view(B, 1, 1, H).expand(B, 1, T, H)
        history_img = history[:, None, :, :]
        dz_img = h_dz.view(B, 1, 1, H).expand(B, 1, T, H)
        h_pos_rel = batch["h_pos_rel"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_sin = batch["az_sin"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_cos = batch["az_cos"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)

        # --- C2: GR shape/derivative channels (drift-invariant matching) ---
        # First difference along depth of each GR trace, normalized (~10 API/step) and
        # clamped. dt broadcast down rows, dh across columns, plus their outer difference.
        dt = torch.zeros_like(t_gr)
        dt[:, 1:] = t_gr[:, 1:] - t_gr[:, :-1]
        dt = torch.clamp(dt / 10.0, -4.0, 4.0)
        dh = torch.zeros_like(h_gr)
        dh[:, 1:] = h_gr[:, 1:] - h_gr[:, :-1]
        dh = torch.clamp(dh / 10.0, -4.0, 4.0)
        dt_img = dt.view(B, 1, T, 1).expand(B, 1, T, H)
        dh_img = dh.view(B, 1, 1, H).expand(B, 1, T, H)
        dgr_diff = torch.clamp(dt_img - dh_img, -4.0, 4.0)

        return torch.cat([gr_pair, gr_diff, mask_img, history_img, dz_img,
                          h_pos_rel, az_sin, az_cos, dt_img, dh_img, dgr_diff], dim=1)

    def forward(self, batch):
        x = self.build_image(batch)

        # Encoder
        e0 = self.e0(x)
        e1 = self.e1(self.pool(e0))
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))

        # Bottleneck: pool -> ASPP -> ConvNeXt -> ECA
        pooled_e3 = self.pool(e3)
        b = self.bott(pooled_e3)
        b = self.cn_bott(b)
        b = self.eca_bott(b)

        # Decoder
        d3 = self.d3(torch.cat([self.u3(b), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        d0 = self.d0(torch.cat([self.u0(d1), e0], dim=1))

        logits = self.head(d0)

        if self.training:
            full_h, full_w = x.shape[2], x.shape[3]
            a3 = self.aux3(d3)
            a2 = self.aux2(d2)
            a1 = self.aux1(d1)
            a3 = F.interpolate(a3, size=(full_h, full_w), mode="bilinear", align_corners=False)
            a2 = F.interpolate(a2, size=(full_h, full_w), mode="bilinear", align_corners=False)
            a1 = F.interpolate(a1, size=(full_h, full_w), mode="bilinear", align_corners=False)
            return logits, a1, a2, a3
        return logits


# ============================================================
# Loss
# ============================================================
def make_sdf_target(logits, batch):
    """Signed TVT-distance field target."""
    dev = logits.device
    t_seg_tvt = batch["t_seg_tvt"].to(dev).float()  # [B, T]
    h_seg_tvt = batch["h_seg_tvt"].to(dev).float()  # [B, H]
    t_mask = batch["t_mask"].to(dev).float()        # [B, T]
    h_mask = batch["h_mask"].to(dev).float()        # [B, H]

    signed_ft = t_seg_tvt[:, None, :, None] - h_seg_tvt[:, None, None, :]
    target = signed_ft / float(Config.SDF_TARGET_SCALE_FT)
    target = target.clamp(-float(Config.SDF_TARGET_CLIP), float(Config.SDF_TARGET_CLIP))
    mask = t_mask[:, None, :, None] * h_mask[:, None, None, :]
    abs_dist = signed_ft.abs()
    near_weight = 1.0 + float(Config.SDF_NEAR_WEIGHT) * torch.exp(
        -(abs_dist ** 2) / (2.0 * float(Config.SDF_NEAR_SIGMA_FT) ** 2)
    )
    return target, mask, near_weight


def sdf_loss_single(pred_sdf, batch):
    """Weighted SmoothL1 regression of the signed distance field."""
    target, mask, near_weight = make_sdf_target(pred_sdf, batch)
    pred_sdf = pred_sdf.float().clamp(
        -float(Config.SDF_TARGET_CLIP) * 1.5,
        float(Config.SDF_TARGET_CLIP) * 1.5,
    )
    loss = F.smooth_l1_loss(pred_sdf, target, reduction="none", beta=0.25)
    weight = mask * near_weight
    return (loss * weight).sum() / (weight.sum() + 1e-8)


def zero_ce_loss_single(pred_sdf, batch):
    """
    Zero-level CE / ranking loss.

    For every future horizontal column h we build row-logits over the typewell
    axis: score[t] = -|sdf[t, h]| / temp. The label is the true crossing row
    target_row[h] = argmin_t |t_tvt[t] - h_tvt[h]|. Cross-entropy on these
    logits forces the true row to have the smallest |sdf| (i.e. be ranked #1),
    directly suppressing false zero-levels and lifting gt_topk_hit.

    Only future columns whose crossing lies inside the typewell window
    (min_dist < ZERO_CE_MAX_DIST_FT) are supervised; out-of-window columns are
    left to the (saturated) regression term.
    """
    dev = pred_sdf.device
    t_seg_tvt = batch["t_seg_tvt"].to(dev).float()   # [B, T]
    h_seg_tvt = batch["h_seg_tvt"].to(dev).float()   # [B, H]
    t_mask = batch["t_mask"].to(dev).float()         # [B, T]
    h_mask = batch["h_mask"].to(dev).float()         # [B, H]

    pred = pred_sdf.float().squeeze(1)               # [B, T, H]
    B, T, H = pred.shape

    row_valid = t_mask > 0                           # [B, T]

    # row-logits per column
    score = -pred.abs() / float(Config.ZERO_CE_TEMP)          # [B, T, H]
    score = score.masked_fill(~row_valid[:, :, None], -1e4)
    logp = F.log_softmax(score, dim=1)                        # [B, T, H]

    # ground-truth crossing row per column
    dist = (t_seg_tvt[:, :, None] - h_seg_tvt[:, None, :]).abs()   # [B, T, H]
    dist = dist.masked_fill(~row_valid[:, :, None], 1e9)
    target_row = dist.argmin(dim=1)                          # [B, H]
    min_dist = dist.gather(1, target_row[:, None, :]).squeeze(1)   # [B, H]

    # supervise only future columns with an in-window crossing
    future = torch.zeros(H, device=dev, dtype=torch.bool)
    future[Config.H_H:] = True
    col_valid = (h_mask > 0) & future[None, :] & (min_dist < float(Config.ZERO_CE_MAX_DIST_FT))

    if float(Config.ZERO_CE_LABEL_SIGMA_FT) > 0.0:
        # Gaussian soft label over rows (softer, less conflict with SmoothL1)
        sigma = float(Config.ZERO_CE_LABEL_SIGMA_FT)
        soft = torch.exp(-(dist ** 2) / (2.0 * sigma * sigma))
        soft = soft.masked_fill(~row_valid[:, :, None], 0.0)
        soft = soft / (soft.sum(dim=1, keepdim=True) + 1e-8)
        nll = -(soft * logp).sum(dim=1)                      # [B, H]
    else:
        # hard CE (spec)
        nll = -logp.gather(1, target_row[:, None, :]).squeeze(1)   # [B, H]

    nll = nll * col_valid.float()
    return nll.sum() / (col_valid.float().sum() + 1e-8)


def sdf_loss(model_out, batch):
    """
    Total loss = weighted SDF SmoothL1 (+ deep supervision) + ZERO_CE_WEIGHT * zero-level CE.
    Returns (total, reg, ce) so the training loop can log the two terms separately.
    """
    if isinstance(model_out, tuple):
        pred_main, a1, a2, a3 = model_out
        w1, w2, w3 = Config.DS_WEIGHTS
        reg = (
            sdf_loss_single(pred_main, batch)
            + w1 * sdf_loss_single(a1, batch)
            + w2 * sdf_loss_single(a2, batch)
            + w3 * sdf_loss_single(a3, batch)
        )
    else:
        pred_main = model_out
        reg = sdf_loss_single(pred_main, batch)

    # CE ranking is applied on the main (full-resolution) head only.
    ce = zero_ce_loss_single(pred_main, batch)
    total = reg + float(Config.ZERO_CE_WEIGHT) * ce
    return total, reg, ce


# ============================================================
# Decode / Eval
# ============================================================
def map_to_original(pred_H, orig_len, h_ps, last_tvt_exact):
    centers = []
    values = []
    for k in range(Config.H_H):
        centers.append(h_ps - k * Config.H_S - (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H - 1 - k])
    centers.append(float(h_ps))
    values.append(float(last_tvt_exact))
    for k in range(Config.H_F):
        centers.append(h_ps + 1 + k * Config.H_S + (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H + k])
    centers = np.asarray(centers, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    order = np.argsort(centers)
    centers = centers[order]
    values = values[order]
    keep = np.r_[True, np.diff(centers) > 1e-6]
    return np.interp(np.arange(orig_len), centers[keep], values[keep])


def decode_sdf(pred_sdf, t_seg_tvt, h_seg_tvt, t_mask, orig_len, h_ps, last_tvt_exact):
    """Pure soft-argmax decode: per future column, TVT = softmax(-|sdf|/temp) . t_tvt."""
    score = -np.abs(pred_sdf.astype(np.float64)) / float(Config.SDF_DECODE_TEMP)
    if t_mask is not None:
        score = np.where(t_mask[:, None] > 0, score, -1e9)
    score = score - score.max(axis=0, keepdims=True)
    prob = np.exp(score)
    prob = prob / (prob.sum(axis=0, keepdims=True) + 1e-12)
    pred_H = (prob * t_seg_tvt[:, None]).sum(axis=0).astype(np.float64)
    pred_H[:Config.H_H] = h_seg_tvt[:Config.H_H]
    return map_to_original(pred_H, orig_len, h_ps, last_tvt_exact)


def compute_topk_hit(pred_sdf, t_seg_tvt, h_seg_tvt, t_mask, h_mask, K):
    """
    Fraction bookkeeping for gt_top-k hit: for each in-window future column,
    is the true crossing row among the K rows with smallest |sdf|?
    Returns (hit_count, valid_column_count).
    """
    T, H = pred_sdf.shape
    valid_t = np.asarray(t_mask) > 0
    abs_sdf = np.abs(pred_sdf.astype(np.float64))
    abs_sdf = np.where(valid_t[:, None], abs_sdf, np.inf)

    dist = np.abs(t_seg_tvt[:, None] - h_seg_tvt[None, :])
    dist = np.where(valid_t[:, None], dist, np.inf)
    target_row = dist.argmin(axis=0)                 # [H]
    min_dist = dist[target_row, np.arange(H)]

    future = np.zeros(H, dtype=bool)
    future[Config.H_H:] = True
    col_valid = (np.asarray(h_mask) > 0) & future & (min_dist < float(Config.ZERO_CE_MAX_DIST_FT))
    if not col_valid.any():
        return 0, 0

    k = min(K, T - 1)
    topk_idx = np.argpartition(abs_sdf, k, axis=0)[:k, :]    # [k, H]
    hit_mask = (topk_idx == target_row[None, :]).any(axis=0) # [H]
    return int((hit_mask & col_valid).sum()), int(col_valid.sum())


@torch.no_grad()
def evaluate(model, loader):
    """Returns (rmse, gt_topk_hit)."""
    model.eval()
    se = 0.0
    n = 0
    hit = 0
    hit_n = 0
    K = Config.ZERO_CE_TOPK
    for batch in loader:
        logits = model(batch)
        logits = logits.squeeze(1).cpu().numpy()
        t_seg = batch["t_seg_tvt"].numpy()
        h_seg = batch["h_seg_tvt"].numpy()
        t_mask = batch["t_mask"].numpy()
        h_mask = batch["h_mask"].numpy()
        y = batch["orig_tvt"].numpy()
        orig_len = batch["orig_len"].numpy()
        h_ps = batch["h_ps"].numpy()
        last = batch["last_tvt_exact"].numpy()
        for b in range(logits.shape[0]):
            olen = int(orig_len[b])
            ps = int(h_ps[b])
            if ps >= olen - 1:
                continue
            pred = decode_sdf(
                logits[b], t_seg[b], h_seg[b], t_mask[b],
                olen, ps, float(last[b]),
            )
            err = pred[ps + 1:] - y[b, :olen][ps + 1:]
            se += np.sum(err ** 2)
            n += len(err)

            h_hit, h_cnt = compute_topk_hit(
                logits[b], t_seg[b], h_seg[b], t_mask[b], h_mask[b], K,
            )
            hit += h_hit
            hit_n += h_cnt
    rmse = np.sqrt(se / max(n, 1))
    topk = hit / max(hit_n, 1)
    return rmse, topk


@torch.no_grad()
def evaluate_last_tvt(loader):
    se = 0.0
    n = 0
    for batch in loader:
        y = batch["orig_tvt"].numpy()
        orig_len = batch["orig_len"].numpy()
        h_ps = batch["h_ps"].numpy()
        for b in range(len(orig_len)):
            olen = int(orig_len[b])
            ps = int(h_ps[b])
            if ps >= olen - 1:
                continue
            true = y[b, :olen]
            err = true[ps] - true[ps + 1:]
            se += np.sum(err ** 2)
            n += len(err)
    return np.sqrt(se / max(n, 1))


# ============================================================
# Alyaev-original-style ROGII synthetic generator
#
# Key idea:
#   generated position = synthetic geo_delta / stratigraphic surface delta
#
#   geo_ps  = last_tvt + z_ps
#   TVT_syn = geo_ps + position - Z_future
#   GR_syn  = interp(typewell_GR, TVT_syn) + synthetic colored noise
#
# This version does NOT use real future TVT or real future TVT range.
# ============================================================


# ============================================================
# Config
# ============================================================

class SynConfig:
    SEED = 20260627

    # Generate multiple independent synthetic variants for each real well.
    # First expansion run: 4x is safer than jumping directly to 8x/16x.
    SYN_PER_WELL = 6

    # Alyaev-like random curve process
    ANGLE_MIN_DEG = 72.0
    ANGLE_MAX_DEG = 108.0

    ANGLE_RANDOM_UNIFORM = 0.010       # original: (random - 0.5) * 0.01
    MAX_ANGLE_CHANGE = 0.035           # original: _max_angle_change = 0.035

    # Fault-like jump in geo_delta domain.
    # For first pretrain run, keep it low.
    FAULT_PROB = 0
    FAULT_MAX_FT = 10.0

    # synthetic TVT-delta trend distribution, independent of true future TVT
    # mixture controls final target TVT range.
    P_FLAT = 0.25
    P_MEDIUM = 0.55
    P_LARGE = 0.18
    P_STRESS = 0.02

    FLAT_RANGE = (8.0, 22.0)
    MEDIUM_RANGE = (22.0, 55.0)
    LARGE_RANGE = (55.0, 95.0)
    STRESS_RANGE = (95.0, 130.0)

    N_CTRL_MIN = 4
    N_CTRL_MAX = 7
    CTRL_BEND_FRAC = 0.25

    # optional local stretch/squeeze of the trend
    WARP_STRENGTH = 0.10

    # GR synthetic noise, based only on typewell statistics
    GR_NOISE_STD_FRAC_LOW = 0.03
    GR_NOISE_STD_FRAC_HIGH = 0.10
    GR_AR_PHI_LOW = 0.70
    GR_AR_PHI_HIGH = 0.95

    GR_SCALE_LOW = 0.98
    GR_SCALE_HIGH = 1.02
    GR_SHIFT_LOW = -2.0
    GR_SHIFT_HIGH = 2.0

    # synthetic missing mask
    POINT_MISSING_PROB_LOW = 0.00
    POINT_MISSING_PROB_HIGH = 0.08
    SEGMENT_MISSING_PROB = 0.25
    SEGMENT_MISSING_LEN = (30, 250)
    HIGH_MISSING_PROB = 0.08
    HIGH_MISSING_FRAC = (0.25, 0.60)

    # typewell boundary
    TVT_MARGIN_FT = 2.0

    # QC
    SYN_TVT_RANGE_MIN = 6.0
    SYN_TVT_RANGE_MAX = 140.0
    MAX_ABS_DTVT_STEP = 3.0
    MAX_ABS_DDTVT_STEP = 1.5

    MIN_FORWARD_GR_STD = 1.0
    MIN_CORR_GR_CLEAN_SYN = 0.75

    # Extra fake-short guard: very small TVT range must still be explained by
    # forward GR rather than mostly synthetic noise.
    FAKE_SHORT_RANGE_FT = 8.0
    FAKE_SHORT_MIN_CORR = 0.85
    FAKE_SHORT_MIN_CLEAN_TO_SYN_STD_RATIO = 0.60

    MAX_TRIES = 50

    # if fail all attempts, skip instead of saving bad sample
    SKIP_IF_FAILED = True


# ============================================================
# Basic utils
# ============================================================

def cot(angle):
    return math.cos(angle) / math.sin(angle)


def angle_from_cot(c):
    # cot(theta)=c -> theta = atan2(1, c)
    return math.atan2(1.0, float(c))


def smooth1d(y, win=101):
    y = np.asarray(y, dtype=float)
    win = int(win)

    if win < 3:
        return y.copy()

    if win % 2 == 0:
        win += 1

    if win >= len(y):
        win = len(y) if len(y) % 2 == 1 else len(y) - 1

    if win < 3:
        return y.copy()

    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    kernel = np.ones(win, dtype=float) / win
    return np.convolve(yp, kernel, mode="valid")


def p95_abs_diff(y, order=1):
    y = np.asarray(y, dtype=float)
    if len(y) <= order:
        return np.nan

    d = y.copy()
    for _ in range(order):
        d = np.diff(d)

    return float(np.nanpercentile(np.abs(d), 95))


def get_ps_index(hw):
    """
    PS = last known TVT_input.
    This does NOT use future TVT.
    """
    if "TVT_input" in hw.columns and hw["TVT_input"].notna().any():
        return int(np.flatnonzero(hw["TVT_input"].notna().to_numpy())[-1])

    raise ValueError("No TVT_input found; cannot anchor synthetic TVT without PS.")


def find_train_pairs(train_dir):
    train_dir = Path(train_dir)

    typewell_files = []
    typewell_files += list(train_dir.glob("*__typewell.csv"))
    typewell_files += list(train_dir.glob("*_typewell.csv"))
    typewell_files += list(train_dir.rglob("*__typewell.csv"))
    typewell_files += list(train_dir.rglob("*_typewell.csv"))

    typewell_files = sorted(set(typewell_files))

    pairs = []
    for tw_path in typewell_files:
        name = tw_path.name

        if name.endswith("__typewell.csv"):
            sid = name.replace("__typewell.csv", "")
            hw_name = f"{sid}__horizontal_well.csv"
        elif name.endswith("_typewell.csv"):
            sid = name.replace("_typewell.csv", "")
            hw_name = f"{sid}_horizontal_well.csv"
        else:
            continue

        hw_path = tw_path.parent / hw_name

        if not hw_path.exists():
            cand = list(train_dir.rglob(f"{sid}__horizontal_well.csv"))
            cand += list(train_dir.rglob(f"{sid}_horizontal_well.csv"))

            if len(cand) == 0:
                continue

            hw_path = cand[0]

        pairs.append((sid, str(tw_path), str(hw_path)))

    return pairs


# ============================================================
# Synthetic curve generation
# ============================================================

def sample_target_tvt_delta_trend(x_norm, rng, cfg=SynConfig):
    """
    Generate a random TVT_delta trend without using real future TVT.

    This is only a guide. It is converted to:
        geo_delta_trend = z_delta + tvt_delta_trend

    Then the Alyaev angle process generates the actual geo_delta curve.
    """
    u = rng.random()

    if u < cfg.P_FLAT:
        target_range = rng.uniform(*cfg.FLAT_RANGE)
    elif u < cfg.P_FLAT + cfg.P_MEDIUM:
        target_range = rng.uniform(*cfg.MEDIUM_RANGE)
    elif u < cfg.P_FLAT + cfg.P_MEDIUM + cfg.P_LARGE:
        target_range = rng.uniform(*cfg.LARGE_RANGE)
    else:
        target_range = rng.uniform(*cfg.STRESS_RANGE)

    sign = rng.choice([-1.0, 1.0])

    # end drift uses a fraction of target_range; internal bends create full range.
    end_delta = sign * rng.uniform(0.35, 1.00) * target_range

    k = int(rng.integers(cfg.N_CTRL_MIN, cfg.N_CTRL_MAX + 1))
    ctrl_x = np.linspace(0.0, 1.0, k)

    ctrl_y = np.linspace(0.0, end_delta, k)

    # low-frequency bends
    bend_amp = cfg.CTRL_BEND_FRAC * target_range
    bend = rng.normal(0.0, bend_amp, size=k)
    bend[0] = 0.0
    bend[-1] = 0.0

    ctrl_y = ctrl_y + bend

    # mild monotone-ish warp in x for stretch/squeeze
    if cfg.WARP_STRENGTH > 0:
        warp_ctrl = np.linspace(0, 1, k)
        warp_noise = rng.normal(0, cfg.WARP_STRENGTH, size=k)
        warp_noise[0] = 0.0
        warp_noise[-1] = 0.0
        warped_x = ctrl_x + warp_noise
        warped_x = np.clip(warped_x, 0, 1)
        warped_x = np.sort(warped_x)
        warped_x[0] = 0.0
        warped_x[-1] = 1.0
        ctrl_x = warped_x

    tvt_delta = np.interp(x_norm, ctrl_x, ctrl_y)

    # smooth low frequency
    L = len(x_norm)
    tvt_delta = smooth1d(
        tvt_delta,
        win=max(31, min(151, L // 25 * 2 + 1)),
    )

    # anchor start to 0
    tvt_delta = tvt_delta - tvt_delta[0]

    return tvt_delta, target_range


def generate_ar_noise(n, std, phi, rng):
    """
    Colored GR noise. Does not use true horizontal residual.
    """
    eps = rng.normal(0.0, std * math.sqrt(max(1e-6, 1 - phi ** 2)), size=n)
    y = np.zeros(n, dtype=float)

    for i in range(1, n):
        y[i] = phi * y[i - 1] + eps[i]

    y = y - np.mean(y)
    if np.std(y) > 1e-6:
        y = y / np.std(y) * std

    return y


def make_synthetic_missing_mask(n, rng, cfg=SynConfig):
    """
    Synthetic missing mask, not copied from horizontal well future GR.
    """
    if rng.random() < cfg.HIGH_MISSING_PROB:
        point_prob = rng.uniform(*cfg.HIGH_MISSING_FRAC)
    else:
        point_prob = rng.uniform(cfg.POINT_MISSING_PROB_LOW, cfg.POINT_MISSING_PROB_HIGH)

    mask = rng.random(n) < point_prob

    if rng.random() < cfg.SEGMENT_MISSING_PROB and n > 200:
        n_seg = int(rng.integers(1, 4))
        for _ in range(n_seg):
            seg_len = int(rng.integers(cfg.SEGMENT_MISSING_LEN[0], cfg.SEGMENT_MISSING_LEN[1] + 1))
            seg_len = min(seg_len, n)
            start = int(rng.integers(0, max(1, n - seg_len + 1)))
            mask[start:start + seg_len] = True

    return mask


# ============================================================
# One synthetic sample
# ============================================================

def _generate_once_original_style(typewell_csv, horizontal_csv, seed, cfg=SynConfig):
    rng = np.random.default_rng(seed)

    tw = pd.read_csv(typewell_csv).sort_values("TVT").reset_index(drop=True)
    hw = pd.read_csv(horizontal_csv).reset_index(drop=True)

    t_tvt = tw["TVT"].astype(float).to_numpy()
    t_gr = tw["GR"].astype(float).to_numpy()

    h_md = hw["MD"].astype(float).to_numpy()
    h_z = hw["Z"].astype(float).to_numpy()

    h_ps = get_ps_index(hw)

    idx_f = np.arange(h_ps + 1, len(hw))
    if len(idx_f) < 32:
        return None, None

    # Anchor only from TVT_input prefix
    tvt_input = hw["TVT_input"].astype(float).to_numpy()
    last_tvt = float(tvt_input[h_ps])

    md_ps = float(h_md[h_ps])
    z_ps = float(h_z[h_ps])
    geo_ps = last_tvt + z_ps

    md_f = h_md[idx_f]
    z_f = h_z[idx_f]

    L = len(idx_f)

    x_dist = md_f - md_ps
    x_norm = (x_dist - x_dist.min()) / (x_dist.max() - x_dist.min() + 1e-12)

    dmd = np.diff(np.r_[md_ps, md_f])
    dmd = np.maximum(dmd, 1e-6)

    z_delta = z_f - z_ps

    # --------------------------------------------------------
    # 1. Generate random TVT_delta trend independent of truth
    # --------------------------------------------------------

    tvt_delta_trend, target_range = sample_target_tvt_delta_trend(
        x_norm=x_norm,
        rng=rng,
        cfg=cfg,
    )

    # Key mapping:
    # position is geo_delta, therefore guide is z_delta + tvt_delta.
    geo_delta_trend = z_delta + tvt_delta_trend
    geo_delta_trend = smooth1d(
        geo_delta_trend,
        win=max(31, min(151, L // 25 * 2 + 1)),
    )

    expected_cot = np.gradient(geo_delta_trend, md_f)
    expected_cot = smooth1d(
        expected_cot,
        win=max(31, min(101, L // 35 * 2 + 1)),
    )

    # --------------------------------------------------------
    # 2. Alyaev-style SubsurfaceState process
    # --------------------------------------------------------

    angle_min = math.radians(cfg.ANGLE_MIN_DEG)
    angle_max = math.radians(cfg.ANGLE_MAX_DEG)

    positions = np.zeros(L, dtype=float)
    angles = np.zeros(L, dtype=float)

    angle = angle_from_cot(expected_cot[0])
    angle = float(np.clip(angle, angle_min, angle_max))

    pos = 0.0
    positions[0] = pos
    angles[0] = angle

    fault_count = 0

    for i in range(1, L):
        new_thl_distance = float(dmd[i])

        # original-like random angle perturbation
        new_angle = angle + (rng.random() - 0.5) * cfg.ANGLE_RANDOM_UNIFORM

        expected_cot_i = float(expected_cot[i])
        new_cot = cot(new_angle)

        delta_angle = 0.0

        # same logic as Alyaev:
        # if current cot is far from expected cot, push angle toward expected trend
        if (0.5 + rng.random()) * 0.03 < abs(expected_cot_i - new_cot) and rng.random() > 0.2:
            sign = -np.sign(expected_cot_i - new_cot)
            magnitude = abs(expected_cot_i - new_cot) * (1.0 + (rng.random() - 0.5) * 0.8)
            delta_angle = sign * min(cfg.MAX_ANGLE_CHANGE, magnitude)

        new_angle = new_angle + delta_angle
        new_angle = float(np.clip(new_angle, angle_min, angle_max))

        # integrate position
        new_position = pos + new_thl_distance * cot(new_angle)

        # original-like fault/position correction toward expected position
        expected_position = float(geo_delta_trend[i] - geo_delta_trend[0])

        delta = 0.0
        if rng.random() < cfg.FAULT_PROB:
            mismatch = abs(expected_position - new_position)
            if mismatch > 1e-6:
                sign = np.sign(expected_position - new_position)
                magnitude = mismatch * (1.0 + (rng.random() - 0.5))
                delta = sign * min(cfg.FAULT_MAX_FT, magnitude)
                fault_count += 1

        new_position = new_position + delta

        angle = new_angle
        pos = new_position

        positions[i] = pos
        angles[i] = angle

    geo_delta_syn = geo_delta_trend[0] + positions

    # --------------------------------------------------------
    # 3. Convert geo_delta -> TVT
    # --------------------------------------------------------

    tvt_syn_f = geo_ps + geo_delta_syn - z_f

    # exact PS continuity taper
    delta0 = tvt_syn_f[0] - last_tvt
    taper = np.exp(-np.linspace(0, 7, L))
    tvt_syn_f = tvt_syn_f - delta0 * taper

    geo_delta_syn = tvt_syn_f + z_f - geo_ps

    # boundary QC, no clipping in accepted sample
    clip_min = float(t_tvt.min() + cfg.TVT_MARGIN_FT)
    clip_max = float(t_tvt.max() - cfg.TVT_MARGIN_FT)

    out_of_bounds = int(np.sum((tvt_syn_f < clip_min) | (tvt_syn_f > clip_max)))

    # --------------------------------------------------------
    # 4. Forward GR from typewell + synthetic colored noise
    # --------------------------------------------------------

    gr_clean = np.interp(tvt_syn_f, t_tvt, t_gr)

    typewell_gr_std = float(np.nanstd(t_gr))
    noise_std = rng.uniform(cfg.GR_NOISE_STD_FRAC_LOW, cfg.GR_NOISE_STD_FRAC_HIGH) * max(typewell_gr_std, 1.0)
    phi = rng.uniform(cfg.GR_AR_PHI_LOW, cfg.GR_AR_PHI_HIGH)

    noise = generate_ar_noise(L, noise_std, phi, rng)

    gr_syn = gr_clean + noise

    gr_syn = (
        gr_syn * rng.uniform(cfg.GR_SCALE_LOW, cfg.GR_SCALE_HIGH)
        + rng.uniform(cfg.GR_SHIFT_LOW, cfg.GR_SHIFT_HIGH)
    )

    missing_mask = make_synthetic_missing_mask(L, rng, cfg)
    gr_syn_raw = gr_syn.copy()
    gr_syn_raw[missing_mask] = np.nan

    # --------------------------------------------------------
    # 5. Build synthetic horizontal well
    # --------------------------------------------------------

    syn_hw = hw.copy()

    # Future target becomes synthetic.
    # Prefix remains original known history.
    syn_hw.loc[idx_f, "TVT"] = tvt_syn_f
    syn_hw.loc[idx_f, "GR"] = gr_syn_raw

    # Keep TVT_input unchanged.
    # This keeps the anchor/prefix exactly like the real task.

    # --------------------------------------------------------
    # 6. Stats / QC
    # --------------------------------------------------------

    syn_tvt_range = float(np.nanmax(tvt_syn_f) - np.nanmin(tvt_syn_f))
    gr_clean_std = float(np.nanstd(gr_clean))
    gr_syn_std = float(np.nanstd(gr_syn))
    noise_std_actual = float(np.nanstd(noise))

    corr = np.nan
    if L > 2 and gr_clean_std > 1e-6 and gr_syn_std > 1e-6:
        corr = float(np.corrcoef(gr_clean, gr_syn)[0, 1])

    stats = {
        "ps_index": int(h_ps),
        "ps_md": float(md_ps),
        "future_rows": int(L),

        "target_range": float(target_range),
        "syn_range_tvt": syn_tvt_range,
        "syn_end_delta_tvt": float(tvt_syn_f[-1] - last_tvt),
        "syn_geo_delta_range": float(np.nanmax(geo_delta_syn) - np.nanmin(geo_delta_syn)),

        "abs_dtvt_p95": p95_abs_diff(tvt_syn_f, 1),
        "abs_ddtvt_p95": p95_abs_diff(tvt_syn_f, 2),

        "gr_clean_std": gr_clean_std,
        "gr_syn_std": gr_syn_std,
        "noise_std": noise_std_actual,
        "corr_gr_clean_syn": corr,

        "missing_frac": float(np.mean(missing_mask)),
        "out_of_bounds_count": out_of_bounds,
        "fault_count": int(fault_count),
    }

    return syn_hw, stats


def generate_one_original_style(
    typewell_csv,
    horizontal_csv,
    out_csv=None,
    seed=SynConfig.SEED,
    cfg=SynConfig,
):
    """
    Regenerate until QC passes.
    Does NOT use true future TVT.
    """
    for attempt in range(cfg.MAX_TRIES):
        syn_hw, stats = _generate_once_original_style(
            typewell_csv=typewell_csv,
            horizontal_csv=horizontal_csv,
            seed=seed + attempt * 10007,
            cfg=cfg,
        )

        if syn_hw is None:
            return None, None

        ok = True

        ok &= stats["out_of_bounds_count"] == 0
        ok &= cfg.SYN_TVT_RANGE_MIN <= stats["syn_range_tvt"] <= cfg.SYN_TVT_RANGE_MAX
        ok &= stats["abs_dtvt_p95"] <= cfg.MAX_ABS_DTVT_STEP
        ok &= stats["abs_ddtvt_p95"] <= cfg.MAX_ABS_DDTVT_STEP
        ok &= stats["gr_clean_std"] >= cfg.MIN_FORWARD_GR_STD
        ok &= stats["corr_gr_clean_syn"] >= cfg.MIN_CORR_GR_CLEAN_SYN

        # Guard against fake-short samples: if TVT barely moves, GR should not
        # be dominated by noise. This keeps real flat wells but rejects noisy
        # synthetic flat wells.
        clean_to_syn_ratio = stats["gr_clean_std"] / (stats["gr_syn_std"] + 1e-6)
        if stats["syn_range_tvt"] < cfg.FAKE_SHORT_RANGE_FT:
            ok &= stats["corr_gr_clean_syn"] >= cfg.FAKE_SHORT_MIN_CORR
            ok &= clean_to_syn_ratio >= cfg.FAKE_SHORT_MIN_CLEAN_TO_SYN_STD_RATIO

        if ok:
            stats["accepted_attempt"] = attempt
            if out_csv is not None:
                syn_hw.to_csv(out_csv, index=False)
            return syn_hw, stats

    if cfg.SKIP_IF_FAILED:
        return None, None

    stats["accepted_attempt"] = -1
    if out_csv is not None:
        syn_hw.to_csv(out_csv, index=False)

    return syn_hw, stats


# ============================================================
# Directory generation
# ============================================================

def generate_synthetic_train_dir_original_style(
    train_dir=Config.TRAIN_DIR,
    out_dir=Config.SYNTHETIC_DIR,
    max_wells=None,
    seed=SynConfig.SEED,
):
    """
    Generate SYN_PER_WELL independent synthetic variants per real well.

    Output naming:
        source sid: 000d7d20
        variant sid: 000d7d20_syn00, 000d7d20_syn01, ...

    This naming is important because the training Dataset looks for:
        {sid}__horizontal_well.csv
        {sid}__typewell.csv
    so each synthetic variant needs its own matching typewell filename.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pairs = find_train_pairs(train_dir)
    print(f"Found train pairs: {len(pairs)}")
    print(f"Synthetic variants per well: {SynConfig.SYN_PER_WELL}")

    if max_wells is not None:
        pairs = pairs[:max_wells]
        print(f"Using first {len(pairs)} wells")

    all_stats = []
    n_ok = 0
    n_skip = 0

    for i, (sid, tw_path, hw_path) in enumerate(pairs):
        if i % 50 == 0:
            print(f"processing source well {i}/{len(pairs)} | ok={n_ok} skip={n_skip}")

        for k in range(int(SynConfig.SYN_PER_WELL)):
            syn_sid = f"{sid}_syn{k:02d}"
            variant_seed = int(seed + i * 1009 + k * 9176)

            tw_out = out_dir / f"{syn_sid}__typewell.csv"
            hw_out = out_dir / f"{syn_sid}__horizontal_well.csv"

            # Copy typewell unchanged, but with the synthetic sid so SDFDataset
            # can resolve it from the synthetic horizontal filename.
            pd.read_csv(tw_path).to_csv(tw_out, index=False)

            syn_hw, stats = generate_one_original_style(
                typewell_csv=tw_path,
                horizontal_csv=hw_path,
                out_csv=hw_out,
                seed=variant_seed,
            )

            if syn_hw is None:
                n_skip += 1
                # Remove copied typewell if no synthetic horizontal was produced.
                if tw_out.exists():
                    tw_out.unlink()
                if hw_out.exists():
                    hw_out.unlink()
                continue

            row = {
                "well": syn_sid,              # keeps compatibility with old QC code
                "syn_sid": syn_sid,
                "source_sid": sid,
                "variant": int(k),
                "seed": int(variant_seed),
            }
            row.update(stats)
            all_stats.append(row)
            n_ok += 1

    stats_df = pd.DataFrame(all_stats)
    stats_path = out_dir / "synthetic_generation_stats.csv"
    stats_df.to_csv(stats_path, index=False)

    print("\nDONE")
    print(f"saved synthetic dir: {out_dir}")
    print(f"stats: {stats_path}")
    print(f"ok synthetic wells: {n_ok} | skipped variants: {n_skip}")
    print(f"expected max: {len(pairs) * int(SynConfig.SYN_PER_WELL)}")

    if len(stats_df):
        print(stats_df.describe(include="all"))

    return stats_df


def make_recommended_keep_drop_from_stats(stats_df, out_dir=Config.SYNTHETIC_DIR):
    """
    Optional QC helper. This uses syn_sid-level filtering, not source-well-level
    filtering, so a bad variant can be dropped while the other variants from
    the same real well are kept.
    """
    out_dir = Path(out_dir)

    if len(stats_df) == 0:
        keep = stats_df.copy()
        drop = stats_df.copy()
    else:
        clean_to_syn_ratio = stats_df["gr_clean_std"] / (stats_df["gr_syn_std"] + 1e-6)
        bad = (
            (stats_df["out_of_bounds_count"] > 0)
            | (stats_df["gr_clean_std"] < SynConfig.MIN_FORWARD_GR_STD)
            | (stats_df["corr_gr_clean_syn"] < SynConfig.MIN_CORR_GR_CLEAN_SYN)
            | (stats_df["abs_dtvt_p95"] > SynConfig.MAX_ABS_DTVT_STEP)
            | (stats_df["abs_ddtvt_p95"] > SynConfig.MAX_ABS_DDTVT_STEP)
            | (
                (stats_df["syn_range_tvt"] < SynConfig.FAKE_SHORT_RANGE_FT)
                & (
                    (stats_df["corr_gr_clean_syn"] < SynConfig.FAKE_SHORT_MIN_CORR)
                    | (clean_to_syn_ratio < SynConfig.FAKE_SHORT_MIN_CLEAN_TO_SYN_STD_RATIO)
                )
            )
        )

        keep = stats_df.loc[~bad].copy()
        drop = stats_df.loc[bad].copy()

    keep_path = out_dir / "synthetic_generation_keep.csv"
    drop_path = out_dir / "synthetic_generation_drop.csv"
    keep.to_csv(keep_path, index=False)
    drop.to_csv(drop_path, index=False)

    print(f"QC keep: {len(keep)} -> {keep_path}")
    print(f"QC drop: {len(drop)} -> {drop_path}")

    return keep, drop


# ============================================================
# Fold-safe pretraining + fine-tuning + inference
# ============================================================
def typewell_hash(horizontal_file):
    horizontal_file = Path(horizontal_file)
    sid = horizontal_file.name.split("__")[0]
    tw = horizontal_file.parent / f"{sid}{Config.TYPEWELL_SUFFIX}"
    try:
        gr = np.round(pd.read_csv(tw)["GR"].values.astype(float), 2)
        return hashlib.md5(gr.tobytes()).hexdigest()
    except Exception:
        return horizontal_file.name


def make_loader(files, is_train=True, aug_repeats=None, start_jitter=None):
    ds = HeatmapDataset(
        files,
        is_train=is_train,
        aug_repeats=aug_repeats,
        start_jitter=start_jitter,
    )
    return DataLoader(
        ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=is_train,
        num_workers=Config.NUM_WORKERS,
        pin_memory=Config.DEVICE.startswith("cuda"),
        persistent_workers=(Config.NUM_WORKERS > 0),
        drop_last=False,
    )


def ensure_synthetic_data():
    """Generate synthetic files only when the output directory has none."""
    syn_dir = Path(Config.SYNTHETIC_DIR)
    existing = sorted(syn_dir.glob(f"*{Config.HORIZONTAL_SUFFIX}")) if syn_dir.exists() else []
    if existing:
        print(f"synthetic cache found: {len(existing)} files in {syn_dir}")
        return

    if not Config.GENERATE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError(
            f"No synthetic files found in {syn_dir}. "
            "Set GENERATE_SYNTHETIC_IF_MISSING=True or run the embedded generator first."
        )

    print("\nNo synthetic cache found; generating Alyaev-style synthetic training data...")
    stats_df = generate_synthetic_train_dir_original_style(
        train_dir=Config.TRAIN_DIR,
        out_dir=Config.SYNTHETIC_DIR,
        max_wells=None,
        seed=SynConfig.SEED,
    )
    if len(stats_df) == 0:
        raise RuntimeError("Synthetic generation produced zero usable wells.")


def select_fold_synthetic_files(train_files):
    """
    Keep only synthetic wells whose typewell hash belongs to this fold's real
    training groups. This is stricter than filename filtering and prevents a
    duplicated typewell assigned to validation from leaking into pretraining.
    """
    train_group_hashes = {typewell_hash(f) for f in train_files}
    all_syn = sorted(Path(Config.SYNTHETIC_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    selected = [f for f in all_syn if typewell_hash(f) in train_group_hashes]
    return selected, all_syn


def print_stage_header(stage, epochs, lr, aug_repeats, n_files):
    cfg = Config
    print(f"\n[{stage}] files={n_files} aug_repeats={aug_repeats} epochs={epochs} lr={lr}")
    print(
        f"START_JITTER={cfg.START_JITTER}  "
        f"RANDOM_CUTOFF_PROB={cfg.RANDOM_CUTOFF_PROB}  "
        f"RANDOM_CUTOFF_FRAC=[{cfg.RANDOM_CUTOFF_MIN_FRAC}, {cfg.RANDOM_CUTOFF_MAX_FRAC}]"
    )
    print(f"deep supervision: ON  weights: {cfg.DS_WEIGHTS}")
    print(
        f"SDF target: scale_ft={cfg.SDF_TARGET_SCALE_FT}  "
        f"clip={cfg.SDF_TARGET_CLIP}  "
        f"near_sigma_ft={cfg.SDF_NEAR_SIGMA_FT}  "
        f"near_weight={cfg.SDF_NEAR_WEIGHT}"
    )
    print(f"SDF decode: pure soft-argmax  decode_temp={cfg.SDF_DECODE_TEMP}")
    print(
        f"zero-CE: weight={cfg.ZERO_CE_WEIGHT}  temp={cfg.ZERO_CE_TEMP}  "
        f"max_dist_ft={cfg.ZERO_CE_MAX_DIST_FT}  "
        f"label_sigma_ft={cfg.ZERO_CE_LABEL_SIGMA_FT}  topk={cfg.ZERO_CE_TOPK}"
    )


def train_stage(
    model,
    train_loader,
    val_loader,
    fold,
    stage,
    epochs,
    lr,
    save_best_path=None,
):
    """
    Run one optimization stage. AdamW and cosine LR are deliberately recreated
    for each stage, so fine-tuning does not inherit synthetic optimizer moments.
    """
    cfg = Config
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(int(epochs), 1),
    )
    use_amp = cfg.DEVICE.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best = float("inf")
    best_topk = 0.0

    for epoch in range(1, int(epochs) + 1):
        model.train()
        total = 0.0
        total_reg = 0.0
        total_ce = 0.0
        n_batches = 0
        rc_count = 0
        sample_count = 0

        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                model_out = model(batch)
                loss, reg_l, ce_l = sdf_loss(model_out, batch)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss in fold={fold}, stage={stage}, epoch={epoch}."
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total += float(loss.item())
            total_reg += float(reg_l.item())
            total_ce += float(ce_l.item())
            n_batches += 1

            if "cutoff_mode" in batch:
                rc_count += int(batch["cutoff_mode"].sum().item())
                sample_count += int(batch["cutoff_mode"].numel())

        scheduler.step()
        val_rmse, val_topk = evaluate(model, val_loader)

        tag = ""
        if save_best_path is not None and val_rmse < best:
            best = float(val_rmse)
            best_topk = float(val_topk)
            torch.save(
                {
                    "model": model.state_dict(),
                    "fold": int(fold),
                    "stage": str(stage),
                    "epoch": int(epoch),
                    "val_rmse": float(val_rmse),
                    "val_topk": float(val_topk),
                    "config": {
                        "pretrain_epochs": int(cfg.PRETRAIN_EPOCHS),
                        "finetune_epochs": int(cfg.FINETUNE_EPOCHS),
                        "pretrain_lr": float(cfg.PRETRAIN_LR),
                        "finetune_lr": float(cfg.FINETUNE_LR),
                    },
                },
                save_best_path,
            )
            tag = "  saved"

        rc_frac = rc_count / max(sample_count, 1)
        nb = max(n_batches, 1)
        print(
            f"fold {fold} | {stage} ep {epoch:02d}/{int(epochs):02d} | "
            f"train {total / nb:.5f} "
            f"(reg {total_reg / nb:.5f} ce {total_ce / nb:.5f}) | "
            f"rc {rc_frac:.2f} | val {val_rmse:.4f} | "
            f"top{cfg.ZERO_CE_TOPK} {val_topk:.3f}{tag}"
        )

    # Pretraining does not select a checkpoint by validation; the current state
    # after exactly PRETRAIN_EPOCHS is handed directly to fine-tuning.
    if save_best_path is None:
        return float(val_rmse), float(val_topk)
    return best, best_topk


def load_model_checkpoint(model, ckpt_path):
    obj = torch.load(ckpt_path, map_location=Config.DEVICE)
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj
    model.load_state_dict(state)
    return obj


def train_one_fold(fold, train_idx, val_idx, files):
    cfg = Config
    train_files = [files[i] for i in train_idx]
    val_files = [files[i] for i in val_idx]

    val_loader = make_loader(val_files, is_train=False, aug_repeats=1)
    print("Last TVT RMSE:", evaluate_last_tvt(val_loader))
    print("real train files:", len(train_files), "real val files:", len(val_files))

    syn_train_files, all_syn = select_fold_synthetic_files(train_files)
    print(
        "synthetic pretrain files:", len(syn_train_files),
        "| all synthetic files:", len(all_syn),
    )
    if not syn_train_files:
        raise RuntimeError(
            f"Fold {fold} has no fold-safe synthetic files in {cfg.SYNTHETIC_DIR}."
        )

    model = HeatmapResUNet(base=16).to(cfg.DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"model params: {n_params:,}")
    print(
        "architecture: SDF ResUNet-noGN + Horizontal Dilated ASPP + "
        "ConvNeXt bottleneck + ECA + GR-derivative (C2)"
    )
    print("training protocol: fold-safe synthetic pretrain -> optimizer reset -> real fine-tune")

    # ---------------- Stage 1: exactly 2 synthetic epochs ----------------
    syn_loader = make_loader(
        syn_train_files,
        is_train=True,
        aug_repeats=cfg.PRETRAIN_AUG_REPEATS,
        start_jitter=cfg.START_JITTER,
    )
    print_stage_header(
        "PRETRAIN",
        cfg.PRETRAIN_EPOCHS,
        cfg.PRETRAIN_LR,
        cfg.PRETRAIN_AUG_REPEATS,
        len(syn_train_files),
    )
    pre_rmse, pre_topk = train_stage(
        model=model,
        train_loader=syn_loader,
        val_loader=val_loader,
        fold=fold,
        stage="pretrain",
        epochs=cfg.PRETRAIN_EPOCHS,
        lr=cfg.PRETRAIN_LR,
        save_best_path=None,
    )
    pre_ckpt = Path(f"fold_{fold}_c2_pretrain2_last.pth")
    torch.save(
        {
            "model": model.state_dict(),
            "fold": int(fold),
            "stage": "pretrain_last",
            "epoch": int(cfg.PRETRAIN_EPOCHS),
            "val_rmse": float(pre_rmse),
            "val_topk": float(pre_topk),
        },
        pre_ckpt,
    )
    print("saved", pre_ckpt)

    # Release the large synthetic loader before real fine-tuning.
    del syn_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---------------- Stage 2: real-data fine-tuning ----------------
    real_loader = make_loader(
        train_files,
        is_train=True,
        aug_repeats=cfg.FINETUNE_AUG_REPEATS,
        start_jitter=cfg.START_JITTER,
    )
    print_stage_header(
        "FINETUNE",
        cfg.FINETUNE_EPOCHS,
        cfg.FINETUNE_LR,
        cfg.FINETUNE_AUG_REPEATS,
        len(train_files),
    )
    final_ckpt = Path(f"fold_{fold}_c2_pretrain2_ft_best.pth")
    best_rmse, best_topk = train_stage(
        model=model,
        train_loader=real_loader,
        val_loader=val_loader,
        fold=fold,
        stage="finetune",
        epochs=cfg.FINETUNE_EPOCHS,
        lr=cfg.FINETUNE_LR,
        save_best_path=final_ckpt,
    )
    print(
        f"Fold {fold} best fine-tune RMSE: {best_rmse:.4f} | "
        f"top{cfg.ZERO_CE_TOPK}: {best_topk:.3f}"
    )
    return best_rmse


def run_training():
    ensure_synthetic_data()

    files = sorted(Path(Config.TRAIN_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"No real training files found in {Config.TRAIN_DIR}")

    groups = [typewell_hash(f) for f in files]
    gkf = GroupKFold(n_splits=Config.N_FOLDS)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(files, groups=groups)):
        if fold not in Config.FOLDS_TO_RUN:
            continue
        print(f"\n===== Fold {fold} =====")
        print("train:", len(train_idx), "val:", len(val_idx))
        score = train_one_fold(fold, train_idx, val_idx, files)
        scores.append(score)

    if scores:
        print("\nCV RMSE:", float(np.mean(scores)))
        print("fold scores:", [float(x) for x in scores])


@torch.no_grad()
def run_inference():
    test_files = sorted(Path(Config.TEST_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    ckpts = sorted(Path(".").glob("fold_*_c2_pretrain2_ft_best.pth"))

    if not test_files or not ckpts:
        print("inference skipped")
        print("test_files:", len(test_files), "ckpts:", len(ckpts))
        return

    models = []
    for ckpt in ckpts:
        model = HeatmapResUNet(base=16).to(Config.DEVICE)
        meta = load_model_checkpoint(model, ckpt)
        model.eval()
        models.append(model)
        if isinstance(meta, dict):
            print(
                "loaded", ckpt.name,
                "val_rmse=", meta.get("val_rmse", "n/a"),
                "epoch=", meta.get("epoch", "n/a"),
            )
        else:
            print("loaded", ckpt.name)

    ds = HeatmapDataset(test_files, is_train=False, aug_repeats=1)
    preds = {}

    for i, f in enumerate(test_files):
        sid = f.name.split("__")[0]
        sample = ds[i]
        batch = {k: v.unsqueeze(0) for k, v in sample.items()}

        logits_list = []
        for model in models:
            logits = model(batch).squeeze(1).float().cpu().numpy()[0]
            logits_list.append(logits)

        avg_logits = np.mean(logits_list, axis=0)
        pred = decode_sdf(
            avg_logits,
            sample["t_seg_tvt"].numpy(),
            sample["h_seg_tvt"].numpy(),
            sample["t_mask"].numpy(),
            int(sample["orig_len"]),
            int(sample["h_ps"]),
            float(sample["last_tvt_exact"]),
        )

        df = pd.read_csv(f)
        if "TVT_input" in df.columns:
            rows = np.flatnonzero(df["TVT_input"].isna().values)
        else:
            rows = np.arange(len(df))

        for r in rows:
            preds[f"{sid}_{int(r)}"] = float(pred[int(r)])
        print("done", sid, "submit rows:", len(rows))

    sub_path = Path(Config.TRAIN_DIR).parent / "sample_submission.csv"
    if sub_path.exists():
        sub = pd.read_csv(sub_path)[["id"]].copy()
        sub["tvt"] = sub["id"].map(preds).fillna(0.0).astype(np.float32)
    else:
        sub = pd.DataFrame({"id": list(preds.keys()), "tvt": list(preds.values())})

    sub.to_csv("submission.csv", index=False)
    print("saved submission.csv", sub.shape)
    print(sub.head())


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    run_training()
    run_inference()


In [ ]:
#private lb = 6.824
# ============================================================
# ROGII complete pipeline + GR Shape/Derivative channels (C2)
#   1) Alyaev-style synthetic generation (only if cache is missing)
#   2) fold-safe synthetic pretraining for exactly 2 epochs
#   3) optimizer-reset real-data fine-tuning
#   4) 4-fold ensemble inference
#
# C2 change (only two edits vs the baseline pipeline):
#   * e0: ResidualConvBlock(9 -> 12, base)
#   * build_image: append 3 GR first-derivative channels (dt_img, dh_img, dgr_diff)
#   The derivative is invariant to per-well GR baseline drift (32-141 API), so it
#   gives a drift-invariant matching prior aligned with the SDF zero-level set.
#   Trunk / SDF head / loss / decode / synthetic generator / all hyperparameters
#   are unchanged. Model params: 2,034,570 -> 2,035,050.
#
# AR ADD-ON (this file, vs the plain synthetic-pretrain pipeline):
#   Ported the AR re-anchor evaluation + AR-based checkpoint selection from
#   model_c2_ar.py, and switched inference to AR. Nothing else changed
#   (synthetic generator, model, loss, decode, fold split are all identical).
#   Concretely:
#     * Config: AR_STEP, AR_TRAIN_EVAL (env-gated).
#     * HeatmapDataset.__getitem__: accept an in-memory (horizontal_df, typewell_df)
#       tuple (needed to feed AR segments).
#     * ar_predict_well / evaluate_ar_files: deployable AR future RMSE.
#     * train_stage: in the SELECTION stage (finetune), evaluate AR each epoch and
#       select the best checkpoint by the AR score (the deployable metric).
#       Pretrain does not select a checkpoint, so AR is skipped there (val_AR=nan).
#     * run_inference: AR re-anchor ensemble submission.
# ============================================================
import hashlib
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Config
# ============================================================
class Config:
    # Real competition data.
    TRAIN_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"
    TEST_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/test"

    # Synthetic data produced by the Alyaev-style generator embedded below.
    # This keeps the original directory name from the supplied generator.
    SYNTHETIC_DIR = "/kaggle/input/datasets/zhuyifanss/sys-6-dataset/content/synthetic_alyaev_original_train_x4"
    GENERATE_SYNTHETIC_IF_MISSING = True

    TYPEWELL_SUFFIX = "__typewell.csv"
    HORIZONTAL_SUFFIX = "__horizontal_well.csv"

    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    SEED = 42
    NUM_WORKERS = 4
    BATCH_SIZE = 4
    WEIGHT_DECAY = 1e-4
    N_FOLDS = 4
    FOLDS_TO_RUN = [0, 1, 2, 3]

    # Stage 1: fold-safe synthetic pretraining.
    PRETRAIN_EPOCHS = 2
    PRETRAIN_LR = 1e-3
    PRETRAIN_AUG_REPEATS = 4

    # Stage 2: real-data fine-tuning. This preserves the original 10-epoch run,
    # but resets AdamW and the cosine scheduler at the start of fine-tuning.
    FINETUNE_EPOCHS = 10
    FINETUNE_LR = 5e-4
    FINETUNE_AUG_REPEATS = 4

    # Grid
    H_S = 16
    H_H = 64
    H_F = 704
    T_H = 128
    T_F = 128
    H_GR_FILTER = 51

    # SDF settings.
    # Target is a signed TVT-distance field:
    #   sdf[t,h] = clip((typewell_tvt[t] - horizontal_tvt[h]) / scale_ft)
    # The correct matching path is the zero-level set.
    SDF_TARGET_SCALE_FT = 32.0
    SDF_TARGET_CLIP = 4.0
    SDF_NEAR_SIGMA_FT = 16.0
    SDF_NEAR_WEIGHT = 2.0
    SDF_DECODE_TEMP = 0.25
    HISTORY_SIGMA = 2.0

    ORIG_PAD_LEN = 16384

    # Zero-level CE / ranking loss.
    ZERO_CE_WEIGHT = 0.3
    ZERO_CE_TEMP = 0.25
    ZERO_CE_MAX_DIST_FT = 8.0
    ZERO_CE_LABEL_SIGMA_FT = 0
    ZERO_CE_TOPK = 9

    # Core augmentation.
    START_JITTER = 512
    RANDOM_CUTOFF_PROB = 0.5
    RANDOM_CUTOFF_MIN_FRAC = 0.2
    RANDOM_CUTOFF_MAX_FRAC = 0.8
    RANDOM_CUTOFF_MIN_HISTORY = 512
    RANDOM_CUTOFF_MIN_FUTURE = 512
    DZ_SCALE = 2.0

    # Deep supervision and ASPP.
    DS_WEIGHTS = [0.4, 0.2, 0.1]
    ASPP_DILATIONS = (1, 4, 8, 16)

    # ---- AR re-anchor inference (ported from model_c2_ar.py) ----
    # Rows per AR segment. 1024 = H_H*H_S (one history span), the measured sweet
    # spot (512 -> 7.81, 1024 -> 7.736, 2048 -> 8.28 on the plain-baseline fold-2).
    # A huge value (e.g. 10**9) degenerates to the old single-pass inference.
    AR_STEP = int(os.environ.get("AR_STEP", "1024"))
    # Evaluate with AR inference INSIDE the fine-tune loop (per epoch) and select
    # the best checkpoint by the AR score (the deployable metric). Adds
    # ~4-5 min/epoch (AR = ~8 forwards/well vs 1). Set AR_TRAIN_EVAL=0 to fall
    # back to cheap single-pass selection. AR is only run in the SELECTION stage
    # (fine-tune); pretrain never selects a checkpoint, so it stays single-pass.
    AR_TRAIN_EVAL = os.environ.get("AR_TRAIN_EVAL", "1") == "1"



np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
print("DEVICE:", Config.DEVICE)


# ============================================================
# Preprocess utils
# ============================================================
def resample_typewell(t, target_step=0.5):
    t_tvt = t["TVT"].values.astype(np.float64)
    t_gr = t["GR"].values.astype(np.float64)
    diffs = np.abs(np.diff(t_tvt))
    diffs = diffs[diffs > 0]
    ratio = (np.median(diffs) if len(diffs) else target_step) / target_step
    if np.isclose(ratio, 1.0):
        return t_tvt, t_gr
    if ratio < 1.0:
        group = max(int(round(1 / ratio)), 1)
        pad = (-len(t_tvt)) % group
        if pad:
            t_tvt = np.pad(t_tvt, (0, pad), mode="edge")
            t_gr = np.pad(t_gr, (0, pad), mode="edge")
        return t_tvt.reshape(-1, group).mean(1), t_gr.reshape(-1, group).mean(1)
    up = max(int(round(ratio)), 1)
    old = np.arange(len(t_tvt))
    new = np.linspace(0, len(t_tvt) - 1, (len(t_tvt) - 1) * up + 1)
    return np.interp(new, old, t_tvt), np.interp(new, old, t_gr)


def bin_mean(arr, step, back):
    arr = np.asarray(arr)
    if len(arr) == 0:
        return arr
    pad = (-len(arr)) % step
    if pad < step // 2:
        if pad:
            arr = np.pad(arr, (0, pad) if back else (pad, 0), mode="edge")
    elif pad:
        arr = arr[:-(step - pad)] if back else arr[(step - pad):]
    if len(arr) == 0:
        return arr
    return arr.reshape(-1, step).mean(1)


def safe_savgol(x, win=51, poly=2):
    x = np.asarray(x, dtype=float)
    if len(x) <= win:
        return x
    win = min(win, len(x))
    if win % 2 == 0:
        win -= 1
    if win < 7:
        return x
    return savgol_filter(x, win, poly)


def _get_numeric_col(h, name, default_value=0.0):
    if name in h.columns:
        return (
            h[name]
            .astype(float)
            .interpolate()
            .bfill()
            .ffill()
            .fillna(default_value)
            .values
        )
    return np.full(len(h), float(default_value), dtype=np.float64)


def resample_horizontal(h, step, offset=0, forced_h_ps=None):
    gr_raw = (
        h["GR"]
        .astype(float)
        .interpolate()
        .bfill()
        .ffill()
        .fillna(85.0)
        .values
    )
    gr = safe_savgol(gr_raw, Config.H_GR_FILTER, 2)

    if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
        tvt = h["TVT"].values.astype(float)
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        tvt = (
            h["TVT_input"]
            .astype(float)
            .ffill()
            .bfill()
            .fillna(0.0)
            .values
        )
    else:
        tvt = h["Z"].values.astype(float)

    z = _get_numeric_col(h, "Z", 0.0)
    md = _get_numeric_col(h, "MD", 0.0)
    if "MD" not in h.columns:
        md = np.arange(len(h), dtype=np.float64)
    x = _get_numeric_col(h, "X", 0.0)
    y = _get_numeric_col(h, "Y", 0.0)

    if forced_h_ps is not None:
        h_ps = int(forced_h_ps)
        h_ps = max(0, min(h_ps, len(h) - 1))
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        h_ps = int(np.flatnonzero(h["TVT_input"].notna().values)[-1]) + offset
        h_ps = max(0, min(h_ps, len(h) - 1))
    else:
        h_ps = len(h) // 2

    tvt0 = bin_mean(tvt[:h_ps + 1], step, back=False)
    gr0 = bin_mean(gr[:h_ps + 1], step, back=False)
    z0 = bin_mean(z[:h_ps + 1], step, back=False)
    md0 = bin_mean(md[:h_ps + 1], step, back=False)
    x0 = bin_mean(x[:h_ps + 1], step, back=False)
    y0 = bin_mean(y[:h_ps + 1], step, back=False)

    tvt1 = bin_mean(tvt[h_ps + 1:], step, back=True)
    gr1 = bin_mean(gr[h_ps + 1:], step, back=True)
    z1 = bin_mean(z[h_ps + 1:], step, back=True)
    md1 = bin_mean(md[h_ps + 1:], step, back=True)
    x1 = bin_mean(x[h_ps + 1:], step, back=True)
    y1 = bin_mean(y[h_ps + 1:], step, back=True)

    return tvt0, gr0, tvt1, gr1, h_ps, z0, z1, md0, md1, x0, x1, y0, y1


def crop_pad_1d(n, center, history, future):
    raw_i0 = center - history
    raw_i1 = center + future
    i0 = max(raw_i0, 0)
    i1 = min(raw_i1, n)
    pad_left = max(0, -raw_i0)
    pad_right = max(0, raw_i1 - n)
    return i0, i1, pad_left, pad_right


def get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt):
    if "TVT_input" in h.columns:
        val = h["TVT_input"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if "TVT" in h.columns:
        val = h["TVT"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if len(h_tvt0):
        return float(h_tvt0[-1])
    return float(t_tvt[len(t_tvt) // 2])


def get_official_h_ps(h):
    if "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        return int(np.flatnonzero(h["TVT_input"].notna().values)[-1])
    return len(h) // 2


def choose_train_cutoff(h, cfg, start_jitter=None):
    n = len(h)
    official_ps = get_official_h_ps(h)
    sj = cfg.START_JITTER if start_jitter is None else int(start_jitter)

    can_random = (
        "TVT" in h.columns
        and h["TVT"].notna().sum() > max(32, cfg.RANDOM_CUTOFF_MIN_HISTORY // 4)
        and n > cfg.RANDOM_CUTOFF_MIN_HISTORY + cfg.RANDOM_CUTOFF_MIN_FUTURE + 8
    )

    if can_random and np.random.rand() < cfg.RANDOM_CUTOFF_PROB:
        lo = max(
            int(round(n * cfg.RANDOM_CUTOFF_MIN_FRAC)),
            int(cfg.RANDOM_CUTOFF_MIN_HISTORY),
            1,
        )
        hi = min(
            int(round(n * cfg.RANDOM_CUTOFF_MAX_FRAC)),
            n - 1 - int(cfg.RANDOM_CUTOFF_MIN_FUTURE),
        )
        if hi > lo:
            return int(np.random.randint(lo, hi + 1)), "random_cutoff"

    offset = 0
    if sj > 0:
        offset = -np.random.randint(0, sj + 1)
    h_ps = int(np.clip(official_ps + offset, 0, n - 1))
    return h_ps, "official_jitter"


def make_history_line(t_seg_tvt, h_seg_tvt, hist_mask):
    diff = np.abs(t_seg_tvt[:, None] - h_seg_tvt[None, :])
    matched = diff.argmin(axis=0)
    rows = np.arange(len(t_seg_tvt))[:, None]
    line = np.exp(
        -0.5 * ((rows - matched[None, :]) / Config.HISTORY_SIGMA) ** 2
    ).astype(np.float32)
    line *= hist_mask[None, :]
    return line



# ============================================================
# Dataset
# ============================================================
class HeatmapDataset(Dataset):
    def __init__(self, files, is_train=True, aug_repeats=None, start_jitter=None):
        self.files = list(files)
        self.is_train = is_train
        default_repeats = Config.FINETUNE_AUG_REPEATS if is_train else 1
        self.aug_repeats = int(aug_repeats if aug_repeats is not None else default_repeats)
        self.start_jitter = int(start_jitter if start_jitter is not None else Config.START_JITTER)

    def __len__(self):
        if self.is_train:
            return len(self.files) * self.aug_repeats
        return len(self.files)

    def __getitem__(self, idx):
        cfg = Config
        if self.is_train:
            idx = idx % len(self.files)
        item = self.files[idx]
        if isinstance(item, tuple):
            # in-memory (horizontal_df, typewell_df) pair — used by AR inference
            h, t = item
        else:
            hp = Path(item)
            sid = hp.name.split("__")[0]
            h = pd.read_csv(hp)
            t = pd.read_csv(hp.parent / f"{sid}{cfg.TYPEWELL_SUFFIX}")
        t_tvt, t_gr = resample_typewell(t, target_step=0.5)

        forced_h_ps = None
        offset = 0
        if self.is_train:
            forced_h_ps, cutoff_mode = choose_train_cutoff(
                h, cfg, start_jitter=self.start_jitter,
            )
        else:
            cutoff_mode = "official"

        h_tvt0, h_gr0, h_tvt1, h_gr1, h_ps, h_z0, h_z1, h_md0, h_md1, h_x0, h_x1, h_y0, h_y1 = resample_horizontal(
            h, cfg.H_S, offset=offset, forced_h_ps=forced_h_ps,
        )

        last_tvt_exact = get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt)
        last_tvt = float(h_tvt0[-1]) if len(h_tvt0) else last_tvt_exact
        last_idx = int(np.abs(t_tvt - last_tvt).argmin()) if len(t_tvt) else 0
        last_z = float(h["Z"].iloc[h_ps])
        last_md = float(h["MD"].iloc[h_ps]) if "MD" in h.columns else float(h_ps)
        last_x = float(h["X"].iloc[h_ps]) if "X" in h.columns else 0.0
        last_y = float(h["Y"].iloc[h_ps]) if "Y" in h.columns else 0.0

        # typewell crop
        i0, i1, pl, pr = crop_pad_1d(len(t_tvt), last_idx + 1, cfg.T_H, cfg.T_F)
        t_mask = np.pad(np.ones(i1 - i0, dtype=np.float32), (pl, pr), constant_values=0.0)
        t_seg_tvt = np.pad(t_tvt[i0:i1], (pl, pr), mode="edge")
        t_seg_gr = np.pad(t_gr[i0:i1], (pl, pr), mode="edge")

        # horizontal history crop
        i0_h, i1_h, p0l, p0r = crop_pad_1d(len(h_tvt0), len(h_tvt0), cfg.H_H, 0)
        hm0 = np.pad(np.ones(i1_h - i0_h, dtype=np.float32), (p0l, p0r), constant_values=0.0)
        if len(h_tvt0):
            ht0 = np.pad(h_tvt0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hg0 = np.pad(h_gr0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hz0 = np.pad(h_z0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hmd0 = np.pad(h_md0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hx0 = np.pad(h_x0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hy0 = np.pad(h_y0[i0_h:i1_h], (p0l, p0r), mode="edge")
        else:
            ht0 = np.zeros(cfg.H_H, dtype=np.float64)
            hg0 = np.zeros(cfg.H_H, dtype=np.float64)
            hz0 = np.full(cfg.H_H, last_z, dtype=np.float64)
            hmd0 = np.full(cfg.H_H, last_md, dtype=np.float64)
            hx0 = np.full(cfg.H_H, last_x, dtype=np.float64)
            hy0 = np.full(cfg.H_H, last_y, dtype=np.float64)

        # horizontal future crop
        i0_f, i1_f, p1l, p1r = crop_pad_1d(len(h_tvt1), 0, 0, cfg.H_F)
        hm1 = np.pad(np.ones(i1_f - i0_f, dtype=np.float32), (p1l, p1r), constant_values=0.0)
        if len(h_tvt1):
            ht1 = np.pad(h_tvt1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hg1 = np.pad(h_gr1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hz1 = np.pad(h_z1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hmd1 = np.pad(h_md1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hx1 = np.pad(h_x1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hy1 = np.pad(h_y1[i0_f:i1_f], (p1l, p1r), mode="edge")
        else:
            ht1 = np.zeros(cfg.H_F, dtype=np.float64)
            hg1 = np.zeros(cfg.H_F, dtype=np.float64)
            hz1 = np.full(cfg.H_F, last_z, dtype=np.float64)
            hmd1 = np.full(cfg.H_F, last_md, dtype=np.float64)
            hx1 = np.full(cfg.H_F, last_x, dtype=np.float64)
            hy1 = np.full(cfg.H_F, last_y, dtype=np.float64)

        h_mask = np.concatenate([hm0, hm1]).astype(np.float32)
        h_seg_tvt = np.concatenate([ht0, ht1]).astype(np.float64)
        h_seg_gr = np.concatenate([hg0, hg1]).astype(np.float64)
        h_seg_z = np.concatenate([hz0, hz1]).astype(np.float64)
        h_seg_md = np.concatenate([hmd0, hmd1]).astype(np.float64)
        h_seg_x = np.concatenate([hx0, hx1]).astype(np.float64)
        h_seg_y = np.concatenate([hy0, hy1]).astype(np.float64)

        hist_mask = np.zeros(cfg.H_H + cfg.H_F, dtype=np.float32)
        hist_mask[:cfg.H_H] = hm0

        h_dz = np.gradient(h_seg_z)
        h_dz = np.clip(h_dz / cfg.DZ_SCALE, -3.0, 3.0).astype(np.float32)

        H = cfg.H_H + cfg.H_F
        col_center = np.zeros(H, dtype=np.float64)
        for j in range(cfg.H_H):
            k = cfg.H_H - 1 - j
            col_center[j] = h_ps - k * cfg.H_S - (cfg.H_S - 1) / 2.0
        for k in range(cfg.H_F):
            col_center[cfg.H_H + k] = h_ps + 1 + k * cfg.H_S + (cfg.H_S - 1) / 2.0
        h_pos_rel = np.clip((col_center - float(h_ps)) / 4096.0, -4.0, 4.0).astype(np.float32)

        dmd = np.gradient(h_seg_md)
        dmd = np.where(np.abs(dmd) < 1e-6, 1.0, dmd)
        dx_dmd = np.gradient(h_seg_x) / dmd
        dy_dmd = np.gradient(h_seg_y) / dmd
        dxy = np.sqrt(dx_dmd ** 2 + dy_dmd ** 2) + 1e-6
        az_cos = np.clip(dx_dmd / dxy, -1.0, 1.0).astype(np.float32)
        az_sin = np.clip(dy_dmd / dxy, -1.0, 1.0).astype(np.float32)

        history_line = make_history_line(t_seg_tvt, h_seg_tvt, hist_mask)

        if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
            orig_tvt = h["TVT"].values.astype(np.float32)
        else:
            orig_tvt = np.zeros(len(h), dtype=np.float32)

        orig_len = len(orig_tvt)
        padded_tvt = np.zeros(cfg.ORIG_PAD_LEN, dtype=np.float32)
        ncopy = min(orig_len, cfg.ORIG_PAD_LEN)
        padded_tvt[:ncopy] = orig_tvt[:ncopy]

        return {
            "t_gr": torch.tensor(t_seg_gr, dtype=torch.float32),
            "h_gr": torch.tensor(h_seg_gr, dtype=torch.float32),
            "hist_mask": torch.tensor(hist_mask, dtype=torch.float32),
            "history_line": torch.tensor(history_line, dtype=torch.float32),
            "h_dz": torch.tensor(h_dz, dtype=torch.float32),
            "h_pos_rel": torch.tensor(h_pos_rel, dtype=torch.float32),
            "az_sin": torch.tensor(az_sin, dtype=torch.float32),
            "az_cos": torch.tensor(az_cos, dtype=torch.float32),
            "t_seg_tvt": torch.tensor(t_seg_tvt, dtype=torch.float32),
            "h_seg_tvt": torch.tensor(h_seg_tvt, dtype=torch.float32),
            "t_mask": torch.tensor(t_mask, dtype=torch.float32),
            "h_mask": torch.tensor(h_mask, dtype=torch.float32),
            "orig_tvt": torch.tensor(padded_tvt, dtype=torch.float32),
            "orig_len": torch.tensor(orig_len, dtype=torch.int64),
            "h_ps": torch.tensor(h_ps, dtype=torch.int64),
            "last_tvt_exact": torch.tensor(last_tvt_exact, dtype=torch.float32),
            "cutoff_mode": torch.tensor(1 if cutoff_mode == "random_cutoff" else 0, dtype=torch.int64),
        }


# ============================================================
# Model
# ============================================================

class ResidualConvBlock(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.conv1 = nn.Conv2d(ci, co, 3, padding=1, bias=False)
        self.norm1 = nn.Identity()
        self.conv2 = nn.Conv2d(co, co, 3, padding=1, bias=False)
        self.norm2 = nn.Identity()
        self.act = nn.GELU()
        self.skip = nn.Conv2d(ci, co, 1, bias=False) if ci != co else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + residual)



class LayerNorm2d(nn.Module):
    """Channel-wise LayerNorm for [B, C, T, H]."""
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(c))
        self.bias = nn.Parameter(torch.zeros(c))
        self.eps = eps

    def forward(self, x):
        u = x.mean(dim=1, keepdim=True)
        s = (x - u).pow(2).mean(dim=1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return x * self.weight[:, None, None] + self.bias[:, None, None]


class ConvNeXt2DBlock(nn.Module):
    """
    Small ConvNeXt-style residual bottleneck block.
    This is applied only after ASPP, so it refines low-resolution global features
    without changing the SDF output format or decode method.
    """
    def __init__(self, c, expansion=4, kernel=7):
        super().__init__()
        self.dw = nn.Conv2d(c, c, kernel, padding=kernel // 2, groups=c, bias=True)
        self.norm = LayerNorm2d(c)
        self.pw1 = nn.Conv2d(c, c * expansion, 1, bias=True)
        self.act = nn.GELU()
        self.pw2 = nn.Conv2d(c * expansion, c, 1, bias=True)
        # Zero-init residual scale keeps the initial network close to the original ASPP baseline.
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, x):
        y = self.dw(x)
        y = self.norm(y)
        y = self.pw2(self.act(self.pw1(y)))
        return x + self.gamma * y


class ECABlock2D(nn.Module):
    """
    Efficient Channel Attention.
    Adds almost no parameters; useful as channel reweighting after ConvNeXt bottleneck.
    """
    def __init__(self, c, k_size=5):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        # Zero-init residual scale keeps the initial network close to ConvNeXt-only.
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        y = self.avg(x).squeeze(-1).transpose(1, 2)  # [B, 1, C]
        y = self.conv(y).transpose(1, 2).unsqueeze(-1).sigmoid()
        return x + self.gamma * (x * y)


class HorizontalDilatedASPP(nn.Module):
    def __init__(self, ci, co, dilations=None):
        super().__init__()
        dilations = dilations or Config.ASPP_DILATIONS
        n_branches = len(dilations)
        branch_ch = co // n_branches
        self.branches = nn.ModuleList()
        for d in dilations:
            self.branches.append(nn.Sequential(
                nn.Conv2d(ci, branch_ch, 3, padding=(1, d), dilation=(1, d), bias=False),
                nn.Identity(),
                nn.GELU(),
            ))
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ci, branch_ch, 1, bias=False),
            nn.Identity(),
            nn.GELU(),
        )
        total_ch = branch_ch * (n_branches + 1)
        self.fuse = nn.Sequential(
            nn.Conv2d(total_ch, co, 1, bias=False),
            nn.Identity(),
            nn.GELU(),
        )

    def forward(self, x):
        outs = [branch(x) for branch in self.branches]
        gp = self.global_pool(x)
        gp = gp.expand(-1, -1, x.shape[2], x.shape[3])
        outs.append(gp)
        return self.fuse(torch.cat(outs, dim=1))


class HeatmapResUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.gr_norm = nn.InstanceNorm2d(2)

        # Encoder
        self.e0 = ResidualConvBlock(12, base)   # 9 baseline + 3 GR-derivative (C2)
        self.e1 = ResidualConvBlock(base, base * 2)
        self.e2 = ResidualConvBlock(base * 2, base * 4)
        self.e3 = ResidualConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        # ASPP bottleneck on pooled encoder feature.
        self.bott = HorizontalDilatedASPP(base * 8, base * 16)

        # Fixed convnext_eca bottleneck refinement.
        # Search result: ConvNeXt bottleneck + ECA is a simple, strong architecture change.
        self.cn_bott = ConvNeXt2DBlock(base * 16, kernel=7)
        self.eca_bott = ECABlock2D(base * 16, k_size=5)

        # Decoder
        self.u3 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.d3 = ResidualConvBlock(base * 16, base * 8)
        self.u2 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.d2 = ResidualConvBlock(base * 8, base * 4)
        self.u1 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.d1 = ResidualConvBlock(base * 4, base * 2)
        self.u0 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.d0 = ResidualConvBlock(base * 2, base)

        # Head
        self.head = nn.Conv2d(base, 1, 1)
        self.aux3 = nn.Conv2d(base * 8, 1, 1)
        self.aux2 = nn.Conv2d(base * 4, 1, 1)
        self.aux1 = nn.Conv2d(base * 2, 1, 1)

    def build_image(self, batch):
        dev = next(self.parameters()).device
        t_gr = batch["t_gr"].to(dev)
        h_gr = batch["h_gr"].to(dev)
        hist_mask = batch["hist_mask"].to(dev)
        history = batch["history_line"].to(dev)
        h_dz = batch["h_dz"].to(dev)
        B, T = t_gr.shape
        _, H = h_gr.shape
        t_img = t_gr.view(B, 1, T, 1).expand(B, 1, T, H)
        h_img = h_gr.view(B, 1, 1, H).expand(B, 1, T, H)
        gr_pair = self.gr_norm(torch.cat([t_img, h_img], dim=1))
        gr_diff = torch.clamp((t_img - h_img) / 40.0, -4.0, 4.0)
        mask_img = hist_mask.view(B, 1, 1, H).expand(B, 1, T, H)
        history_img = history[:, None, :, :]
        dz_img = h_dz.view(B, 1, 1, H).expand(B, 1, T, H)
        h_pos_rel = batch["h_pos_rel"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_sin = batch["az_sin"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_cos = batch["az_cos"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)

        # --- C2: GR shape/derivative channels (drift-invariant matching) ---
        # First difference along depth of each GR trace, normalized (~10 API/step) and
        # clamped. dt broadcast down rows, dh across columns, plus their outer difference.
        dt = torch.zeros_like(t_gr)
        dt[:, 1:] = t_gr[:, 1:] - t_gr[:, :-1]
        dt = torch.clamp(dt / 10.0, -4.0, 4.0)
        dh = torch.zeros_like(h_gr)
        dh[:, 1:] = h_gr[:, 1:] - h_gr[:, :-1]
        dh = torch.clamp(dh / 10.0, -4.0, 4.0)
        dt_img = dt.view(B, 1, T, 1).expand(B, 1, T, H)
        dh_img = dh.view(B, 1, 1, H).expand(B, 1, T, H)
        dgr_diff = torch.clamp(dt_img - dh_img, -4.0, 4.0)

        return torch.cat([gr_pair, gr_diff, mask_img, history_img, dz_img,
                          h_pos_rel, az_sin, az_cos, dt_img, dh_img, dgr_diff], dim=1)

    def forward(self, batch):
        x = self.build_image(batch)

        # Encoder
        e0 = self.e0(x)
        e1 = self.e1(self.pool(e0))
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))

        # Bottleneck: pool -> ASPP -> ConvNeXt -> ECA
        pooled_e3 = self.pool(e3)
        b = self.bott(pooled_e3)
        b = self.cn_bott(b)
        b = self.eca_bott(b)

        # Decoder
        d3 = self.d3(torch.cat([self.u3(b), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        d0 = self.d0(torch.cat([self.u0(d1), e0], dim=1))

        logits = self.head(d0)

        if self.training:
            full_h, full_w = x.shape[2], x.shape[3]
            a3 = self.aux3(d3)
            a2 = self.aux2(d2)
            a1 = self.aux1(d1)
            a3 = F.interpolate(a3, size=(full_h, full_w), mode="bilinear", align_corners=False)
            a2 = F.interpolate(a2, size=(full_h, full_w), mode="bilinear", align_corners=False)
            a1 = F.interpolate(a1, size=(full_h, full_w), mode="bilinear", align_corners=False)
            return logits, a1, a2, a3
        return logits


# ============================================================
# Loss
# ============================================================
def make_sdf_target(logits, batch):
    """Signed TVT-distance field target."""
    dev = logits.device
    t_seg_tvt = batch["t_seg_tvt"].to(dev).float()  # [B, T]
    h_seg_tvt = batch["h_seg_tvt"].to(dev).float()  # [B, H]
    t_mask = batch["t_mask"].to(dev).float()        # [B, T]
    h_mask = batch["h_mask"].to(dev).float()        # [B, H]

    signed_ft = t_seg_tvt[:, None, :, None] - h_seg_tvt[:, None, None, :]
    target = signed_ft / float(Config.SDF_TARGET_SCALE_FT)
    target = target.clamp(-float(Config.SDF_TARGET_CLIP), float(Config.SDF_TARGET_CLIP))
    mask = t_mask[:, None, :, None] * h_mask[:, None, None, :]
    abs_dist = signed_ft.abs()
    near_weight = 1.0 + float(Config.SDF_NEAR_WEIGHT) * torch.exp(
        -(abs_dist ** 2) / (2.0 * float(Config.SDF_NEAR_SIGMA_FT) ** 2)
    )
    return target, mask, near_weight


def sdf_loss_single(pred_sdf, batch):
    """Weighted SmoothL1 regression of the signed distance field."""
    target, mask, near_weight = make_sdf_target(pred_sdf, batch)
    pred_sdf = pred_sdf.float().clamp(
        -float(Config.SDF_TARGET_CLIP) * 1.5,
        float(Config.SDF_TARGET_CLIP) * 1.5,
    )
    loss = F.smooth_l1_loss(pred_sdf, target, reduction="none", beta=0.25)
    weight = mask * near_weight
    return (loss * weight).sum() / (weight.sum() + 1e-8)


def zero_ce_loss_single(pred_sdf, batch):
    """
    Zero-level CE / ranking loss.

    For every future horizontal column h we build row-logits over the typewell
    axis: score[t] = -|sdf[t, h]| / temp. The label is the true crossing row
    target_row[h] = argmin_t |t_tvt[t] - h_tvt[h]|. Cross-entropy on these
    logits forces the true row to have the smallest |sdf| (i.e. be ranked #1),
    directly suppressing false zero-levels and lifting gt_topk_hit.

    Only future columns whose crossing lies inside the typewell window
    (min_dist < ZERO_CE_MAX_DIST_FT) are supervised; out-of-window columns are
    left to the (saturated) regression term.
    """
    dev = pred_sdf.device
    t_seg_tvt = batch["t_seg_tvt"].to(dev).float()   # [B, T]
    h_seg_tvt = batch["h_seg_tvt"].to(dev).float()   # [B, H]
    t_mask = batch["t_mask"].to(dev).float()         # [B, T]
    h_mask = batch["h_mask"].to(dev).float()         # [B, H]

    pred = pred_sdf.float().squeeze(1)               # [B, T, H]
    B, T, H = pred.shape

    row_valid = t_mask > 0                           # [B, T]

    # row-logits per column
    score = -pred.abs() / float(Config.ZERO_CE_TEMP)          # [B, T, H]
    score = score.masked_fill(~row_valid[:, :, None], -1e4)
    logp = F.log_softmax(score, dim=1)                        # [B, T, H]

    # ground-truth crossing row per column
    dist = (t_seg_tvt[:, :, None] - h_seg_tvt[:, None, :]).abs()   # [B, T, H]
    dist = dist.masked_fill(~row_valid[:, :, None], 1e9)
    target_row = dist.argmin(dim=1)                          # [B, H]
    min_dist = dist.gather(1, target_row[:, None, :]).squeeze(1)   # [B, H]

    # supervise only future columns with an in-window crossing
    future = torch.zeros(H, device=dev, dtype=torch.bool)
    future[Config.H_H:] = True
    col_valid = (h_mask > 0) & future[None, :] & (min_dist < float(Config.ZERO_CE_MAX_DIST_FT))

    if float(Config.ZERO_CE_LABEL_SIGMA_FT) > 0.0:
        # Gaussian soft label over rows (softer, less conflict with SmoothL1)
        sigma = float(Config.ZERO_CE_LABEL_SIGMA_FT)
        soft = torch.exp(-(dist ** 2) / (2.0 * sigma * sigma))
        soft = soft.masked_fill(~row_valid[:, :, None], 0.0)
        soft = soft / (soft.sum(dim=1, keepdim=True) + 1e-8)
        nll = -(soft * logp).sum(dim=1)                      # [B, H]
    else:
        # hard CE (spec)
        nll = -logp.gather(1, target_row[:, None, :]).squeeze(1)   # [B, H]

    nll = nll * col_valid.float()
    return nll.sum() / (col_valid.float().sum() + 1e-8)


def sdf_loss(model_out, batch):
    """
    Total loss = weighted SDF SmoothL1 (+ deep supervision) + ZERO_CE_WEIGHT * zero-level CE.
    Returns (total, reg, ce) so the training loop can log the two terms separately.
    """
    if isinstance(model_out, tuple):
        pred_main, a1, a2, a3 = model_out
        w1, w2, w3 = Config.DS_WEIGHTS
        reg = (
            sdf_loss_single(pred_main, batch)
            + w1 * sdf_loss_single(a1, batch)
            + w2 * sdf_loss_single(a2, batch)
            + w3 * sdf_loss_single(a3, batch)
        )
    else:
        pred_main = model_out
        reg = sdf_loss_single(pred_main, batch)

    # CE ranking is applied on the main (full-resolution) head only.
    ce = zero_ce_loss_single(pred_main, batch)
    total = reg + float(Config.ZERO_CE_WEIGHT) * ce
    return total, reg, ce


# ============================================================
# Decode / Eval
# ============================================================
def map_to_original(pred_H, orig_len, h_ps, last_tvt_exact):
    centers = []
    values = []
    for k in range(Config.H_H):
        centers.append(h_ps - k * Config.H_S - (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H - 1 - k])
    centers.append(float(h_ps))
    values.append(float(last_tvt_exact))
    for k in range(Config.H_F):
        centers.append(h_ps + 1 + k * Config.H_S + (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H + k])
    centers = np.asarray(centers, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    order = np.argsort(centers)
    centers = centers[order]
    values = values[order]
    keep = np.r_[True, np.diff(centers) > 1e-6]
    return np.interp(np.arange(orig_len), centers[keep], values[keep])


def decode_sdf(pred_sdf, t_seg_tvt, h_seg_tvt, t_mask, orig_len, h_ps, last_tvt_exact):
    """Pure soft-argmax decode: per future column, TVT = softmax(-|sdf|/temp) . t_tvt."""
    score = -np.abs(pred_sdf.astype(np.float64)) / float(Config.SDF_DECODE_TEMP)
    if t_mask is not None:
        score = np.where(t_mask[:, None] > 0, score, -1e9)
    score = score - score.max(axis=0, keepdims=True)
    prob = np.exp(score)
    prob = prob / (prob.sum(axis=0, keepdims=True) + 1e-12)
    pred_H = (prob * t_seg_tvt[:, None]).sum(axis=0).astype(np.float64)
    pred_H[:Config.H_H] = h_seg_tvt[:Config.H_H]
    return map_to_original(pred_H, orig_len, h_ps, last_tvt_exact)


def compute_topk_hit(pred_sdf, t_seg_tvt, h_seg_tvt, t_mask, h_mask, K):
    """
    Fraction bookkeeping for gt_top-k hit: for each in-window future column,
    is the true crossing row among the K rows with smallest |sdf|?
    Returns (hit_count, valid_column_count).
    """
    T, H = pred_sdf.shape
    valid_t = np.asarray(t_mask) > 0
    abs_sdf = np.abs(pred_sdf.astype(np.float64))
    abs_sdf = np.where(valid_t[:, None], abs_sdf, np.inf)

    dist = np.abs(t_seg_tvt[:, None] - h_seg_tvt[None, :])
    dist = np.where(valid_t[:, None], dist, np.inf)
    target_row = dist.argmin(axis=0)                 # [H]
    min_dist = dist[target_row, np.arange(H)]

    future = np.zeros(H, dtype=bool)
    future[Config.H_H:] = True
    col_valid = (np.asarray(h_mask) > 0) & future & (min_dist < float(Config.ZERO_CE_MAX_DIST_FT))
    if not col_valid.any():
        return 0, 0

    k = min(K, T - 1)
    topk_idx = np.argpartition(abs_sdf, k, axis=0)[:k, :]    # [k, H]
    hit_mask = (topk_idx == target_row[None, :]).any(axis=0) # [H]
    return int((hit_mask & col_valid).sum()), int(col_valid.sum())


@torch.no_grad()
def evaluate(model, loader):
    """Returns (rmse, gt_topk_hit). Single-pass (cheap, monitoring only)."""
    model.eval()
    se = 0.0
    n = 0
    hit = 0
    hit_n = 0
    K = Config.ZERO_CE_TOPK
    for batch in loader:
        logits = model(batch)
        logits = logits.squeeze(1).cpu().numpy()
        t_seg = batch["t_seg_tvt"].numpy()
        h_seg = batch["h_seg_tvt"].numpy()
        t_mask = batch["t_mask"].numpy()
        h_mask = batch["h_mask"].numpy()
        y = batch["orig_tvt"].numpy()
        orig_len = batch["orig_len"].numpy()
        h_ps = batch["h_ps"].numpy()
        last = batch["last_tvt_exact"].numpy()
        for b in range(logits.shape[0]):
            olen = int(orig_len[b])
            ps = int(h_ps[b])
            if ps >= olen - 1:
                continue
            pred = decode_sdf(
                logits[b], t_seg[b], h_seg[b], t_mask[b],
                olen, ps, float(last[b]),
            )
            err = pred[ps + 1:] - y[b, :olen][ps + 1:]
            se += np.sum(err ** 2)
            n += len(err)

            h_hit, h_cnt = compute_topk_hit(
                logits[b], t_seg[b], h_seg[b], t_mask[b], h_mask[b], K,
            )
            hit += h_hit
            hit_n += h_cnt
    rmse = np.sqrt(se / max(n, 1))
    topk = hit / max(hit_n, 1)
    return rmse, topk


@torch.no_grad()
def evaluate_last_tvt(loader):
    se = 0.0
    n = 0
    for batch in loader:
        y = batch["orig_tvt"].numpy()
        orig_len = batch["orig_len"].numpy()
        h_ps = batch["h_ps"].numpy()
        for b in range(len(orig_len)):
            olen = int(orig_len[b])
            ps = int(h_ps[b])
            if ps >= olen - 1:
                continue
            true = y[b, :olen]
            err = true[ps] - true[ps + 1:]
            se += np.sum(err ** 2)
            n += len(err)
    return np.sqrt(se / max(n, 1))


# ============================================================
# AR re-anchor inference / evaluation (ported from model_c2_ar.py)
# ============================================================
@torch.no_grad()
def ar_predict_well(models, df, tw):
    """AR re-anchor prediction for one well (honest: known prefix + own predictions).

    Returns (work, ps0, n_steps): work = full-length TVT series, rows :ps0+1 known,
    rows ps0+1: predicted. Uses TVT_input up to PS on test wells; falls back to TVT
    for train-style files (only the prefix is read — no future truth enters).
    """
    olen = len(df)
    ps0 = get_official_h_ps(df)
    if "TVT_input" in df.columns and df["TVT_input"].notna().sum() > 0:
        known = df["TVT_input"].astype(float).interpolate().bfill().ffill().values
    elif "TVT" in df.columns and df["TVT"].notna().sum() > 0:
        known = df["TVT"].astype(float).interpolate().bfill().ffill().values
    else:
        known = np.zeros(olen)

    work = np.full(olen, np.nan)
    work[: ps0 + 1] = known[: ps0 + 1]
    cur, n_steps = ps0, 0
    while cur < olen - 1:
        w = df.copy()
        w["TVT"] = pd.Series(work).ffill().bfill().values
        ti = np.full(olen, np.nan)
        ti[: cur + 1] = w["TVT"].values[: cur + 1]
        w["TVT_input"] = ti
        sample = HeatmapDataset([(w, tw)], is_train=False)[0]
        batch = {k: v.unsqueeze(0) for k, v in sample.items()}
        avg_logits = np.mean(
            [m(batch).squeeze(1).float().cpu().numpy()[0] for m in models], axis=0)
        pred = decode_sdf(
            avg_logits,
            sample["t_seg_tvt"].numpy(), sample["h_seg_tvt"].numpy(),
            sample["t_mask"].numpy(),
            int(sample["orig_len"]), int(sample["h_ps"]),
            float(sample["last_tvt_exact"]),
        )
        end = min(cur + Config.AR_STEP, olen - 1)
        work[cur + 1: end + 1] = pred[cur + 1: end + 1]
        cur, n_steps = end, n_steps + 1
    return work, ps0, n_steps


@torch.no_grad()
def evaluate_ar_files(model, files):
    """Pooled AR-inference future RMSE over validation well files (TVT known)."""
    model.eval()
    se, n = 0.0, 0
    for f in files:
        f = Path(f)
        df = pd.read_csv(f)
        if "TVT" not in df.columns or df["TVT"].notna().sum() == 0:
            continue
        sid = f.name.split("__")[0]
        tw = pd.read_csv(f.parent / f"{sid}{Config.TYPEWELL_SUFFIX}")
        work, ps0, _ = ar_predict_well([model], df, tw)
        if ps0 >= len(df) - 1:
            continue
        err = work[ps0 + 1:] - df["TVT"].values.astype(float)[ps0 + 1:]
        se += float(np.sum(err ** 2))
        n += len(err)
    return float(np.sqrt(se / max(n, 1)))


# ============================================================
# Alyaev-original-style ROGII synthetic generator
#
# Key idea:
#   generated position = synthetic geo_delta / stratigraphic surface delta
#
#   geo_ps  = last_tvt + z_ps
#   TVT_syn = geo_ps + position - Z_future
#   GR_syn  = interp(typewell_GR, TVT_syn) + synthetic colored noise
#
# This version does NOT use real future TVT or real future TVT range.
# ============================================================


# ============================================================
# Config
# ============================================================

class SynConfig:
    SEED = 20260627

    # Generate multiple independent synthetic variants for each real well.
    # First expansion run: 4x is safer than jumping directly to 8x/16x.
    SYN_PER_WELL = 6

    # Alyaev-like random curve process
    ANGLE_MIN_DEG = 72.0
    ANGLE_MAX_DEG = 108.0

    ANGLE_RANDOM_UNIFORM = 0.010       # original: (random - 0.5) * 0.01
    MAX_ANGLE_CHANGE = 0.035           # original: _max_angle_change = 0.035

    # Fault-like jump in geo_delta domain.
    # For first pretrain run, keep it low.
    FAULT_PROB = 0
    FAULT_MAX_FT = 10.0

    # synthetic TVT-delta trend distribution, independent of true future TVT
    # mixture controls final target TVT range.
    P_FLAT = 0.25
    P_MEDIUM = 0.55
    P_LARGE = 0.18
    P_STRESS = 0.02

    FLAT_RANGE = (8.0, 22.0)
    MEDIUM_RANGE = (22.0, 55.0)
    LARGE_RANGE = (55.0, 95.0)
    STRESS_RANGE = (95.0, 130.0)

    N_CTRL_MIN = 4
    N_CTRL_MAX = 7
    CTRL_BEND_FRAC = 0.25

    # optional local stretch/squeeze of the trend
    WARP_STRENGTH = 0.10

    # GR synthetic noise, based only on typewell statistics
    GR_NOISE_STD_FRAC_LOW = 0.03
    GR_NOISE_STD_FRAC_HIGH = 0.10
    GR_AR_PHI_LOW = 0.70
    GR_AR_PHI_HIGH = 0.95

    GR_SCALE_LOW = 0.98
    GR_SCALE_HIGH = 1.02
    GR_SHIFT_LOW = -2.0
    GR_SHIFT_HIGH = 2.0

    # synthetic missing mask
    POINT_MISSING_PROB_LOW = 0.00
    POINT_MISSING_PROB_HIGH = 0.08
    SEGMENT_MISSING_PROB = 0.25
    SEGMENT_MISSING_LEN = (30, 250)
    HIGH_MISSING_PROB = 0.08
    HIGH_MISSING_FRAC = (0.25, 0.60)

    # typewell boundary
    TVT_MARGIN_FT = 2.0

    # QC
    SYN_TVT_RANGE_MIN = 6.0
    SYN_TVT_RANGE_MAX = 140.0
    MAX_ABS_DTVT_STEP = 3.0
    MAX_ABS_DDTVT_STEP = 1.5

    MIN_FORWARD_GR_STD = 1.0
    MIN_CORR_GR_CLEAN_SYN = 0.75

    # Extra fake-short guard: very small TVT range must still be explained by
    # forward GR rather than mostly synthetic noise.
    FAKE_SHORT_RANGE_FT = 8.0
    FAKE_SHORT_MIN_CORR = 0.85
    FAKE_SHORT_MIN_CLEAN_TO_SYN_STD_RATIO = 0.60

    MAX_TRIES = 50

    # if fail all attempts, skip instead of saving bad sample
    SKIP_IF_FAILED = True


# ============================================================
# Basic utils
# ============================================================

def cot(angle):
    return math.cos(angle) / math.sin(angle)


def angle_from_cot(c):
    # cot(theta)=c -> theta = atan2(1, c)
    return math.atan2(1.0, float(c))


def smooth1d(y, win=101):
    y = np.asarray(y, dtype=float)
    win = int(win)

    if win < 3:
        return y.copy()

    if win % 2 == 0:
        win += 1

    if win >= len(y):
        win = len(y) if len(y) % 2 == 1 else len(y) - 1

    if win < 3:
        return y.copy()

    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    kernel = np.ones(win, dtype=float) / win
    return np.convolve(yp, kernel, mode="valid")


def p95_abs_diff(y, order=1):
    y = np.asarray(y, dtype=float)
    if len(y) <= order:
        return np.nan

    d = y.copy()
    for _ in range(order):
        d = np.diff(d)

    return float(np.nanpercentile(np.abs(d), 95))


def get_ps_index(hw):
    """
    PS = last known TVT_input.
    This does NOT use future TVT.
    """
    if "TVT_input" in hw.columns and hw["TVT_input"].notna().any():
        return int(np.flatnonzero(hw["TVT_input"].notna().to_numpy())[-1])

    raise ValueError("No TVT_input found; cannot anchor synthetic TVT without PS.")


def find_train_pairs(train_dir):
    train_dir = Path(train_dir)

    typewell_files = []
    typewell_files += list(train_dir.glob("*__typewell.csv"))
    typewell_files += list(train_dir.glob("*_typewell.csv"))
    typewell_files += list(train_dir.rglob("*__typewell.csv"))
    typewell_files += list(train_dir.rglob("*_typewell.csv"))

    typewell_files = sorted(set(typewell_files))

    pairs = []
    for tw_path in typewell_files:
        name = tw_path.name

        if name.endswith("__typewell.csv"):
            sid = name.replace("__typewell.csv", "")
            hw_name = f"{sid}__horizontal_well.csv"
        elif name.endswith("_typewell.csv"):
            sid = name.replace("_typewell.csv", "")
            hw_name = f"{sid}_horizontal_well.csv"
        else:
            continue

        hw_path = tw_path.parent / hw_name

        if not hw_path.exists():
            cand = list(train_dir.rglob(f"{sid}__horizontal_well.csv"))
            cand += list(train_dir.rglob(f"{sid}_horizontal_well.csv"))

            if len(cand) == 0:
                continue

            hw_path = cand[0]

        pairs.append((sid, str(tw_path), str(hw_path)))

    return pairs


# ============================================================
# Synthetic curve generation
# ============================================================

def sample_target_tvt_delta_trend(x_norm, rng, cfg=SynConfig):
    """
    Generate a random TVT_delta trend without using real future TVT.

    This is only a guide. It is converted to:
        geo_delta_trend = z_delta + tvt_delta_trend

    Then the Alyaev angle process generates the actual geo_delta curve.
    """
    u = rng.random()

    if u < cfg.P_FLAT:
        target_range = rng.uniform(*cfg.FLAT_RANGE)
    elif u < cfg.P_FLAT + cfg.P_MEDIUM:
        target_range = rng.uniform(*cfg.MEDIUM_RANGE)
    elif u < cfg.P_FLAT + cfg.P_MEDIUM + cfg.P_LARGE:
        target_range = rng.uniform(*cfg.LARGE_RANGE)
    else:
        target_range = rng.uniform(*cfg.STRESS_RANGE)

    sign = rng.choice([-1.0, 1.0])

    # end drift uses a fraction of target_range; internal bends create full range.
    end_delta = sign * rng.uniform(0.35, 1.00) * target_range

    k = int(rng.integers(cfg.N_CTRL_MIN, cfg.N_CTRL_MAX + 1))
    ctrl_x = np.linspace(0.0, 1.0, k)

    ctrl_y = np.linspace(0.0, end_delta, k)

    # low-frequency bends
    bend_amp = cfg.CTRL_BEND_FRAC * target_range
    bend = rng.normal(0.0, bend_amp, size=k)
    bend[0] = 0.0
    bend[-1] = 0.0

    ctrl_y = ctrl_y + bend

    # mild monotone-ish warp in x for stretch/squeeze
    if cfg.WARP_STRENGTH > 0:
        warp_ctrl = np.linspace(0, 1, k)
        warp_noise = rng.normal(0, cfg.WARP_STRENGTH, size=k)
        warp_noise[0] = 0.0
        warp_noise[-1] = 0.0
        warped_x = ctrl_x + warp_noise
        warped_x = np.clip(warped_x, 0, 1)
        warped_x = np.sort(warped_x)
        warped_x[0] = 0.0
        warped_x[-1] = 1.0
        ctrl_x = warped_x

    tvt_delta = np.interp(x_norm, ctrl_x, ctrl_y)

    # smooth low frequency
    L = len(x_norm)
    tvt_delta = smooth1d(
        tvt_delta,
        win=max(31, min(151, L // 25 * 2 + 1)),
    )

    # anchor start to 0
    tvt_delta = tvt_delta - tvt_delta[0]

    return tvt_delta, target_range


def generate_ar_noise(n, std, phi, rng):
    """
    Colored GR noise. Does not use true horizontal residual.
    """
    eps = rng.normal(0.0, std * math.sqrt(max(1e-6, 1 - phi ** 2)), size=n)
    y = np.zeros(n, dtype=float)

    for i in range(1, n):
        y[i] = phi * y[i - 1] + eps[i]

    y = y - np.mean(y)
    if np.std(y) > 1e-6:
        y = y / np.std(y) * std

    return y


def make_synthetic_missing_mask(n, rng, cfg=SynConfig):
    """
    Synthetic missing mask, not copied from horizontal well future GR.
    """
    if rng.random() < cfg.HIGH_MISSING_PROB:
        point_prob = rng.uniform(*cfg.HIGH_MISSING_FRAC)
    else:
        point_prob = rng.uniform(cfg.POINT_MISSING_PROB_LOW, cfg.POINT_MISSING_PROB_HIGH)

    mask = rng.random(n) < point_prob

    if rng.random() < cfg.SEGMENT_MISSING_PROB and n > 200:
        n_seg = int(rng.integers(1, 4))
        for _ in range(n_seg):
            seg_len = int(rng.integers(cfg.SEGMENT_MISSING_LEN[0], cfg.SEGMENT_MISSING_LEN[1] + 1))
            seg_len = min(seg_len, n)
            start = int(rng.integers(0, max(1, n - seg_len + 1)))
            mask[start:start + seg_len] = True

    return mask


# ============================================================
# One synthetic sample
# ============================================================

def _generate_once_original_style(typewell_csv, horizontal_csv, seed, cfg=SynConfig):
    rng = np.random.default_rng(seed)

    tw = pd.read_csv(typewell_csv).sort_values("TVT").reset_index(drop=True)
    hw = pd.read_csv(horizontal_csv).reset_index(drop=True)

    t_tvt = tw["TVT"].astype(float).to_numpy()
    t_gr = tw["GR"].astype(float).to_numpy()

    h_md = hw["MD"].astype(float).to_numpy()
    h_z = hw["Z"].astype(float).to_numpy()

    h_ps = get_ps_index(hw)

    idx_f = np.arange(h_ps + 1, len(hw))
    if len(idx_f) < 32:
        return None, None

    # Anchor only from TVT_input prefix
    tvt_input = hw["TVT_input"].astype(float).to_numpy()
    last_tvt = float(tvt_input[h_ps])

    md_ps = float(h_md[h_ps])
    z_ps = float(h_z[h_ps])
    geo_ps = last_tvt + z_ps

    md_f = h_md[idx_f]
    z_f = h_z[idx_f]

    L = len(idx_f)

    x_dist = md_f - md_ps
    x_norm = (x_dist - x_dist.min()) / (x_dist.max() - x_dist.min() + 1e-12)

    dmd = np.diff(np.r_[md_ps, md_f])
    dmd = np.maximum(dmd, 1e-6)

    z_delta = z_f - z_ps

    # --------------------------------------------------------
    # 1. Generate random TVT_delta trend independent of truth
    # --------------------------------------------------------

    tvt_delta_trend, target_range = sample_target_tvt_delta_trend(
        x_norm=x_norm,
        rng=rng,
        cfg=cfg,
    )

    # Key mapping:
    # position is geo_delta, therefore guide is z_delta + tvt_delta.
    geo_delta_trend = z_delta + tvt_delta_trend
    geo_delta_trend = smooth1d(
        geo_delta_trend,
        win=max(31, min(151, L // 25 * 2 + 1)),
    )

    expected_cot = np.gradient(geo_delta_trend, md_f)
    expected_cot = smooth1d(
        expected_cot,
        win=max(31, min(101, L // 35 * 2 + 1)),
    )

    # --------------------------------------------------------
    # 2. Alyaev-style SubsurfaceState process
    # --------------------------------------------------------

    angle_min = math.radians(cfg.ANGLE_MIN_DEG)
    angle_max = math.radians(cfg.ANGLE_MAX_DEG)

    positions = np.zeros(L, dtype=float)
    angles = np.zeros(L, dtype=float)

    angle = angle_from_cot(expected_cot[0])
    angle = float(np.clip(angle, angle_min, angle_max))

    pos = 0.0
    positions[0] = pos
    angles[0] = angle

    fault_count = 0

    for i in range(1, L):
        new_thl_distance = float(dmd[i])

        # original-like random angle perturbation
        new_angle = angle + (rng.random() - 0.5) * cfg.ANGLE_RANDOM_UNIFORM

        expected_cot_i = float(expected_cot[i])
        new_cot = cot(new_angle)

        delta_angle = 0.0

        # same logic as Alyaev:
        # if current cot is far from expected cot, push angle toward expected trend
        if (0.5 + rng.random()) * 0.03 < abs(expected_cot_i - new_cot) and rng.random() > 0.2:
            sign = -np.sign(expected_cot_i - new_cot)
            magnitude = abs(expected_cot_i - new_cot) * (1.0 + (rng.random() - 0.5) * 0.8)
            delta_angle = sign * min(cfg.MAX_ANGLE_CHANGE, magnitude)

        new_angle = new_angle + delta_angle
        new_angle = float(np.clip(new_angle, angle_min, angle_max))

        # integrate position
        new_position = pos + new_thl_distance * cot(new_angle)

        # original-like fault/position correction toward expected position
        expected_position = float(geo_delta_trend[i] - geo_delta_trend[0])

        delta = 0.0
        if rng.random() < cfg.FAULT_PROB:
            mismatch = abs(expected_position - new_position)
            if mismatch > 1e-6:
                sign = np.sign(expected_position - new_position)
                magnitude = mismatch * (1.0 + (rng.random() - 0.5))
                delta = sign * min(cfg.FAULT_MAX_FT, magnitude)
                fault_count += 1

        new_position = new_position + delta

        angle = new_angle
        pos = new_position

        positions[i] = pos
        angles[i] = angle

    geo_delta_syn = geo_delta_trend[0] + positions

    # --------------------------------------------------------
    # 3. Convert geo_delta -> TVT
    # --------------------------------------------------------

    tvt_syn_f = geo_ps + geo_delta_syn - z_f

    # exact PS continuity taper
    delta0 = tvt_syn_f[0] - last_tvt
    taper = np.exp(-np.linspace(0, 7, L))
    tvt_syn_f = tvt_syn_f - delta0 * taper

    geo_delta_syn = tvt_syn_f + z_f - geo_ps

    # boundary QC, no clipping in accepted sample
    clip_min = float(t_tvt.min() + cfg.TVT_MARGIN_FT)
    clip_max = float(t_tvt.max() - cfg.TVT_MARGIN_FT)

    out_of_bounds = int(np.sum((tvt_syn_f < clip_min) | (tvt_syn_f > clip_max)))

    # --------------------------------------------------------
    # 4. Forward GR from typewell + synthetic colored noise
    # --------------------------------------------------------

    gr_clean = np.interp(tvt_syn_f, t_tvt, t_gr)

    typewell_gr_std = float(np.nanstd(t_gr))
    noise_std = rng.uniform(cfg.GR_NOISE_STD_FRAC_LOW, cfg.GR_NOISE_STD_FRAC_HIGH) * max(typewell_gr_std, 1.0)
    phi = rng.uniform(cfg.GR_AR_PHI_LOW, cfg.GR_AR_PHI_HIGH)

    noise = generate_ar_noise(L, noise_std, phi, rng)

    gr_syn = gr_clean + noise

    gr_syn = (
        gr_syn * rng.uniform(cfg.GR_SCALE_LOW, cfg.GR_SCALE_HIGH)
        + rng.uniform(cfg.GR_SHIFT_LOW, cfg.GR_SHIFT_HIGH)
    )

    missing_mask = make_synthetic_missing_mask(L, rng, cfg)
    gr_syn_raw = gr_syn.copy()
    gr_syn_raw[missing_mask] = np.nan

    # --------------------------------------------------------
    # 5. Build synthetic horizontal well
    # --------------------------------------------------------

    syn_hw = hw.copy()

    # Future target becomes synthetic.
    # Prefix remains original known history.
    syn_hw.loc[idx_f, "TVT"] = tvt_syn_f
    syn_hw.loc[idx_f, "GR"] = gr_syn_raw

    # Keep TVT_input unchanged.
    # This keeps the anchor/prefix exactly like the real task.

    # --------------------------------------------------------
    # 6. Stats / QC
    # --------------------------------------------------------

    syn_tvt_range = float(np.nanmax(tvt_syn_f) - np.nanmin(tvt_syn_f))
    gr_clean_std = float(np.nanstd(gr_clean))
    gr_syn_std = float(np.nanstd(gr_syn))
    noise_std_actual = float(np.nanstd(noise))

    corr = np.nan
    if L > 2 and gr_clean_std > 1e-6 and gr_syn_std > 1e-6:
        corr = float(np.corrcoef(gr_clean, gr_syn)[0, 1])

    stats = {
        "ps_index": int(h_ps),
        "ps_md": float(md_ps),
        "future_rows": int(L),

        "target_range": float(target_range),
        "syn_range_tvt": syn_tvt_range,
        "syn_end_delta_tvt": float(tvt_syn_f[-1] - last_tvt),
        "syn_geo_delta_range": float(np.nanmax(geo_delta_syn) - np.nanmin(geo_delta_syn)),

        "abs_dtvt_p95": p95_abs_diff(tvt_syn_f, 1),
        "abs_ddtvt_p95": p95_abs_diff(tvt_syn_f, 2),

        "gr_clean_std": gr_clean_std,
        "gr_syn_std": gr_syn_std,
        "noise_std": noise_std_actual,
        "corr_gr_clean_syn": corr,

        "missing_frac": float(np.mean(missing_mask)),
        "out_of_bounds_count": out_of_bounds,
        "fault_count": int(fault_count),
    }

    return syn_hw, stats


def generate_one_original_style(
    typewell_csv,
    horizontal_csv,
    out_csv=None,
    seed=SynConfig.SEED,
    cfg=SynConfig,
):
    """
    Regenerate until QC passes.
    Does NOT use true future TVT.
    """
    for attempt in range(cfg.MAX_TRIES):
        syn_hw, stats = _generate_once_original_style(
            typewell_csv=typewell_csv,
            horizontal_csv=horizontal_csv,
            seed=seed + attempt * 10007,
            cfg=cfg,
        )

        if syn_hw is None:
            return None, None

        ok = True

        ok &= stats["out_of_bounds_count"] == 0
        ok &= cfg.SYN_TVT_RANGE_MIN <= stats["syn_range_tvt"] <= cfg.SYN_TVT_RANGE_MAX
        ok &= stats["abs_dtvt_p95"] <= cfg.MAX_ABS_DTVT_STEP
        ok &= stats["abs_ddtvt_p95"] <= cfg.MAX_ABS_DDTVT_STEP
        ok &= stats["gr_clean_std"] >= cfg.MIN_FORWARD_GR_STD
        ok &= stats["corr_gr_clean_syn"] >= cfg.MIN_CORR_GR_CLEAN_SYN

        # Guard against fake-short samples: if TVT barely moves, GR should not
        # be dominated by noise. This keeps real flat wells but rejects noisy
        # synthetic flat wells.
        clean_to_syn_ratio = stats["gr_clean_std"] / (stats["gr_syn_std"] + 1e-6)
        if stats["syn_range_tvt"] < cfg.FAKE_SHORT_RANGE_FT:
            ok &= stats["corr_gr_clean_syn"] >= cfg.FAKE_SHORT_MIN_CORR
            ok &= clean_to_syn_ratio >= cfg.FAKE_SHORT_MIN_CLEAN_TO_SYN_STD_RATIO

        if ok:
            stats["accepted_attempt"] = attempt
            if out_csv is not None:
                syn_hw.to_csv(out_csv, index=False)
            return syn_hw, stats

    if cfg.SKIP_IF_FAILED:
        return None, None

    stats["accepted_attempt"] = -1
    if out_csv is not None:
        syn_hw.to_csv(out_csv, index=False)

    return syn_hw, stats


# ============================================================
# Directory generation
# ============================================================

def generate_synthetic_train_dir_original_style(
    train_dir=Config.TRAIN_DIR,
    out_dir=Config.SYNTHETIC_DIR,
    max_wells=None,
    seed=SynConfig.SEED,
):
    """
    Generate SYN_PER_WELL independent synthetic variants per real well.

    Output naming:
        source sid: 000d7d20
        variant sid: 000d7d20_syn00, 000d7d20_syn01, ...

    This naming is important because the training Dataset looks for:
        {sid}__horizontal_well.csv
        {sid}__typewell.csv
    so each synthetic variant needs its own matching typewell filename.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pairs = find_train_pairs(train_dir)
    print(f"Found train pairs: {len(pairs)}")
    print(f"Synthetic variants per well: {SynConfig.SYN_PER_WELL}")

    if max_wells is not None:
        pairs = pairs[:max_wells]
        print(f"Using first {len(pairs)} wells")

    all_stats = []
    n_ok = 0
    n_skip = 0

    for i, (sid, tw_path, hw_path) in enumerate(pairs):
        if i % 50 == 0:
            print(f"processing source well {i}/{len(pairs)} | ok={n_ok} skip={n_skip}")

        for k in range(int(SynConfig.SYN_PER_WELL)):
            syn_sid = f"{sid}_syn{k:02d}"
            variant_seed = int(seed + i * 1009 + k * 9176)

            tw_out = out_dir / f"{syn_sid}__typewell.csv"
            hw_out = out_dir / f"{syn_sid}__horizontal_well.csv"

            # Copy typewell unchanged, but with the synthetic sid so SDFDataset
            # can resolve it from the synthetic horizontal filename.
            pd.read_csv(tw_path).to_csv(tw_out, index=False)

            syn_hw, stats = generate_one_original_style(
                typewell_csv=tw_path,
                horizontal_csv=hw_path,
                out_csv=hw_out,
                seed=variant_seed,
            )

            if syn_hw is None:
                n_skip += 1
                # Remove copied typewell if no synthetic horizontal was produced.
                if tw_out.exists():
                    tw_out.unlink()
                if hw_out.exists():
                    hw_out.unlink()
                continue

            row = {
                "well": syn_sid,              # keeps compatibility with old QC code
                "syn_sid": syn_sid,
                "source_sid": sid,
                "variant": int(k),
                "seed": int(variant_seed),
            }
            row.update(stats)
            all_stats.append(row)
            n_ok += 1

    stats_df = pd.DataFrame(all_stats)
    stats_path = out_dir / "synthetic_generation_stats.csv"
    stats_df.to_csv(stats_path, index=False)

    print("\nDONE")
    print(f"saved synthetic dir: {out_dir}")
    print(f"stats: {stats_path}")
    print(f"ok synthetic wells: {n_ok} | skipped variants: {n_skip}")
    print(f"expected max: {len(pairs) * int(SynConfig.SYN_PER_WELL)}")

    if len(stats_df):
        print(stats_df.describe(include="all"))

    return stats_df


def make_recommended_keep_drop_from_stats(stats_df, out_dir=Config.SYNTHETIC_DIR):
    """
    Optional QC helper. This uses syn_sid-level filtering, not source-well-level
    filtering, so a bad variant can be dropped while the other variants from
    the same real well are kept.
    """
    out_dir = Path(out_dir)

    if len(stats_df) == 0:
        keep = stats_df.copy()
        drop = stats_df.copy()
    else:
        clean_to_syn_ratio = stats_df["gr_clean_std"] / (stats_df["gr_syn_std"] + 1e-6)
        bad = (
            (stats_df["out_of_bounds_count"] > 0)
            | (stats_df["gr_clean_std"] < SynConfig.MIN_FORWARD_GR_STD)
            | (stats_df["corr_gr_clean_syn"] < SynConfig.MIN_CORR_GR_CLEAN_SYN)
            | (stats_df["abs_dtvt_p95"] > SynConfig.MAX_ABS_DTVT_STEP)
            | (stats_df["abs_ddtvt_p95"] > SynConfig.MAX_ABS_DDTVT_STEP)
            | (
                (stats_df["syn_range_tvt"] < SynConfig.FAKE_SHORT_RANGE_FT)
                & (
                    (stats_df["corr_gr_clean_syn"] < SynConfig.FAKE_SHORT_MIN_CORR)
                    | (clean_to_syn_ratio < SynConfig.FAKE_SHORT_MIN_CLEAN_TO_SYN_STD_RATIO)
                )
            )
        )

        keep = stats_df.loc[~bad].copy()
        drop = stats_df.loc[bad].copy()

    keep_path = out_dir / "synthetic_generation_keep.csv"
    drop_path = out_dir / "synthetic_generation_drop.csv"
    keep.to_csv(keep_path, index=False)
    drop.to_csv(drop_path, index=False)

    print(f"QC keep: {len(keep)} -> {keep_path}")
    print(f"QC drop: {len(drop)} -> {drop_path}")

    return keep, drop


# ============================================================
# Fold-safe pretraining + fine-tuning + inference
# ============================================================
def typewell_hash(horizontal_file):
    horizontal_file = Path(horizontal_file)
    sid = horizontal_file.name.split("__")[0]
    tw = horizontal_file.parent / f"{sid}{Config.TYPEWELL_SUFFIX}"
    try:
        gr = np.round(pd.read_csv(tw)["GR"].values.astype(float), 2)
        return hashlib.md5(gr.tobytes()).hexdigest()
    except Exception:
        return horizontal_file.name


def make_loader(files, is_train=True, aug_repeats=None, start_jitter=None):
    ds = HeatmapDataset(
        files,
        is_train=is_train,
        aug_repeats=aug_repeats,
        start_jitter=start_jitter,
    )
    return DataLoader(
        ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=is_train,
        num_workers=Config.NUM_WORKERS,
        pin_memory=Config.DEVICE.startswith("cuda"),
        persistent_workers=(Config.NUM_WORKERS > 0),
        drop_last=False,
    )


def ensure_synthetic_data():
    """Generate synthetic files only when the output directory has none."""
    syn_dir = Path(Config.SYNTHETIC_DIR)
    existing = sorted(syn_dir.glob(f"*{Config.HORIZONTAL_SUFFIX}")) if syn_dir.exists() else []
    if existing:
        print(f"synthetic cache found: {len(existing)} files in {syn_dir}")
        return

    if not Config.GENERATE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError(
            f"No synthetic files found in {syn_dir}. "
            "Set GENERATE_SYNTHETIC_IF_MISSING=True or run the embedded generator first."
        )

    print("\nNo synthetic cache found; generating Alyaev-style synthetic training data...")
    stats_df = generate_synthetic_train_dir_original_style(
        train_dir=Config.TRAIN_DIR,
        out_dir=Config.SYNTHETIC_DIR,
        max_wells=None,
        seed=SynConfig.SEED,
    )
    if len(stats_df) == 0:
        raise RuntimeError("Synthetic generation produced zero usable wells.")


def select_fold_synthetic_files(train_files):
    """
    Keep only synthetic wells whose typewell hash belongs to this fold's real
    training groups. This is stricter than filename filtering and prevents a
    duplicated typewell assigned to validation from leaking into pretraining.
    """
    train_group_hashes = {typewell_hash(f) for f in train_files}
    all_syn = sorted(Path(Config.SYNTHETIC_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    selected = [f for f in all_syn if typewell_hash(f) in train_group_hashes]
    return selected, all_syn


def print_stage_header(stage, epochs, lr, aug_repeats, n_files):
    cfg = Config
    print(f"\n[{stage}] files={n_files} aug_repeats={aug_repeats} epochs={epochs} lr={lr}")
    print(
        f"START_JITTER={cfg.START_JITTER}  "
        f"RANDOM_CUTOFF_PROB={cfg.RANDOM_CUTOFF_PROB}  "
        f"RANDOM_CUTOFF_FRAC=[{cfg.RANDOM_CUTOFF_MIN_FRAC}, {cfg.RANDOM_CUTOFF_MAX_FRAC}]"
    )
    print(f"deep supervision: ON  weights: {cfg.DS_WEIGHTS}")
    print(
        f"SDF target: scale_ft={cfg.SDF_TARGET_SCALE_FT}  "
        f"clip={cfg.SDF_TARGET_CLIP}  "
        f"near_sigma_ft={cfg.SDF_NEAR_SIGMA_FT}  "
        f"near_weight={cfg.SDF_NEAR_WEIGHT}"
    )
    print(f"SDF decode: pure soft-argmax  decode_temp={cfg.SDF_DECODE_TEMP}")
    print(
        f"zero-CE: weight={cfg.ZERO_CE_WEIGHT}  temp={cfg.ZERO_CE_TEMP}  "
        f"max_dist_ft={cfg.ZERO_CE_MAX_DIST_FT}  "
        f"label_sigma_ft={cfg.ZERO_CE_LABEL_SIGMA_FT}  topk={cfg.ZERO_CE_TOPK}"
    )


def train_stage(
    model,
    train_loader,
    val_loader,
    fold,
    stage,
    epochs,
    lr,
    save_best_path=None,
    val_files=None,
):
    """
    Run one optimization stage. AdamW and cosine LR are deliberately recreated
    for each stage, so fine-tuning does not inherit synthetic optimizer moments.

    Checkpoint selection (only when save_best_path is not None):
      * If AR_TRAIN_EVAL and val_files are provided, evaluate the deployable AR
        re-anchor RMSE each epoch and select the best checkpoint by that AR score.
      * Otherwise fall back to single-pass val RMSE selection.
    Pretraining (save_best_path is None) does not select a checkpoint, so AR is
    skipped there to avoid the extra ~8 forwards/well cost.
    """
    cfg = Config
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(int(epochs), 1),
    )
    use_amp = cfg.DEVICE.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    is_selection = save_best_path is not None
    use_ar = bool(cfg.AR_TRAIN_EVAL and is_selection and val_files is not None)

    best = float("inf")
    best_topk = 0.0
    last_val_rmse = float("nan")
    last_val_topk = 0.0

    for epoch in range(1, int(epochs) + 1):
        model.train()
        total = 0.0
        total_reg = 0.0
        total_ce = 0.0
        n_batches = 0
        rc_count = 0
        sample_count = 0

        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                model_out = model(batch)
                loss, reg_l, ce_l = sdf_loss(model_out, batch)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss in fold={fold}, stage={stage}, epoch={epoch}."
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total += float(loss.item())
            total_reg += float(reg_l.item())
            total_ce += float(ce_l.item())
            n_batches += 1

            if "cutoff_mode" in batch:
                rc_count += int(batch["cutoff_mode"].sum().item())
                sample_count += int(batch["cutoff_mode"].numel())

        scheduler.step()

        # Single-pass val (cheap, monitoring + top-k).
        val_rmse, val_topk = evaluate(model, val_loader)
        last_val_rmse, last_val_topk = val_rmse, val_topk

        # Deployable AR metric drives checkpoint selection in the selection stage.
        if use_ar:
            ar_rmse = evaluate_ar_files(model, val_files)
            score = ar_rmse
        else:
            ar_rmse = float("nan")
            score = val_rmse

        tag = ""
        if is_selection and score < best:
            best = float(score)
            best_topk = float(val_topk)
            torch.save(
                {
                    "model": model.state_dict(),
                    "fold": int(fold),
                    "stage": str(stage),
                    "epoch": int(epoch),
                    "val_rmse": float(val_rmse),      # single-pass
                    "val_ar": float(ar_rmse),         # deployable AR (selection metric when use_ar)
                    "val_topk": float(val_topk),
                    "selected_by": "ar" if use_ar else "single_pass",
                    "config": {
                        "pretrain_epochs": int(cfg.PRETRAIN_EPOCHS),
                        "finetune_epochs": int(cfg.FINETUNE_EPOCHS),
                        "pretrain_lr": float(cfg.PRETRAIN_LR),
                        "finetune_lr": float(cfg.FINETUNE_LR),
                        "ar_step": int(cfg.AR_STEP),
                    },
                },
                save_best_path,
            )
            tag = "  saved"

        rc_frac = rc_count / max(sample_count, 1)
        nb = max(n_batches, 1)
        print(
            f"fold {fold} | {stage} ep {epoch:02d}/{int(epochs):02d} | "
            f"train {total / nb:.5f} "
            f"(reg {total_reg / nb:.5f} ce {total_ce / nb:.5f}) | "
            f"rc {rc_frac:.2f} | val_sp {val_rmse:.4f} | val_AR {ar_rmse:.4f} | "
            f"top{cfg.ZERO_CE_TOPK} {val_topk:.3f}{tag}"
        )

    # Pretraining does not select a checkpoint; the current state after exactly
    # PRETRAIN_EPOCHS is handed directly to fine-tuning.
    if not is_selection:
        return float(last_val_rmse), float(last_val_topk)
    return best, best_topk


def load_model_checkpoint(model, ckpt_path):
    obj = torch.load(ckpt_path, map_location=Config.DEVICE)
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj
    model.load_state_dict(state)
    return obj


def train_one_fold(fold, train_idx, val_idx, files):
    cfg = Config
    train_files = [files[i] for i in train_idx]
    val_files = [files[i] for i in val_idx]

    val_loader = make_loader(val_files, is_train=False, aug_repeats=1)
    print("Last TVT RMSE:", evaluate_last_tvt(val_loader))
    print("real train files:", len(train_files), "real val files:", len(val_files))

    syn_train_files, all_syn = select_fold_synthetic_files(train_files)
    print(
        "synthetic pretrain files:", len(syn_train_files),
        "| all synthetic files:", len(all_syn),
    )
    if not syn_train_files:
        raise RuntimeError(
            f"Fold {fold} has no fold-safe synthetic files in {cfg.SYNTHETIC_DIR}."
        )

    model = HeatmapResUNet(base=16).to(cfg.DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"model params: {n_params:,}")
    print(
        "architecture: SDF ResUNet-noGN + Horizontal Dilated ASPP + "
        "ConvNeXt bottleneck + ECA + GR-derivative (C2)"
    )
    print("training protocol: fold-safe synthetic pretrain -> optimizer reset -> real fine-tune")
    print(
        f"checkpoint selection: {'AR re-anchor (AR_STEP=%d)' % cfg.AR_STEP if cfg.AR_TRAIN_EVAL else 'single-pass'}"
    )

    # ---------------- Stage 1: exactly 2 synthetic epochs ----------------
    syn_loader = make_loader(
        syn_train_files,
        is_train=True,
        aug_repeats=cfg.PRETRAIN_AUG_REPEATS,
        start_jitter=cfg.START_JITTER,
    )
    print_stage_header(
        "PRETRAIN",
        cfg.PRETRAIN_EPOCHS,
        cfg.PRETRAIN_LR,
        cfg.PRETRAIN_AUG_REPEATS,
        len(syn_train_files),
    )
    pre_rmse, pre_topk = train_stage(
        model=model,
        train_loader=syn_loader,
        val_loader=val_loader,
        fold=fold,
        stage="pretrain",
        epochs=cfg.PRETRAIN_EPOCHS,
        lr=cfg.PRETRAIN_LR,
        save_best_path=None,       # pretrain does not select -> AR skipped here
        val_files=val_files,
    )
    pre_ckpt = Path(f"fold_{fold}_c2_pretrain2_last.pth")
    torch.save(
        {
            "model": model.state_dict(),
            "fold": int(fold),
            "stage": "pretrain_last",
            "epoch": int(cfg.PRETRAIN_EPOCHS),
            "val_rmse": float(pre_rmse),
            "val_topk": float(pre_topk),
        },
        pre_ckpt,
    )
    print("saved", pre_ckpt)

    # Release the large synthetic loader before real fine-tuning.
    del syn_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---------------- Stage 2: real-data fine-tuning ----------------
    real_loader = make_loader(
        train_files,
        is_train=True,
        aug_repeats=cfg.FINETUNE_AUG_REPEATS,
        start_jitter=cfg.START_JITTER,
    )
    print_stage_header(
        "FINETUNE",
        cfg.FINETUNE_EPOCHS,
        cfg.FINETUNE_LR,
        cfg.FINETUNE_AUG_REPEATS,
        len(train_files),
    )
    final_ckpt = Path(f"fold_{fold}_c2_pretrain2_ft_best.pth")
    best_rmse, best_topk = train_stage(
        model=model,
        train_loader=real_loader,
        val_loader=val_loader,
        fold=fold,
        stage="finetune",
        epochs=cfg.FINETUNE_EPOCHS,
        lr=cfg.FINETUNE_LR,
        save_best_path=final_ckpt,
        val_files=val_files,        # enables AR selection in the fine-tune stage
    )
    metric_name = "AR RMSE" if cfg.AR_TRAIN_EVAL else "single-pass RMSE"
    print(
        f"Fold {fold} best fine-tune {metric_name}: {best_rmse:.4f} | "
        f"top{cfg.ZERO_CE_TOPK}: {best_topk:.3f}"
    )
    return best_rmse


def run_training():
    ensure_synthetic_data()

    files = sorted(Path(Config.TRAIN_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"No real training files found in {Config.TRAIN_DIR}")

    groups = [typewell_hash(f) for f in files]
    gkf = GroupKFold(n_splits=Config.N_FOLDS)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(files, groups=groups)):
        if fold not in Config.FOLDS_TO_RUN:
            continue
        print(f"\n===== Fold {fold} =====")
        print("train:", len(train_idx), "val:", len(val_idx))
        score = train_one_fold(fold, train_idx, val_idx, files)
        scores.append(score)

    if scores:
        metric_name = "AR" if Config.AR_TRAIN_EVAL else "single-pass"
        print(f"\nCV {metric_name} RMSE:", float(np.mean(scores)))
        print("fold scores:", [float(x) for x in scores])


@torch.no_grad()
def run_ar_validation():
    """Post-hoc AR-inference RMSE on each held-out validation fold for the saved
    fine-tune checkpoints. Redundant when AR_TRAIN_EVAL=1 (val_AR is already
    printed per epoch), so it is off by default and only runs when AR_VAL=1."""
    files = sorted(Path(Config.TRAIN_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    groups = [typewell_hash(f) for f in files]
    gkf = GroupKFold(n_splits=Config.N_FOLDS)
    for fold, (train_idx, val_idx) in enumerate(gkf.split(files, groups=groups)):
        if fold not in Config.FOLDS_TO_RUN:
            continue
        ckpt = Path(f"fold_{fold}_c2_pretrain2_ft_best.pth")
        if not ckpt.exists():
            print(f"AR validation skipped (no {ckpt.name})")
            continue
        model = HeatmapResUNet(base=16).to(Config.DEVICE)
        load_model_checkpoint(model, ckpt)
        model.eval()
        val_files = [files[i] for i in val_idx]
        ar = evaluate_ar_files(model, val_files)
        print(f"\nfold {fold} AR-inference val RMSE: {ar:.4f}")


@torch.no_grad()
def run_inference():
    test_files = sorted(Path(Config.TEST_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    ckpts = sorted(Path(".").glob("fold_*_c2_pretrain2_ft_best.pth"))

    if not test_files or not ckpts:
        print("inference skipped")
        print("test_files:", len(test_files), "ckpts:", len(ckpts))
        return

    models = []
    for ckpt in ckpts:
        model = HeatmapResUNet(base=16).to(Config.DEVICE)
        meta = load_model_checkpoint(model, ckpt)
        model.eval()
        models.append(model)
        if isinstance(meta, dict):
            print(
                "loaded", ckpt.name,
                "val_rmse=", meta.get("val_rmse", "n/a"),
                "val_ar=", meta.get("val_ar", "n/a"),
                "epoch=", meta.get("epoch", "n/a"),
            )
        else:
            print("loaded", ckpt.name)

    preds = {}
    for f in test_files:
        sid = f.name.split("__")[0]
        df = pd.read_csv(f)
        tw = pd.read_csv(f.parent / f"{sid}{Config.TYPEWELL_SUFFIX}")
        work, ps0, n_steps = ar_predict_well(models, df, tw)

        if "TVT_input" in df.columns:
            rows = np.flatnonzero(df["TVT_input"].isna().values)
        else:
            rows = np.arange(len(df))

        for r in rows:
            preds[f"{sid}_{int(r)}"] = float(work[int(r)])
        print(f"done {sid}  AR steps: {n_steps}  submit rows: {len(rows)}")

    sub_path = Path(Config.TRAIN_DIR).parent / "sample_submission.csv"
    if sub_path.exists():
        sub = pd.read_csv(sub_path)[["id"]].copy()
        sub["tvt"] = sub["id"].map(preds).fillna(0.0).astype(np.float32)
    else:
        sub = pd.DataFrame({"id": list(preds.keys()), "tvt": list(preds.values())})

    sub.to_csv("submission.csv", index=False)
    print("saved submission.csv", sub.shape)
    print(sub.head())


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    run_training()
    # post-hoc AR check on the saved ckpt; redundant when AR_TRAIN_EVAL=1 already
    # prints val_AR per epoch, so default off. Set AR_VAL=1 to run it anyway.
    if os.environ.get("AR_VAL", "0") == "1":
        run_ar_validation()
    run_inference()

In [ ]:
#private lb = 6.710
# ============================================================
# ROGII - INFERENCE-ONLY submission script
#         SDF ResUNet-noGN + Horizontal Dilated ASPP
#         + ConvNeXt Bottleneck + ECA + C2 GR-derivative channels
#         + AR RE-ANCHOR INFERENCE
# ------------------------------------------------------------
# Pure inference version of model_c2_ar_synpre.py:
#   - No training, no loss, no CV split. Loads trained checkpoints
#     fold_*_c2_synpre_best.pth from CKPT_DIR and produces submission.csv
#     via AR re-anchor inference (fold ensemble = mean of SDF logits).
#   - Model / dataset / decode are byte-identical to the training script,
#     so checkpoints load without any key remapping.
#
# Usage (Kaggle):
#   put checkpoints in a dataset, then e.g.
#   CKPT_DIR=/kaggle/input/rogii-synpre-ckpts python infer_c2_ar_synpre.py
#   (default CKPT_DIR="." — checkpoints next to the script)
# ============================================================
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset


# ============================================================
# Config
# ============================================================
class Config:
    TEST_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/test"
    TRAIN_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"  # only for sample_submission path
    TYPEWELL_SUFFIX = "__typewell.csv"
    HORIZONTAL_SUFFIX = "__horizontal_well.csv"

    # Directory containing fold_*_c2_synpre_best.pth
    CKPT_DIR = os.environ.get("CKPT_DIR", "/kaggle/input/datasets/zhuyifanss/testsad/kaggle/working")
    CKPT_GLOB = "fold_*_c2_synpre_best.pth"

    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

    # Grid (must match training)
    H_S = 16
    H_H = 64
    H_F = 704
    T_H = 128
    T_F = 128
    H_GR_FILTER = 51

    # SDF settings (must match training)
    SDF_DECODE_TEMP = 0.25
    HISTORY_SIGMA = 2.0

    ORIG_PAD_LEN = 16384

    # Trajectory Z
    DZ_SCALE = 2.0

    # ASPP dilation rates (along H-axis)
    ASPP_DILATIONS = (1, 4, 8, 16)

    # AR re-anchor inference
    AR_STEP = int(os.environ.get("AR_STEP", "1024"))


print("DEVICE:", Config.DEVICE)


# ============================================================
# Preprocess utils
# ============================================================
def resample_typewell(t, target_step=0.5):
    t_tvt = t["TVT"].values.astype(np.float64)
    t_gr = t["GR"].values.astype(np.float64)
    diffs = np.abs(np.diff(t_tvt))
    diffs = diffs[diffs > 0]
    ratio = (np.median(diffs) if len(diffs) else target_step) / target_step
    if np.isclose(ratio, 1.0):
        return t_tvt, t_gr
    if ratio < 1.0:
        group = max(int(round(1 / ratio)), 1)
        pad = (-len(t_tvt)) % group
        if pad:
            t_tvt = np.pad(t_tvt, (0, pad), mode="edge")
            t_gr = np.pad(t_gr, (0, pad), mode="edge")
        return t_tvt.reshape(-1, group).mean(1), t_gr.reshape(-1, group).mean(1)
    up = max(int(round(ratio)), 1)
    old = np.arange(len(t_tvt))
    new = np.linspace(0, len(t_tvt) - 1, (len(t_tvt) - 1) * up + 1)
    return np.interp(new, old, t_tvt), np.interp(new, old, t_gr)


def bin_mean(arr, step, back):
    arr = np.asarray(arr)
    if len(arr) == 0:
        return arr
    pad = (-len(arr)) % step
    if pad < step // 2:
        if pad:
            arr = np.pad(arr, (0, pad) if back else (pad, 0), mode="edge")
    elif pad:
        arr = arr[:-(step - pad)] if back else arr[(step - pad):]
    if len(arr) == 0:
        return arr
    return arr.reshape(-1, step).mean(1)


def safe_savgol(x, win=51, poly=2):
    x = np.asarray(x, dtype=float)
    if len(x) <= win:
        return x
    win = min(win, len(x))
    if win % 2 == 0:
        win -= 1
    if win < 7:
        return x
    return savgol_filter(x, win, poly)


def _get_numeric_col(h, name, default_value=0.0):
    if name in h.columns:
        return (
            h[name]
            .astype(float)
            .interpolate()
            .bfill()
            .ffill()
            .fillna(default_value)
            .values
        )
    return np.full(len(h), float(default_value), dtype=np.float64)


def resample_horizontal(h, step, offset=0, forced_h_ps=None):
    gr_raw = (
        h["GR"]
        .astype(float)
        .interpolate()
        .bfill()
        .ffill()
        .fillna(85.0)
        .values
    )
    gr = safe_savgol(gr_raw, Config.H_GR_FILTER, 2)

    if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
        tvt = h["TVT"].values.astype(float)
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        tvt = (
            h["TVT_input"]
            .astype(float)
            .ffill()
            .bfill()
            .fillna(0.0)
            .values
        )
    else:
        tvt = h["Z"].values.astype(float)

    z = _get_numeric_col(h, "Z", 0.0)
    md = _get_numeric_col(h, "MD", 0.0)
    if "MD" not in h.columns:
        md = np.arange(len(h), dtype=np.float64)
    x = _get_numeric_col(h, "X", 0.0)
    y = _get_numeric_col(h, "Y", 0.0)

    if forced_h_ps is not None:
        h_ps = int(forced_h_ps)
        h_ps = max(0, min(h_ps, len(h) - 1))
    elif "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        h_ps = int(np.flatnonzero(h["TVT_input"].notna().values)[-1]) + offset
        h_ps = max(0, min(h_ps, len(h) - 1))
    else:
        h_ps = len(h) // 2

    tvt0 = bin_mean(tvt[:h_ps + 1], step, back=False)
    gr0 = bin_mean(gr[:h_ps + 1], step, back=False)
    z0 = bin_mean(z[:h_ps + 1], step, back=False)
    md0 = bin_mean(md[:h_ps + 1], step, back=False)
    x0 = bin_mean(x[:h_ps + 1], step, back=False)
    y0 = bin_mean(y[:h_ps + 1], step, back=False)

    tvt1 = bin_mean(tvt[h_ps + 1:], step, back=True)
    gr1 = bin_mean(gr[h_ps + 1:], step, back=True)
    z1 = bin_mean(z[h_ps + 1:], step, back=True)
    md1 = bin_mean(md[h_ps + 1:], step, back=True)
    x1 = bin_mean(x[h_ps + 1:], step, back=True)
    y1 = bin_mean(y[h_ps + 1:], step, back=True)

    return tvt0, gr0, tvt1, gr1, h_ps, z0, z1, md0, md1, x0, x1, y0, y1


def crop_pad_1d(n, center, history, future):
    raw_i0 = center - history
    raw_i1 = center + future
    i0 = max(raw_i0, 0)
    i1 = min(raw_i1, n)
    pad_left = max(0, -raw_i0)
    pad_right = max(0, raw_i1 - n)
    return i0, i1, pad_left, pad_right


def get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt):
    if "TVT_input" in h.columns:
        val = h["TVT_input"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if "TVT" in h.columns:
        val = h["TVT"].iloc[h_ps]
        if pd.notna(val):
            return float(val)
    if len(h_tvt0):
        return float(h_tvt0[-1])
    return float(t_tvt[len(t_tvt) // 2])


def get_official_h_ps(h):
    if "TVT_input" in h.columns and h["TVT_input"].notna().sum() > 0:
        return int(np.flatnonzero(h["TVT_input"].notna().values)[-1])
    return len(h) // 2


def make_history_line(t_seg_tvt, h_seg_tvt, hist_mask):
    diff = np.abs(t_seg_tvt[:, None] - h_seg_tvt[None, :])
    matched = diff.argmin(axis=0)
    rows = np.arange(len(t_seg_tvt))[:, None]
    line = np.exp(
        -0.5 * ((rows - matched[None, :]) / Config.HISTORY_SIGMA) ** 2
    ).astype(np.float32)
    line *= hist_mask[None, :]
    return line


# ============================================================
# Dataset (inference-only path of the training HeatmapDataset)
# ============================================================
class HeatmapDataset(Dataset):
    def __init__(self, files, is_train=False, aug_repeats=None, start_jitter=None):
        # is_train kept in signature for interface parity; always eval here.
        self.files = list(files)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        cfg = Config
        item = self.files[idx]
        if isinstance(item, tuple):
            # in-memory (horizontal_df, typewell_df) pair — used by AR inference
            h, t = item
        else:
            hp = Path(item)
            sid = hp.name.split("__")[0]
            h = pd.read_csv(hp)
            t = pd.read_csv(hp.parent / f"{sid}{cfg.TYPEWELL_SUFFIX}")
        t_tvt, t_gr = resample_typewell(t, target_step=0.5)

        h_tvt0, h_gr0, h_tvt1, h_gr1, h_ps, h_z0, h_z1, h_md0, h_md1, h_x0, h_x1, h_y0, h_y1 = resample_horizontal(
            h, cfg.H_S, offset=0, forced_h_ps=None,
        )

        last_tvt_exact = get_last_tvt_exact(h, h_ps, h_tvt0, t_tvt)
        last_tvt = float(h_tvt0[-1]) if len(h_tvt0) else last_tvt_exact
        last_idx = int(np.abs(t_tvt - last_tvt).argmin()) if len(t_tvt) else 0
        last_z = float(h["Z"].iloc[h_ps])
        last_md = float(h["MD"].iloc[h_ps]) if "MD" in h.columns else float(h_ps)
        last_x = float(h["X"].iloc[h_ps]) if "X" in h.columns else 0.0
        last_y = float(h["Y"].iloc[h_ps]) if "Y" in h.columns else 0.0

        # typewell crop
        i0, i1, pl, pr = crop_pad_1d(len(t_tvt), last_idx + 1, cfg.T_H, cfg.T_F)
        t_mask = np.pad(np.ones(i1 - i0, dtype=np.float32), (pl, pr), constant_values=0.0)
        t_seg_tvt = np.pad(t_tvt[i0:i1], (pl, pr), mode="edge")
        t_seg_gr = np.pad(t_gr[i0:i1], (pl, pr), mode="edge")

        # horizontal history crop
        i0_h, i1_h, p0l, p0r = crop_pad_1d(len(h_tvt0), len(h_tvt0), cfg.H_H, 0)
        hm0 = np.pad(np.ones(i1_h - i0_h, dtype=np.float32), (p0l, p0r), constant_values=0.0)
        if len(h_tvt0):
            ht0 = np.pad(h_tvt0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hg0 = np.pad(h_gr0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hz0 = np.pad(h_z0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hmd0 = np.pad(h_md0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hx0 = np.pad(h_x0[i0_h:i1_h], (p0l, p0r), mode="edge")
            hy0 = np.pad(h_y0[i0_h:i1_h], (p0l, p0r), mode="edge")
        else:
            ht0 = np.zeros(cfg.H_H, dtype=np.float64)
            hg0 = np.zeros(cfg.H_H, dtype=np.float64)
            hz0 = np.full(cfg.H_H, last_z, dtype=np.float64)
            hmd0 = np.full(cfg.H_H, last_md, dtype=np.float64)
            hx0 = np.full(cfg.H_H, last_x, dtype=np.float64)
            hy0 = np.full(cfg.H_H, last_y, dtype=np.float64)

        # horizontal future crop
        i0_f, i1_f, p1l, p1r = crop_pad_1d(len(h_tvt1), 0, 0, cfg.H_F)
        hm1 = np.pad(np.ones(i1_f - i0_f, dtype=np.float32), (p1l, p1r), constant_values=0.0)
        if len(h_tvt1):
            ht1 = np.pad(h_tvt1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hg1 = np.pad(h_gr1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hz1 = np.pad(h_z1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hmd1 = np.pad(h_md1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hx1 = np.pad(h_x1[i0_f:i1_f], (p1l, p1r), mode="edge")
            hy1 = np.pad(h_y1[i0_f:i1_f], (p1l, p1r), mode="edge")
        else:
            ht1 = np.zeros(cfg.H_F, dtype=np.float64)
            hg1 = np.zeros(cfg.H_F, dtype=np.float64)
            hz1 = np.full(cfg.H_F, last_z, dtype=np.float64)
            hmd1 = np.full(cfg.H_F, last_md, dtype=np.float64)
            hx1 = np.full(cfg.H_F, last_x, dtype=np.float64)
            hy1 = np.full(cfg.H_F, last_y, dtype=np.float64)

        h_mask = np.concatenate([hm0, hm1]).astype(np.float32)
        h_seg_tvt = np.concatenate([ht0, ht1]).astype(np.float64)
        h_seg_gr = np.concatenate([hg0, hg1]).astype(np.float64)
        h_seg_z = np.concatenate([hz0, hz1]).astype(np.float64)
        h_seg_md = np.concatenate([hmd0, hmd1]).astype(np.float64)
        h_seg_x = np.concatenate([hx0, hx1]).astype(np.float64)
        h_seg_y = np.concatenate([hy0, hy1]).astype(np.float64)

        hist_mask = np.zeros(cfg.H_H + cfg.H_F, dtype=np.float32)
        hist_mask[:cfg.H_H] = hm0

        h_dz = np.gradient(h_seg_z)
        h_dz = np.clip(h_dz / cfg.DZ_SCALE, -3.0, 3.0).astype(np.float32)

        H = cfg.H_H + cfg.H_F
        col_center = np.zeros(H, dtype=np.float64)
        for j in range(cfg.H_H):
            k = cfg.H_H - 1 - j
            col_center[j] = h_ps - k * cfg.H_S - (cfg.H_S - 1) / 2.0
        for k in range(cfg.H_F):
            col_center[cfg.H_H + k] = h_ps + 1 + k * cfg.H_S + (cfg.H_S - 1) / 2.0
        h_pos_rel = np.clip((col_center - float(h_ps)) / 4096.0, -4.0, 4.0).astype(np.float32)

        dmd = np.gradient(h_seg_md)
        dmd = np.where(np.abs(dmd) < 1e-6, 1.0, dmd)
        dx_dmd = np.gradient(h_seg_x) / dmd
        dy_dmd = np.gradient(h_seg_y) / dmd
        dxy = np.sqrt(dx_dmd ** 2 + dy_dmd ** 2) + 1e-6
        az_cos = np.clip(dx_dmd / dxy, -1.0, 1.0).astype(np.float32)
        az_sin = np.clip(dy_dmd / dxy, -1.0, 1.0).astype(np.float32)

        history_line = make_history_line(t_seg_tvt, h_seg_tvt, hist_mask)

        if "TVT" in h.columns and h["TVT"].notna().sum() > 0:
            orig_tvt = h["TVT"].values.astype(np.float32)
        else:
            orig_tvt = np.zeros(len(h), dtype=np.float32)

        orig_len = len(orig_tvt)
        padded_tvt = np.zeros(cfg.ORIG_PAD_LEN, dtype=np.float32)
        ncopy = min(orig_len, cfg.ORIG_PAD_LEN)
        padded_tvt[:ncopy] = orig_tvt[:ncopy]

        return {
            "t_gr": torch.tensor(t_seg_gr, dtype=torch.float32),
            "h_gr": torch.tensor(h_seg_gr, dtype=torch.float32),
            "hist_mask": torch.tensor(hist_mask, dtype=torch.float32),
            "history_line": torch.tensor(history_line, dtype=torch.float32),
            "h_dz": torch.tensor(h_dz, dtype=torch.float32),
            "h_pos_rel": torch.tensor(h_pos_rel, dtype=torch.float32),
            "az_sin": torch.tensor(az_sin, dtype=torch.float32),
            "az_cos": torch.tensor(az_cos, dtype=torch.float32),
            "t_seg_tvt": torch.tensor(t_seg_tvt, dtype=torch.float32),
            "h_seg_tvt": torch.tensor(h_seg_tvt, dtype=torch.float32),
            "t_mask": torch.tensor(t_mask, dtype=torch.float32),
            "h_mask": torch.tensor(h_mask, dtype=torch.float32),
            "orig_tvt": torch.tensor(padded_tvt, dtype=torch.float32),
            "orig_len": torch.tensor(orig_len, dtype=torch.int64),
            "h_ps": torch.tensor(h_ps, dtype=torch.int64),
            "last_tvt_exact": torch.tensor(last_tvt_exact, dtype=torch.float32),
        }


# ============================================================
# Model (identical module tree to training -> ckpt keys match,
# including aux heads, which exist in the state_dict)
# ============================================================

class ResidualConvBlock(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.conv1 = nn.Conv2d(ci, co, 3, padding=1, bias=False)
        self.norm1 = nn.Identity()
        self.conv2 = nn.Conv2d(co, co, 3, padding=1, bias=False)
        self.norm2 = nn.Identity()
        self.act = nn.GELU()
        self.skip = nn.Conv2d(ci, co, 1, bias=False) if ci != co else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + residual)


class LayerNorm2d(nn.Module):
    """Channel-wise LayerNorm for [B, C, T, H]."""
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(c))
        self.bias = nn.Parameter(torch.zeros(c))
        self.eps = eps

    def forward(self, x):
        u = x.mean(dim=1, keepdim=True)
        s = (x - u).pow(2).mean(dim=1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return x * self.weight[:, None, None] + self.bias[:, None, None]


class ConvNeXt2DBlock(nn.Module):
    def __init__(self, c, expansion=4, kernel=7):
        super().__init__()
        self.dw = nn.Conv2d(c, c, kernel, padding=kernel // 2, groups=c, bias=True)
        self.norm = LayerNorm2d(c)
        self.pw1 = nn.Conv2d(c, c * expansion, 1, bias=True)
        self.act = nn.GELU()
        self.pw2 = nn.Conv2d(c * expansion, c, 1, bias=True)
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, x):
        y = self.dw(x)
        y = self.norm(y)
        y = self.pw2(self.act(self.pw1(y)))
        return x + self.gamma * y


class ECABlock2D(nn.Module):
    def __init__(self, c, k_size=5):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        y = self.avg(x).squeeze(-1).transpose(1, 2)  # [B, 1, C]
        y = self.conv(y).transpose(1, 2).unsqueeze(-1).sigmoid()
        return x + self.gamma * (x * y)


class HorizontalDilatedASPP(nn.Module):
    def __init__(self, ci, co, dilations=None):
        super().__init__()
        dilations = dilations or Config.ASPP_DILATIONS
        n_branches = len(dilations)
        branch_ch = co // n_branches
        self.branches = nn.ModuleList()
        for d in dilations:
            self.branches.append(nn.Sequential(
                nn.Conv2d(ci, branch_ch, 3, padding=(1, d), dilation=(1, d), bias=False),
                nn.Identity(),
                nn.GELU(),
            ))
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ci, branch_ch, 1, bias=False),
            nn.Identity(),
            nn.GELU(),
        )
        total_ch = branch_ch * (n_branches + 1)
        self.fuse = nn.Sequential(
            nn.Conv2d(total_ch, co, 1, bias=False),
            nn.Identity(),
            nn.GELU(),
        )

    def forward(self, x):
        outs = [branch(x) for branch in self.branches]
        gp = self.global_pool(x)
        gp = gp.expand(-1, -1, x.shape[2], x.shape[3])
        outs.append(gp)
        return self.fuse(torch.cat(outs, dim=1))


class HeatmapResUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.gr_norm = nn.InstanceNorm2d(2)

        # Encoder
        self.e0 = ResidualConvBlock(12, base)   # 9 baseline + 3 GR-derivative (C2)
        self.e1 = ResidualConvBlock(base, base * 2)
        self.e2 = ResidualConvBlock(base * 2, base * 4)
        self.e3 = ResidualConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        self.bott = HorizontalDilatedASPP(base * 8, base * 16)
        self.cn_bott = ConvNeXt2DBlock(base * 16, kernel=7)
        self.eca_bott = ECABlock2D(base * 16, k_size=5)

        # Decoder
        self.u3 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.d3 = ResidualConvBlock(base * 16, base * 8)
        self.u2 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.d2 = ResidualConvBlock(base * 8, base * 4)
        self.u1 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.d1 = ResidualConvBlock(base * 4, base * 2)
        self.u0 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.d0 = ResidualConvBlock(base * 2, base)

        # Heads (aux heads kept so state_dict keys match exactly)
        self.head = nn.Conv2d(base, 1, 1)
        self.aux3 = nn.Conv2d(base * 8, 1, 1)
        self.aux2 = nn.Conv2d(base * 4, 1, 1)
        self.aux1 = nn.Conv2d(base * 2, 1, 1)

    def build_image(self, batch):
        dev = next(self.parameters()).device
        t_gr = batch["t_gr"].to(dev)
        h_gr = batch["h_gr"].to(dev)
        hist_mask = batch["hist_mask"].to(dev)
        history = batch["history_line"].to(dev)
        h_dz = batch["h_dz"].to(dev)
        B, T = t_gr.shape
        _, H = h_gr.shape
        t_img = t_gr.view(B, 1, T, 1).expand(B, 1, T, H)
        h_img = h_gr.view(B, 1, 1, H).expand(B, 1, T, H)
        gr_pair = self.gr_norm(torch.cat([t_img, h_img], dim=1))
        gr_diff = torch.clamp((t_img - h_img) / 40.0, -4.0, 4.0)
        mask_img = hist_mask.view(B, 1, 1, H).expand(B, 1, T, H)
        history_img = history[:, None, :, :]
        dz_img = h_dz.view(B, 1, 1, H).expand(B, 1, T, H)
        h_pos_rel = batch["h_pos_rel"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_sin = batch["az_sin"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)
        az_cos = batch["az_cos"].to(dev).view(B, 1, 1, H).expand(B, 1, T, H)

        # --- C2: GR shape/derivative channels ---
        dt = torch.zeros_like(t_gr)
        dt[:, 1:] = t_gr[:, 1:] - t_gr[:, :-1]
        dt = torch.clamp(dt / 10.0, -4.0, 4.0)
        dh = torch.zeros_like(h_gr)
        dh[:, 1:] = h_gr[:, 1:] - h_gr[:, :-1]
        dh = torch.clamp(dh / 10.0, -4.0, 4.0)
        dt_img = dt.view(B, 1, T, 1).expand(B, 1, T, H)
        dh_img = dh.view(B, 1, 1, H).expand(B, 1, T, H)
        dgr_diff = torch.clamp(dt_img - dh_img, -4.0, 4.0)

        return torch.cat([gr_pair, gr_diff, mask_img, history_img, dz_img,
                          h_pos_rel, az_sin, az_cos, dt_img, dh_img, dgr_diff], dim=1)

    def forward(self, batch):
        x = self.build_image(batch)

        e0 = self.e0(x)
        e1 = self.e1(self.pool(e0))
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))

        pooled_e3 = self.pool(e3)
        b = self.bott(pooled_e3)
        b = self.cn_bott(b)
        b = self.eca_bott(b)

        d3 = self.d3(torch.cat([self.u3(b), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        d0 = self.d0(torch.cat([self.u0(d1), e0], dim=1))

        return self.head(d0)


# ============================================================
# Decode
# ============================================================
def map_to_original(pred_H, orig_len, h_ps, last_tvt_exact):
    centers = []
    values = []
    for k in range(Config.H_H):
        centers.append(h_ps - k * Config.H_S - (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H - 1 - k])
    centers.append(float(h_ps))
    values.append(float(last_tvt_exact))
    for k in range(Config.H_F):
        centers.append(h_ps + 1 + k * Config.H_S + (Config.H_S - 1) / 2.0)
        values.append(pred_H[Config.H_H + k])
    centers = np.asarray(centers, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    order = np.argsort(centers)
    centers = centers[order]
    values = values[order]
    keep = np.r_[True, np.diff(centers) > 1e-6]
    return np.interp(np.arange(orig_len), centers[keep], values[keep])


def decode_sdf(pred_sdf, t_seg_tvt, h_seg_tvt, t_mask, orig_len, h_ps, last_tvt_exact):
    """Pure soft-argmax decode: per future column, TVT = softmax(-|sdf|/temp) . t_tvt."""
    score = -np.abs(pred_sdf.astype(np.float64)) / float(Config.SDF_DECODE_TEMP)
    if t_mask is not None:
        score = np.where(t_mask[:, None] > 0, score, -1e9)
    score = score - score.max(axis=0, keepdims=True)
    prob = np.exp(score)
    prob = prob / (prob.sum(axis=0, keepdims=True) + 1e-12)
    pred_H = (prob * t_seg_tvt[:, None]).sum(axis=0).astype(np.float64)
    pred_H[:Config.H_H] = h_seg_tvt[:Config.H_H]
    return map_to_original(pred_H, orig_len, h_ps, last_tvt_exact)


# ============================================================
# AR re-anchor inference
# ============================================================
@torch.no_grad()
def ar_predict_well(models, df, tw):
    """AR re-anchor prediction for one well (honest: known prefix + own predictions)."""
    olen = len(df)
    ps0 = get_official_h_ps(df)
    if "TVT_input" in df.columns and df["TVT_input"].notna().sum() > 0:
        known = df["TVT_input"].astype(float).interpolate().bfill().ffill().values
    elif "TVT" in df.columns and df["TVT"].notna().sum() > 0:
        known = df["TVT"].astype(float).interpolate().bfill().ffill().values
    else:
        known = np.zeros(olen)

    work = np.full(olen, np.nan)
    work[: ps0 + 1] = known[: ps0 + 1]
    cur, n_steps = ps0, 0
    while cur < olen - 1:
        w = df.copy()
        w["TVT"] = pd.Series(work).ffill().bfill().values
        ti = np.full(olen, np.nan)
        ti[: cur + 1] = w["TVT"].values[: cur + 1]
        w["TVT_input"] = ti
        sample = HeatmapDataset([(w, tw)])[0]
        batch = {k: v.unsqueeze(0) for k, v in sample.items()}
        avg_logits = np.mean(
            [m(batch).squeeze(1).float().cpu().numpy()[0] for m in models], axis=0)
        pred = decode_sdf(
            avg_logits,
            sample["t_seg_tvt"].numpy(), sample["h_seg_tvt"].numpy(),
            sample["t_mask"].numpy(),
            int(sample["orig_len"]), int(sample["h_ps"]),
            float(sample["last_tvt_exact"]),
        )
        end = min(cur + Config.AR_STEP, olen - 1)
        work[cur + 1: end + 1] = pred[cur + 1: end + 1]
        cur, n_steps = end, n_steps + 1
    return work, ps0, n_steps


@torch.no_grad()
def run_inference():
    test_files = sorted(Path(Config.TEST_DIR).glob(f"*{Config.HORIZONTAL_SUFFIX}"))
    ckpts = sorted(Path(Config.CKPT_DIR).glob(Config.CKPT_GLOB))
    print(f"CKPT_DIR: {Config.CKPT_DIR}")
    print(f"test wells: {len(test_files)}  checkpoints: {len(ckpts)}")
    if not test_files:
        raise RuntimeError(f"no test wells found in {Config.TEST_DIR}")
    if not ckpts:
        raise RuntimeError(
            f"no checkpoints matching '{Config.CKPT_GLOB}' in {Config.CKPT_DIR} "
            f"(set CKPT_DIR env var)")

    models = []
    for ckpt in ckpts:
        model = HeatmapResUNet(base=16).to(Config.DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=Config.DEVICE))
        model.eval()
        models.append(model)
        print("loaded", ckpt.name)

    preds = {}
    for f in test_files:
        sid = f.name.split("__")[0]
        df = pd.read_csv(f)
        tw = pd.read_csv(f.parent / f"{sid}{Config.TYPEWELL_SUFFIX}")
        work, ps0, n_steps = ar_predict_well(models, df, tw)
        if "TVT_input" in df.columns:
            rows = np.flatnonzero(df["TVT_input"].isna().values)
        else:
            rows = np.arange(len(df))
        for r in rows:
            preds[f"{sid}_{int(r)}"] = float(work[int(r)])
        print(f"done {sid}  AR steps: {n_steps}  submit rows: {len(rows)}")

    sub_path = Path(Config.TRAIN_DIR).parent / "sample_submission.csv"
    if sub_path.exists():
        sub = pd.read_csv(sub_path)[["id"]].copy()
        sub["tvt"] = sub["id"].map(preds).fillna(0.0).astype(np.float32)
    else:
        sub = pd.DataFrame({"id": list(preds.keys()), "tvt": list(preds.values())})
    sub.to_csv("submission.csv", index=False)
    print("saved submission.csv", sub.shape)
    print(sub.head())


if __name__ == "__main__":
    run_inference()